# 提出(graft): 0.3·**v28 GBM** + 0.7·**v27 近傍ヒューリスティック** → 7191 下流(射影/0.55/override/gold)

7191『現状最強』ノートの**下流チェーンを丸ごと維持**し、予測の**出力だけ差し替え**た graft 版。

- **PF/selector(0.7 側) → v27 近傍ヒューリスティック**(r0 GPU smooth PF + XY近傍疑似typewell + beam+hold+pp, LB6.697)= `_V27_PRED`。
- **tabular/GBM(0.3 側 と fleongg) → v28 GBM**(LightGBM/CatBoost/XGB 各3 = 9モデル平均 + apply_pp + per-well sg, **Ridgeなし**)= `_V28_PRED`。
- **7:3(cell21) / robust射影(cell22) / 0.55 sp45×fleongg ブレンド(cell37) / contact override(cell38) / gold(cell40) は維持**。
- 旧GBM_A(cell7-17,23)はスキップ。selector(cell4-6)と fleongg lib(cell25-34)は **gold が再利用**するため残置。
  fleongg main(cell35)は `init_imputers`(gold用 `_FI`)＋出力=v28 に軽量化。

**Kaggle input**: コンペデータ(train/test) ＋ 学習済 `artifacts_v28`(models/features.json/pp_params.json) を mount。
**注意(過学習覚悟の提出)**: v28/v27/selector/gold で smooth PF/PF を複数回まわすため実行は重い。

In [ ]:
# ===== エンジン(create_train_parquet_v28.py を持込み。ロード時に v9 FI/DI を train で fit) =====
"""
v26 train.parquet ビルダー ― v9 の build_well 特徴(そのまま持ち込み)＋ PF を v25 smooth PF(固定ラグ lag512)に注入
==========================================================================================
方針(ユーザー指示):
  - **入れる特徴量は v9(pf_tuning_create_train_csv_multi_pf_v9.ipynb) と同じ**。v9 の build_well を
    **exec でなくコードとして持ち込み(下記 "v9 INLINE" ブロック)**。beam/formation/NCC/pf_z 等を全部生成。
  - **run_pf_ancc だけ v25 smooth PF(固定ラグ SMOOTH_LAG=192)に monkeypatch** して注入。
  - **PF複数**(PF_BANK)。**gsf/heur は入れない**(beam は v9 build_well 内包)。
  - パラメータは冒頭ハードコード(PF_LIB / PF_BANK / SMOOTH_LAG …)。

出力: v26/artifacts_v26/data/train.parquet
実行: rogii_claude/.venv/Scripts/python.exe v26/create_train_parquet_v26.py
※ 生成方法: scratchpad/build_create_v26.py(v9 cell3/5/7 を持ち込み)。lag512はバッファ大→OOMなら WELL_CHUNK↓。
"""

import os, time, glob, json, threading, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore"); np.seterr(all="ignore")

# ============================== CONFIG(v26) ==============================
def _detect_data():   # Kaggle / ローカル 両対応(v9 CFG.dataset_path と共有)
    for _p in ["/kaggle/input/rogii-wellbore-geology-prediction",
               "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
               r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction"]:
        if os.path.exists(_p): return Path(_p)
    _h = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    return Path(os.path.dirname(_h[0])) if _h else Path(r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction")
_V26_DATA = _detect_data()
PROJECT_V26 = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
DATA_DIR_V26 = _V26_DATA
OUT_PARQUET = PROJECT_V26 / "v28" / "artifacts_v28_2" / "data" / "train.parquet"
V26_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
V26_DTYPE  = torch.float32

SPLIT       = "train"       # "train"(真値あり) / "test"
IS_TRAIN    = True
N_WELLS     = None          # None=全坑井
SEED_V26    = 0
N_SEED      = 32            # ★r0_seed32 regime(6.786)
SMOOTH_LAG  = 192           # ★smooth PF 固定ラグ(先読みL)。v25で最適(lag192)。
CLAMP_FT    = None          # smoothed が forward から±これ超で外れた行は forward に(注入安定化, None=無効)。今回はNone。
WELL_CHUNK  = 50             # GPUバッチ坑井数(OOMなら下げる)
N_JOBS_CPU  = 4


PF_LIB = {
    "pf_1": {"gr_power": 2.1947763195255123, "gr_sig_def": 114.11439615641156, "gr_sig_max": 32.79346108797473,
             "gr_sig_min": 7.495879205722816, "gr_sig_mult": 2.7482077023565075, "hgr_smooth_r": 3,
             "init_pos_std": 0.43824435375436277, "init_rate_std": 0.001840222365629066, "jump_prob": 0.002,
             "jump_std": 5.0, "likelihood_scale": 20.0, "mom": 0.998833416480357, "n_particles": 600,
             "pn": 0.006788762926604484, "rate_clip": 2.0, "resamp": 0.060642973044944336, "rp": 0.6864944651997384,
             "rr": 0.00015873788792956448, "tvt_clip_margin": 113.08092552242186, "tw_gr_smooth_r": 0,
             "vn": 0.0013871256161405515, "phys_sig": 0.05095402721954007, "use_phys": False},
    "pf_2": {"gr_sig_def": 71.37105679394953, "gr_sig_max": 44.7373292550486, "gr_sig_min": 6.520200142977573,
             "gr_sig_mult": 1.9000654781073127, "init_pos_std": 1.1039311766848976, "init_rate_std": 0.009213697911348712,
             "likelihood_scale": 18.584958539746772, "mom": 0.9994265753132063, "n_particles": 600,
             "pn": 0.011257078147676399, "resamp": 0.30834451379977984, "rp": 0.49711567429512854,
             "rr": 0.00031680690540158854, "tvt_clip_margin": 63.752292532272826, "vn": 0.0010650490363481595,
             "rate_clip": 0.0, "gr_power": 2.0, "hgr_smooth_r": 0, "tw_gr_smooth_r": 0, "jump_prob": 0.0,
             "jump_std": 10.0, "phys_sig": 0.05095402721954007, "use_phys": False},
    "pf_3": {"gr_power": 2.074149546524975, "gr_sig_def": 66.97913258656452, "gr_sig_max": 50.843656140023995,
             "gr_sig_min": 9.427442041004895, "gr_sig_mult": 2.4485083695548218, "hgr_smooth_r": 0,
             "init_pos_std": 2.3279966476603975, "init_rate_std": 0.03938758391370206, "jump_prob": 0.002,
             "jump_std": 5.0, "likelihood_scale": 14.890876838217093, "mom": 0.9999769474035675, "n_particles": 600,
             "pn": 0.04394815893193551, "rate_clip": 0.0, "resamp": 0.054606961294962585, "rp": 0.42322457711970995,
             "rr": 0.002148810696831248, "tvt_clip_margin": 91.4794941498565, "tw_gr_smooth_r": 0,
             "vn": 0.0006883646035942229, "phys_sig": 0.05095402721954007, "use_phys": False},
}
PF_BANK = ["pf_1", "pf_2", "pf_3"]     # ← 使うPF(順序=pf_1/2/3 の接尾)

# --- v28 追加: self-PF / neighbor-PF / Z傾斜 ---
K_MAX      = 3
MAX_DIST   = 1500.0   # 近傍(XY重心距離)の上限
NEED_REFS  = 2        # 近傍<2本 -> neighbor無し=tw(通常実行v20相当)にフォールバック
SELF_MIX_W = 1.0      # self疑似typewell(自分prefix)の grid置換率
NBR_MIX_W  = 1.0      # neighbor疑似typewell(近傍train)の grid置換率
ZGRAD_R    = 25       # Z傾斜(dZ/dMD)の移動平均半径(断層の影響回避)
# =======================================================================

_TLS = threading.local()
_FWD_ENS = {}    # (wid, pf_name) -> dict(mean=smoothed, std=smoothed_std)

def _patched_run_pf_ancc(hw, tw_tvt, tw_gr, p):
    """v9 build_well が呼ぶ run_pf_ancc を差替。前計算した smooth PF(平滑平均/std)を返す。"""
    r = _FWD_ENS.get((getattr(_TLS, "wid", None), p.get("name")))
    if r is None:
        return np.array([], np.float32), np.array([], np.float32)
    return r["mean"].astype(np.float32), r["std"].astype(np.float32)


# ==================== v9 INLINE(cell3+5+7 をそのまま持込, exec無し) ====================
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
from pathlib import Path
from numba import njit
import multiprocessing
import pandas as pd
import numpy as np
import warnings
import time
import gc
import os

warnings.filterwarnings("ignore")


class CFG:
    # Kaggle competition dataset path
    dataset_path = _V26_DATA

    # Output paths
    output_dir = Path(r"C:\Users\kosaka256\Documents\rogii\tuning_pf_output_v9")
    part_dir = output_dir / "train_feature_parts"
    train_csv_path = output_dir / "train.csv"

    # Resume behavior
    overwrite_parts = False
    overwrite_train_csv = True

    # Debug: set None for all wells, or e.g. 5 for quick test
    max_wells = None


SEED=0
# joblib n_jobs=0 is invalid. Keep this safe for both local Windows and Kaggle.
NCPU=max(1, min(4, multiprocessing.cpu_count()))

FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5

BEAMS=[
    (15, 7.885250829793783, 69.70049362431021, 3,"gr"),
    (20, 49.43859691674568, 148.592415555271, 3, "hard"),
    (12, 11.84508580886079, 58.844708544034425, 2, "loose"),
    (15, 12.769364767855363, 89.75744127131894, 3, "mid"),
    (15, 10.693883733824851, 118.15908775467301, 8, "smooth"),
    (40, 67.99404057104671,101.2117946585323, 1, "best")
]

# -----------------------------------------------------------------------------
# Particle Filter parameter sets
# -----------------------------------------------------------------------------
# Add 2-3 good PF parameter dictionaries here.
# Columns are created with suffixes: 1st set -> *_1, 2nd set -> *_2, ...
# KEEP_FIRST_PF_ALIAS=True also keeps the old unsuffixed columns for the 1st set
# so existing training notebooks do not break.
PF_SUFFIX_START = 1
KEEP_FIRST_PF_ALIAS = True

PF_PARAM_SETS_RAW = [
    {
        "name": "pf_1",
        "gr_power": 2.1947763195255123,
        "gr_sig_def": 114.11439615641156,
        "gr_sig_max": 32.79346108797473,
        "gr_sig_min": 7.495879205722816,
        "gr_sig_mult": 2.7482077023565075,
        "hgr_smooth_r": 3,
        "init_pos_std": 0.43824435375436277,
        "init_rate_std": 0.001840222365629066,
        "jump_prob": 0.002,
        "jump_std": 5.0,
        "likelihood_scale": 20.0,
        "mom": 0.998833416480357,
        "n_particles": 600,
        "n_seeds": 32,
        "pn": 0.006788762926604484,
        "rate_clip": 2.0,
        "resamp": 0.060642973044944336,
        "rp": 0.6864944651997384,
        "rr": 0.00015873788792956448,
        "tvt_clip_margin": 113.08092552242186,
        "tw_gr_smooth_r": 0,
        "vn": 0.0013871256161405515,
    },
    {
        "name": "pf_2",
        "gr_sig_def": 71.37105679394953,
        "gr_sig_max": 44.7373292550486,
        "gr_sig_min": 6.520200142977573,
        "gr_sig_mult": 1.9000654781073127,
        "init_pos_std": 1.1039311766848976,
        "init_rate_std": 0.009213697911348712,
        "likelihood_scale": 18.584958539746772,
        "mom": 0.9994265753132063,
        "n_particles": 600,
        "n_seeds": 32,
        "pn": 0.011257078147676399,
        "resamp": 0.30834451379977984,
        "rp": 0.49711567429512854,
        "rr": 0.00031680690540158854,
        "tvt_clip_margin": 63.752292532272826,
        "vn": 0.0010650490363481595, 
        "rate_clip": 0.0,
        "gr_power": 2.0,
        "hgr_smooth_r": 0,
        "tw_gr_smooth_r": 0,
        "jump_prob": 0.0,
        "jump_std": 10.0,
    },
    {
        "name": "pf_3",
        "gr_power": 2.074149546524975, 
        "gr_sig_def": 66.97913258656452, 
        "gr_sig_max": 50.843656140023995, 
        "gr_sig_min": 9.427442041004895, 
        "gr_sig_mult": 2.4485083695548218, 
        "hgr_smooth_r": 0, 
        "init_pos_std": 2.3279966476603975, 
        "init_rate_std": 0.03938758391370206, 
        "jump_prob": 0.002, 
        "jump_std": 5.0, 
        "likelihood_scale": 14.890876838217093, 
        "mom": 0.9999769474035675, 
        "n_particles": 600, 
        "n_seeds": 32, 
        "pn": 0.04394815893193551, 
        "rate_clip": 0.0, 
        "resamp": 0.054606961294962585, 
        "rp": 0.42322457711970995, 
        "rr": 0.002148810696831248, 
        "seed0": 0, 
        "tvt_clip_margin": 91.4794941498565, 
        "tw_gr_smooth_r": 0, 
        "vn": 0.0006883646035942229
    }
]


def _make_pf_params(raw, idx):
    """Map Optuna-style PF parameter dict to internal names used by this notebook."""
    r = dict(raw)
    p = {}
    p["name"] = str(r.get("name", f"pf_{idx + PF_SUFFIX_START}"))
    p["suffix"] = str(r.get("suffix", f"_{idx + PF_SUFFIX_START}"))
    p["PF_N"] = int(r.get("PF_N", r.get("n_particles", 600)))
    p["ANCC_N"] = int(r.get("ANCC_N", r.get("n_particles", p["PF_N"])))
    p["PF_N_SEEDS"] = int(r.get("PF_N_SEEDS", r.get("n_seeds", 8)))
    p["PF_SEED0"] = int(r.get("PF_SEED0", SEED + idx * 100000))
    p["PF_LIKELIHOOD_SCALE"] = float(r.get("PF_LIKELIHOOD_SCALE", r.get("likelihood_scale", 20.0)))
    p["PF_INIT_SPR"] = float(r.get("PF_INIT_SPR", r.get("init_pos_std", 0.5)))
    p["PF_INIT_V_STD"] = float(r.get("PF_INIT_V_STD", r.get("init_rate_std", 0.01)))
    p["PF_MOM"] = float(r.get("PF_MOM", r.get("mom", 0.999)))
    p["PF_VN"] = float(r.get("PF_VN", r.get("vn", 0.001)))
    p["PF_PN"] = float(r.get("PF_PN", r.get("pn", 0.005)))
    p["PF_ROUGH_P"] = float(r.get("PF_ROUGH_P", r.get("rp", 0.5)))
    p["PF_ROUGH_V"] = float(r.get("PF_ROUGH_V", r.get("rr", 0.001)))
    p["PF_RESAMP"] = float(r.get("PF_RESAMP", r.get("resamp", 0.5)))
    p["PF_GR_SIG_MIN"] = float(r.get("PF_GR_SIG_MIN", r.get("gr_sig_min", 5.0)))
    p["PF_GR_SIG_MAX"] = float(r.get("PF_GR_SIG_MAX", r.get("gr_sig_max", 50.0)))
    p["PF_GR_SIG_DEF"] = float(r.get("PF_GR_SIG_DEF", r.get("gr_sig_def", 100.0)))
    p["PF_GR_SIG_MULT"] = float(r.get("PF_GR_SIG_MULT", r.get("gr_sig_mult", 2.0)))
    p["PF_TVT_CLIP_MARGIN"] = float(r.get("PF_TVT_CLIP_MARGIN", r.get("tvt_clip_margin", 50.0)))
    p["PF_RATE_CLIP"] = float(r.get("PF_RATE_CLIP", r.get("rate_clip", 0.0)))
    p["PF_GR_POWER"] = float(r.get("PF_GR_POWER", r.get("gr_power", 2.0)))
    p["PF_HGR_SMOOTH_R"] = int(r.get("PF_HGR_SMOOTH_R", r.get("hgr_smooth_r", 0)))
    p["PF_TW_GR_SMOOTH_R"] = int(r.get("PF_TW_GR_SMOOTH_R", r.get("tw_gr_smooth_r", 0)))
    p["PF_JUMP_PROB"] = float(r.get("PF_JUMP_PROB", r.get("jump_prob", 0.0)))
    p["PF_JUMP_STD"] = float(r.get("PF_JUMP_STD", r.get("jump_std", 0.0)))
    p["PF_GR_WIN"] = max(1, 2 * int(p["PF_HGR_SMOOTH_R"]) + 1)
    p["PF_GR_WT"] = float(r.get("PF_GR_WT", r.get("gr_wt", 0.3)))

    # ANCC-PF tracks TVT+Z. Defaults reuse the tuned PF transition/init/resampling params.
    p["ANCC_ALPHA"] = float(r.get("ANCC_ALPHA", p["PF_MOM"]))
    p["ANCC_RN"] = float(r.get("ANCC_RN", p["PF_VN"]))
    p["ANCC_PN"] = float(r.get("ANCC_PN", p["PF_PN"]))
    p["ANCC_IR"] = float(r.get("ANCC_IR", p["PF_INIT_V_STD"]))
    p["ANCC_IS"] = float(r.get("ANCC_IS", p["PF_INIT_SPR"]))
    p["ANCC_RP"] = float(r.get("ANCC_RP", p["PF_ROUGH_P"]))
    p["ANCC_RR"] = float(r.get("ANCC_RR", p["PF_ROUGH_V"]))
    return p

PF_PARAM_SETS = [_make_pf_params(raw, i) for i, raw in enumerate(PF_PARAM_SETS_RAW)]
if len(PF_PARAM_SETS) == 0:
    raise ValueError("PF_PARAM_SETS_RAW must contain at least one parameter set.")
print("PF parameter sets:")
for p in PF_PARAM_SETS:
    print(f"  {p['name']}: suffix={p['suffix']}, N={p['PF_N']}, seeds={p['PF_N_SEEDS']}, jump=({p['PF_JUMP_PROB']}, {p['PF_JUMP_STD']})")

@njit(cache=False)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

@njit(cache=False)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N)
    np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn()
        na[j] =aux[ci]+rv*np.random.randn()
    return np2,na

@njit(cache=False)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    """Beam search ±2 delta, Numba JIT."""
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):            # ±2: TVT can go down
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

@njit(cache=False)
def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
              ALPHA,RN,PN,IR,IS,RP,RR,RESAMP,
              RATE_CLIP,GR_POWER,JUMP_PROB,JUMP_STD,CLIP_MARGIN,SEED_VALUE):
    if SEED_VALUE >= 0:
        np.random.seed(SEED_VALUE)
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+IR*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    log_lik=0.
    tvt_lo=vmin-CLIP_MARGIN
    tvt_hi=vmin+(len(gg)-1)*step+CLIP_MARGIN
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            if RATE_CLIP>0.:
                if rate[j] > RATE_CLIP: rate[j] = RATE_CLIP
                elif rate[j] < -RATE_CLIP: rate[j] = -RATE_CLIP
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            # Rare jump particles for extreme TVT movement. jump_prob=0 disables this.
            if JUMP_PROB>0. and JUMP_STD>0. and np.random.random()<JUMP_PROB:
                pos[j]+=JUMP_STD*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,tvt_lo); tvt_j=min(tvt_j,tvt_hi)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.; avg_lk=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=abs((gr_v[i]-eg)/gs)
                dp=d**GR_POWER
                lk=max(np.exp(-0.5*dp) if dp<600. else 0.,1e-300)
                old_w=w[j]
                avg_lk+=old_w*lk
                w[j]=old_w*lk; ws+=w[j]
            log_lik+=np.log(max(avg_lk,1e-300))
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            if RATE_CLIP>0.:
                for j in range(N):
                    if rate[j] > RATE_CLIP: rate[j] = RATE_CLIP
                    elif rate[j] < -RATE_CLIP: rate[j] = -RATE_CLIP
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_,log_lik

@njit(cache=False)
def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
          gs,ip,iv,beta,icpt,zsig,N,
          MOM,VN,PN,GR_WT,RP,RV,RESAMP,IP_STD,IV_STD,
          RATE_CLIP,GR_POWER,JUMP_PROB,JUMP_STD,CLIP_MARGIN,SEED_VALUE):
    if SEED_VALUE >= 0:
        np.random.seed(SEED_VALUE)
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+IP_STD*np.random.randn()
        vel[j]=iv+IV_STD*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            if RATE_CLIP>0.:
                if vel[j] > RATE_CLIP: vel[j] = RATE_CLIP
                elif vel[j] < -RATE_CLIP: vel[j] = -RATE_CLIP
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            if JUMP_PROB>0. and JUMP_STD>0. and np.random.random()<JUMP_PROB:
                pos[j]+=JUMP_STD*np.random.randn()
            pos[j]=max(pos[j],vmin-CLIP_MARGIN); pos[j]=min(pos[j],vmin+(len(gg_p)-1)*step+CLIP_MARGIN)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step)
                dp=abs((gr_v[i]-ep)/gs)
                dpow=dp**GR_POWER
                lp=max(np.exp(-0.5*dpow) if dpow<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es=_interp1(gg_s,pos[j],vmin,step)
                    ds=abs((gr_sm_v[i]-es)/(gs*1.5))
                    dsp=ds**GR_POWER
                    ls=max(np.exp(-0.5*dsp) if dsp<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005)
            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_

# Dense grid for O(1) typewell lookup
def _smooth_radius_values(vals, fb, r):
    r = int(r)
    s = pd.Series(vals, dtype='float32').interpolate(limit_direction='both').fillna(float(fb))
    if r <= 0:
        return s.to_numpy(np.float32)
    return s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt,tw_gr,step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
    tvt_g=np.arange(tmin,tmax+step,step)
    return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)

def _gr_sig(hw,tw_tvt,tw_gr,p):
    # Match pf_tuning_4_early_resume: estimate sigma using the same smoothed
    # horizontal GR and typewell GR that PF likelihood will use.
    gr_fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    hgr=_smooth_radius_values(hw['GR'].astype(float).to_numpy(), gr_fb, p["PF_HGR_SMOOTH_R"])
    kn=hw[hw['TVT_input'].notna()]
    if len(kn)<20:
        gs=float(p["PF_GR_SIG_DEF"])
    else:
        kpos=hw.index.get_indexer(kn.index)
        resid=hgr[kpos]-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)
        gs=float(np.nanstd(resid))
        if not np.isfinite(gs) or gs<=0:
            gs=float(p["PF_GR_SIG_DEF"])
    return float(np.clip(gs*p["PF_GR_SIG_MULT"],p["PF_GR_SIG_MIN"],p["PF_GR_SIG_MAX"]))

def _nn(arr,v):
    i=int(np.searchsorted(arr,v,'left'))
    if i>=len(arr): return len(arr)-1
    if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
    return i

def _smooth(vals,fb,r):
    s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
    si=_nn(tw_tvt,start_tvt)
    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
    path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
    return tw_tvt[path].astype(np.float32)

def run_pf_ancc(hw,tw_tvt,tw_gr,p):
    tw_fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf=_smooth_radius_values(tw_gr, tw_fb, p["PF_TW_GR_SMOOTH_R"]).astype(np.float64)
    gs=_gr_sig(hw,tw_tvt,tw_gr_pf,p)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr_pf)
    md_ev=ev['MD'].values.astype(np.float64)
    z_ev=ev['Z'].values.astype(np.float64)
    # Tuning notebook interpolates + optionally smooths horizontal GR before likelihood.
    gr_fb=tw_fb
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(gr_fb).to_numpy(dtype=float)
    gr_proc=_smooth_radius_values(gr_full, gr_fb, p["PF_HGR_SMOOTH_R"])
    ev_pos=hw.index.get_indexer(ev.index)
    gr_ev=gr_proc[ev_pos].astype(np.float64)

    preds=[]; stds=[]; liks=[]
    for s in range(int(p["PF_N_SEEDS"])):
        pts,std,ll=_pf_ancc(md_ev,z_ev,gr_ev,gg,gmin,gst,
                            gs,ls,ir,int(p["ANCC_N"]),
                            p["ANCC_ALPHA"],p["ANCC_RN"],p["ANCC_PN"],p["ANCC_IR"],p["ANCC_IS"],p["ANCC_RP"],p["ANCC_RR"],
                            p["PF_RESAMP"],p["PF_RATE_CLIP"],p["PF_GR_POWER"],p["PF_JUMP_PROB"],p["PF_JUMP_STD"],
                            p["PF_TVT_CLIP_MARGIN"],int(p["PF_SEED0"])+s)
        preds.append(pts); stds.append(std); liks.append(ll)
    pred_arr=np.stack(preds,0)
    std_arr=np.stack(stds,0)
    lik_arr=np.asarray(liks,dtype=np.float64)
    lik_arr=lik_arr-np.nanmax(lik_arr)
    ww=np.exp(lik_arr/max(float(p["PF_LIKELIHOOD_SCALE"]),1e-6))
    ww=ww/max(float(ww.sum()),1e-300)
    ens=(ww[:,None]*pred_arr).sum(0)
    ens_var=(ww[:,None]*(std_arr**2+(pred_arr-ens[None,:])**2)).sum(0)
    return ens.astype(np.float32),np.sqrt(np.maximum(ens_var,0.)).astype(np.float32)

def run_pf_z(hw,tw_tvt,tw_gr,p):
    tw_fb=float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf=_smooth_radius_values(tw_gr, tw_fb, p["PF_TW_GR_SMOOTH_R"]).astype(np.float64)
    gs=_gr_sig(hw,tw_tvt,tw_gr_pf,p)
    tw_s=pd.Series(tw_gr_pf).rolling(p["PF_GR_WIN"],center=True,min_periods=1).mean().values.astype(np.float32)
    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
    if m2.sum()>=10:
        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
        A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
    else: beta,icpt,zsig=-1.,0.,0.1
    t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr_pf)
    gs2,_,_=_grid(tw_tvt,tw_s)
    gr_fb=tw_fb
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(gr_fb).to_numpy(dtype=float)
    gr_proc=_smooth_radius_values(gr_full, gr_fb, p["PF_HGR_SMOOTH_R"])
    gr_sm=pd.Series(gr_proc).rolling(p["PF_GR_WIN"],center=True,min_periods=1).mean().to_numpy(dtype=float)
    ev_pos=hw.index.get_indexer(ev.index)
    pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                   gr_proc[ev_pos].astype(np.float64),
                   gr_sm[ev_pos].astype(np.float64),
                   gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                   beta,icpt,zsig,int(p["PF_N"]),
                   p["PF_MOM"],p["PF_VN"],p["PF_PN"],p["PF_GR_WT"],p["PF_ROUGH_P"],p["PF_ROUGH_V"],p["PF_RESAMP"],
                   p["PF_INIT_SPR"],p["PF_INIT_V_STD"],p["PF_RATE_CLIP"],p["PF_GR_POWER"],p["PF_JUMP_PROB"],p["PF_JUMP_STD"],
                   p["PF_TVT_CLIP_MARGIN"],int(p["PF_SEED0"]))
    return pts.astype(np.float32),std.astype(np.float32)


_md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)
_gg=np.linspace(45,55,100,np.float64)
_p0=PF_PARAM_SETS[0]
_pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,
         _p0["ANCC_ALPHA"],_p0["ANCC_RN"],_p0["ANCC_PN"],_p0["ANCC_IR"],_p0["ANCC_IS"],_p0["ANCC_RP"],_p0["ANCC_RR"],_p0["PF_RESAMP"],
         _p0["PF_RATE_CLIP"],_p0["PF_GR_POWER"],_p0["PF_JUMP_PROB"],_p0["PF_JUMP_STD"],_p0["PF_TVT_CLIP_MARGIN"],123)
_pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,
      _p0["PF_MOM"],_p0["PF_VN"],_p0["PF_PN"],_p0["PF_GR_WT"],_p0["PF_ROUGH_P"],_p0["PF_ROUGH_V"],_p0["PF_RESAMP"],_p0["PF_INIT_SPR"],_p0["PF_INIT_V_STD"],
      _p0["PF_RATE_CLIP"],_p0["PF_GR_POWER"],_p0["PF_JUMP_PROB"],_p0["PF_JUMP_STD"],_p0["PF_TVT_CLIP_MARGIN"],123)
_beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)

def robust_slope(x,y,w=None):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def affine_cal(kgr,tw_at_k,min_pts=20):
    v=np.isfinite(kgr)&np.isfinite(tw_at_k)
    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
        return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

def seg_b_well(ktvt,kz,form_col):
    """Segment b_well: early/mid/late thirds + full prefix.
    Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness."""
    bv=ktvt+kz-form_col; n=len(bv)
    b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    t1,t2=n//3, 2*n//3
    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
    # WLS (tail-upweighted)
    w=np.exp(0.02*np.arange(n)); w/=w.sum()
    b_wls=float(np.dot(w,bv))
    return b_full,b_early,b_mid,b_late,b_wls

def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
    """Multi-scale NCC. Returns score-weighted ensemble + per-scale signals."""
    out=[]
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
    # Score-weighted ensemble (NEW: softmax-weighted combination)
    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
    sc_ens=(tvts*sw).sum(1).astype(np.float32)
    return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble

class FormationPlaneKNN:
    def __init__(self,well_ids,data_dir):
        rows=[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
            except: continue
            if len(df)==0: continue
            row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
            rows.append(row)
        self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
        xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
        self.tree=cKDTree(xy/self.scale)
        self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
        self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self,xy_q,self_wid=None,k=PLANE_K):
        q=xy_q/self.scale; nf=min(k+5,len(self.df))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
        xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
        A=np.zeros((len(q),3,3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
        try: coef=np.linalg.solve(A,rhs)
        except:
            coef=np.zeros((len(q),3,6))
            for r in range(len(q)):
                try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq=xy_q[:,0]; Yq=xy_q[:,1]
        pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)]=self.fa.mean(0)
        return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)

class DenseANCCImputer:
    def __init__(self,well_ids,data_dir,spw=DENSE_SPW):
        xs,ys,anccs,wids=[],[],[],[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
            except: continue
            if len(df)==0: continue
            ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
        self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
        self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)
        self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
        self.tree=cKDTree(self.xy/self.scale)

    def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
        xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
        sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
        ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
        var=((an-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)

hw_paths=sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
FI=FormationPlaneKNN(train_wids,CFG.dataset_path / "train")
DI=DenseANCCImputer(train_wids,CFG.dataset_path / "train")

_FI=FI; _DI=DI
ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)


def _rolling_mean_np(x, w):
    return pd.Series(np.asarray(x, np.float32)).rolling(w, center=True, min_periods=1).mean().to_numpy(np.float32)

def _rolling_std_np(x, w):
    return pd.Series(np.asarray(x, np.float32)).rolling(w, center=True, min_periods=1).std().fillna(0.).to_numpy(np.float32)

def _safe_grad(y, x=None):
    y = np.asarray(y, np.float32)
    n = len(y)
    if n <= 1:
        return np.zeros(n, np.float32)
    if x is None:
        g = np.gradient(y).astype(np.float32)
    else:
        x = np.asarray(x, np.float32)
        dx = np.gradient(x).astype(np.float32)
        dx = np.where(np.abs(dx) < 1e-6, np.nan, dx)
        g = np.gradient(y).astype(np.float32) / dx
        g = np.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return g.astype(np.float32)

def _add_path_shape_features(feats, prefix, path, md=None):
    """Local geometry of a candidate TVT path. No target is used."""
    path = np.asarray(path, np.float32)
    rate = _safe_grad(path, md)
    acc = _safe_grad(rate, md)
    feats[f'{prefix}_rate'] = rate.astype(np.float32)
    feats[f'{prefix}_acc'] = acc.astype(np.float32)
    feats[f'{prefix}_rate_abs'] = np.abs(rate).astype(np.float32)
    feats[f'{prefix}_acc_abs'] = np.abs(acc).astype(np.float32)
    feats[f'{prefix}_rate_rmean21'] = _rolling_mean_np(rate, 21)
    feats[f'{prefix}_rate_rstd21'] = _rolling_std_np(rate, 21)
    feats[f'{prefix}_acc_rmean21'] = _rolling_mean_np(acc, 21)

def _add_gr_match_features(feats, prefix, path, hgr, tw_tvt, tw_gr, offs):
    """How well a candidate TVT path explains the observed horizontal GR."""
    path = np.asarray(path, np.float32)
    hgr = np.asarray(hgr, np.float32)
    offs = np.asarray(offs, np.float32)
    gr_mat = np.stack([np.interp(path + float(o), tw_tvt, tw_gr) for o in offs], axis=1).astype(np.float32)
    resid = (hgr[:, None] - gr_mat).astype(np.float32)
    abs_resid = np.abs(resid)
    zero_idx = int(np.argmin(np.abs(offs)))
    best_idx = abs_resid.argmin(axis=1)
    row_idx = np.arange(len(path))
    r0 = resid[:, zero_idx].astype(np.float32)
    best_signed = resid[row_idx, best_idx].astype(np.float32)
    feats[f'{prefix}_gr_resid0'] = r0
    feats[f'{prefix}_gr_abs0'] = np.abs(r0).astype(np.float32)
    feats[f'{prefix}_gr_abs_min'] = abs_resid[row_idx, best_idx].astype(np.float32)
    feats[f'{prefix}_gr_best_off'] = offs[best_idx].astype(np.float32)
    feats[f'{prefix}_gr_best_resid'] = best_signed
    feats[f'{prefix}_gr_resid0_rmean21'] = _rolling_mean_np(r0, 21)
    feats[f'{prefix}_gr_resid0_rstd21'] = _rolling_std_np(r0, 21)
    feats[f'{prefix}_gr_abs0_rmean21'] = _rolling_mean_np(np.abs(r0), 21)

def _pairwise_abs_stats(mat):
    mat = np.asarray(mat, np.float32)
    k = mat.shape[1]
    if k < 2:
        z = np.zeros(mat.shape[0], np.float32)
        return z, z
    vals = []
    for i in range(k):
        for j in range(i + 1, k):
            vals.append(np.abs(mat[:, i] - mat[:, j]).astype(np.float32))
    diffs = np.stack(vals, axis=1)
    return diffs.mean(axis=1).astype(np.float32), diffs.max(axis=1).astype(np.float32)

def _corr_summary(mat):
    mat = np.asarray(mat, np.float32)
    k = mat.shape[1]
    if k < 2:
        return 0.0, 0.0, 0.0
    vals = []
    for i in range(k):
        xi = mat[:, i]
        xi_std = float(np.std(xi))
        for j in range(i + 1, k):
            xj = mat[:, j]
            xj_std = float(np.std(xj))
            if xi_std < 1e-6 or xj_std < 1e-6:
                vals.append(0.0)
            else:
                vals.append(float(np.corrcoef(xi, xj)[0, 1]))
    vals = np.nan_to_num(np.asarray(vals, np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return float(vals.mean()), float(vals.min()), float(vals.max())


def _add_pf_feature_columns(feats, pf_runs, last_tvt, tvt_f_ancc, tvt_dense, hgr, tw_tvt, tw_gr, sc,
                            hmd=None, beam_mean=None, sc_ens=None, hyb_ref=None):
    """Add PF-derived feature columns for all parameter sets.

    Added feature families:
      - per-PF GR residual quality around the candidate path
      - per-PF local rate/curvature of the candidate path
      - pairwise PF differences/correlations and PF ensemble consensus
      - PF ensemble disagreement vs beam/sc/spatial/dense references

    Naming rule:
      - set 1 -> suffix _1, set 2 -> suffix _2, ...
      - if KEEP_FIRST_PF_ALIAS=True, also add old unsuffixed columns from set 1.
    """
    def add_one(r, suffix):
        pf_a = r["pf_ancc"].astype(np.float32)
        std_a = r["pf_ancc_std"].astype(np.float32)
        has_z = bool(r["has_z"])
        pf_z = r["pf_z"].astype(np.float32) if has_z else sc(last_tvt)
        feats[f'pf_ancc{suffix}'] = pf_a
        feats[f'pf_ancc_std{suffix}'] = std_a
        feats[f'pf_ancc_delta{suffix}'] = (pf_a - np.float32(last_tvt)).astype(np.float32)
        feats[f'pf_z{suffix}'] = pf_z
        feats[f'pf_z_delta{suffix}'] = (pf_z - np.float32(last_tvt)).astype(np.float32) if has_z else sc(0.)
        feats[f'pf_vs_z{suffix}'] = (pf_a - pf_z).astype(np.float32) if has_z else sc(0.)
        feats[f'pf_vs_spatial{suffix}'] = (pf_a - tvt_f_ancc).astype(np.float32)
        feats[f'pf_vs_dense{suffix}'] = (pf_a - tvt_dense).astype(np.float32)
        for o in PF_OFFS:
            feats[f'tdpf{int(o)}{suffix}'] = hgr - np.interp(pf_a + o, tw_tvt, tw_gr).astype(np.float32)

        prefix = 'pf' if suffix == '' else f'pf{suffix}'
        _add_gr_match_features(feats, prefix, pf_a, hgr, tw_tvt, tw_gr, PF_OFFS)
        if hmd is not None:
            _add_path_shape_features(feats, prefix, pf_a, hmd)

    if KEEP_FIRST_PF_ALIAS and len(pf_runs) > 0:
        add_one(pf_runs[0], "")
    for r in pf_runs:
        add_one(r, r["suffix"])

    if len(pf_runs) >= 2:
        pf_stack = np.stack([r["pf_ancc"].astype(np.float32) for r in pf_runs], axis=1)
        pf_mean = pf_stack.mean(1).astype(np.float32)
        pf_med = np.median(pf_stack, axis=1).astype(np.float32)
        pf_pair_mean, pf_pair_max = _pairwise_abs_stats(pf_stack)
        c_mean, c_min, c_max = _corr_summary(pf_stack)
        feats['pf_ancc_mean'] = pf_mean
        feats['pf_ancc_med'] = pf_med
        feats['pf_ancc_std_between'] = pf_stack.std(1).astype(np.float32)
        feats['pf_ancc_range_between'] = (pf_stack.max(1) - pf_stack.min(1)).astype(np.float32)
        feats['pf_ancc_iqr_between'] = (np.percentile(pf_stack, 75, axis=1) - np.percentile(pf_stack, 25, axis=1)).astype(np.float32)
        feats['pf_ancc_pair_absmean_between'] = pf_pair_mean
        feats['pf_ancc_pair_absmax_between'] = pf_pair_max
        feats['pf_ancc_mean_delta'] = (pf_mean - np.float32(last_tvt)).astype(np.float32)
        feats['pf_ancc_med_delta'] = (pf_med - np.float32(last_tvt)).astype(np.float32)
        feats['pf_ancc_corr_mean'] = sc(c_mean)
        feats['pf_ancc_corr_min'] = sc(c_min)
        feats['pf_ancc_corr_max'] = sc(c_max)

        for i in range(len(pf_runs)):
            si = pf_runs[i]['suffix']
            feats[f'pf_dev_mean{si}'] = (pf_stack[:, i] - pf_mean).astype(np.float32)
            feats[f'pf_dev_med{si}'] = (pf_stack[:, i] - pf_med).astype(np.float32)
        for i in range(len(pf_runs)):
            for j in range(i + 1, len(pf_runs)):
                si = pf_runs[i]['suffix'].replace('_', '')
                sj = pf_runs[j]['suffix'].replace('_', '')
                diff = (pf_stack[:, i] - pf_stack[:, j]).astype(np.float32)
                feats[f'pf_diff_{si}_{sj}'] = diff
                feats[f'pf_absdiff_{si}_{sj}'] = np.abs(diff).astype(np.float32)

        _add_gr_match_features(feats, 'pf_mean', pf_mean, hgr, tw_tvt, tw_gr, PF_OFFS)
        _add_gr_match_features(feats, 'pf_med', pf_med, hgr, tw_tvt, tw_gr, PF_OFFS)
        if hmd is not None:
            _add_path_shape_features(feats, 'pf_mean', pf_mean, hmd)
            _add_path_shape_features(feats, 'pf_med', pf_med, hmd)

        if beam_mean is not None:
            feats['pf_mean_vs_beam_mean'] = (pf_mean - np.asarray(beam_mean, np.float32)).astype(np.float32)
            feats['pf_med_vs_beam_mean'] = (pf_med - np.asarray(beam_mean, np.float32)).astype(np.float32)
        if sc_ens is not None:
            feats['pf_mean_vs_sc_ens'] = (pf_mean - np.asarray(sc_ens, np.float32)).astype(np.float32)
            feats['pf_med_vs_sc_ens'] = (pf_med - np.asarray(sc_ens, np.float32)).astype(np.float32)
        if hyb_ref is not None:
            feats['pf_mean_vs_hyb'] = (pf_mean - np.asarray(hyb_ref, np.float32)).astype(np.float32)
        feats['pf_mean_vs_spatial'] = (pf_mean - tvt_f_ancc).astype(np.float32)
        feats['pf_mean_vs_dense'] = (pf_mean - tvt_dense).astype(np.float32)

        z_list = [r["pf_z"].astype(np.float32) for r in pf_runs if bool(r["has_z"])]
        if len(z_list) >= 2:
            z_stack = np.stack(z_list, axis=1)
            z_mean = z_stack.mean(1).astype(np.float32)
            z_med = np.median(z_stack, axis=1).astype(np.float32)
            z_pair_mean, z_pair_max = _pairwise_abs_stats(z_stack)
            zc_mean, zc_min, zc_max = _corr_summary(z_stack)
            feats['pf_z_mean'] = z_mean
            feats['pf_z_med'] = z_med
            feats['pf_z_std_between'] = z_stack.std(1).astype(np.float32)
            feats['pf_z_range_between'] = (z_stack.max(1) - z_stack.min(1)).astype(np.float32)
            feats['pf_z_pair_absmean_between'] = z_pair_mean
            feats['pf_z_pair_absmax_between'] = z_pair_max
            feats['pf_z_corr_mean'] = sc(zc_mean)
            feats['pf_z_corr_min'] = sc(zc_min)
            feats['pf_z_corr_max'] = sc(zc_max)
            feats['pf_mean_vs_z_mean'] = (pf_mean - z_mean).astype(np.float32)
            feats['pf_med_vs_z_med'] = (pf_med - z_med).astype(np.float32)


def build_well(hw_path,tw_path,is_train):
    global _FI,_DI
    wid=Path(hw_path).stem.replace('__horizontal_well','')
    try:
        hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
    except: return None
    if is_train and 'TVT' not in hw.columns: return None
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)<10: return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
    if len(tw_tvt)<3: return None

    pf_runs=[]
    for pconf in PF_PARAM_SETS:
        pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr,pconf)
        if len(pf_a)==0: return None
        pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr,pconf)
        has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))
        pf_runs.append({
            "name": pconf["name"],
            "suffix": pconf["suffix"],
            "pf_ancc": pf_a.astype(np.float32),
            "pf_ancc_std": std_a.astype(np.float32),
            "pf_z": pf_z.astype(np.float32) if len(pf_z)==len(pf_a) else np.full(len(pf_a), np.nan, np.float32),
            "pf_z_std": std_z.astype(np.float32) if len(std_z)==len(pf_a) else np.full(len(pf_a), np.nan, np.float32),
            "has_z": has_z,
        })
    pf_primary=pf_runs[0]
    pf_use=pf_primary["pf_ancc"].astype(np.float32)
    std_use=pf_primary["pf_ancc_std"].astype(np.float32)
    pf_z=pf_primary["pf_z"].astype(np.float32)
    has_z=bool(pf_primary["has_z"])

    lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)

    # 7 beams (Numba JIT ±2)
    bpaths={}
    for (bs,mc,es,r,tag) in BEAMS:
        bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
    # Robust reference beam: older versions used tags 'cons'/'sm5', this notebook uses 'gr'/'smooth'.
    _beam_a=bpaths['cons'] if 'cons' in bpaths else bpaths.get('gr', next(iter(bpaths.values())))
    _beam_b=bpaths['sm5'] if 'sm5' in bpaths else bpaths.get('smooth', _beam_a)
    beam_ref=(_beam_a+_beam_b)/2.

    # Multi-scale NCC → score-weighted ensemble
    ktvt=kn['TVT_input'].to_numpy(np.float32)
    sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
    sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
    sc_cons=(sc8+sc15+sc25)/3.
    sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
    hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens  # use ensemble not single

    tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
    a_cal,b_cal=affine_cal(kgr,tw_at_k)
    kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
    pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
    slp_z=robust_slope(kz,ktvt)

    swid=wid if is_train else None
    xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
    form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
    form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
    z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)

    # Per-formation: segment b_well (early/mid/late/wls) + TVT + known-zone RMSE
    tvt_fs={}; form_rmse={}; form_list=[]
    for fi2,fn in enumerate(FORMATIONS):
        b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
        tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)
        tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)
        tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)
        tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw
        tvt_fs[f'tvtF50_{fn}']=tvt_f50
        tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
        tvt_fs[f'bw50_{fn}']=np.float32(b_late)
        tvt_fs[f'bw_early_{fn}']=np.float32(b_early)   # NEW: early segment
        tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)       # NEW: mid segment
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
        form_list.append(tvt_f)

    fs=np.stack(form_list,1)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
    form_std_d =fs.std(1).astype(np.float32)
    form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)

    d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
    d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
    b_vd=ktvt+z_kn-d_kn
    _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)
    b_d=float(np.median(b_vd))
    tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)
    tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)
    tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)
    res_kn=ktvt+z_kn-d_kn
    d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))

    candidate_names = [f"pf_ancc{r['suffix']}" for r in pf_runs] + [f"beam_{k}" for k in bpaths.keys()] + [
        'sc8', 'sc15', 'sc25', 'sc_ens', 'spatial_ancc', 'dense_ancc'
    ]
    all_sigs=[r['pf_ancc'] for r in pf_runs]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
    sig_mat=np.stack(all_sigs,1).astype(np.float32)
    sig_std=sig_mat.std(1).astype(np.float32)
    sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)
    sig_med=np.median(sig_mat,axis=1).astype(np.float32)
    sig_q10=np.percentile(sig_mat,10,axis=1).astype(np.float32)
    sig_q25=np.percentile(sig_mat,25,axis=1).astype(np.float32)
    sig_q75=np.percentile(sig_mat,75,axis=1).astype(np.float32)
    sig_q90=np.percentile(sig_mat,90,axis=1).astype(np.float32)
    sig_iqr=(sig_q75-sig_q25).astype(np.float32)
    sig_range=(sig_mat.max(1)-sig_mat.min(1)).astype(np.float32)
    sig_mad=np.mean(np.abs(sig_mat-sig_med[:,None]),axis=1).astype(np.float32)
    sig_centered=(sig_mat-sig_mat.mean(1,keepdims=True)).astype(np.float32)
    sig_skew=(np.mean(sig_centered**3,axis=1)/(np.maximum(sig_std,1e-6)**3)).astype(np.float32)

    sig_gr_pred_mat=np.stack([np.interp(sig_mat[:,j],tw_tvt,tw_gr) for j in range(sig_mat.shape[1])],axis=1).astype(np.float32)
    sig_gr_resid_mat=(hgr[:,None]-sig_gr_pred_mat).astype(np.float32)
    sig_gr_abs_mat=np.abs(sig_gr_resid_mat).astype(np.float32)
    sig_gr_best_idx=sig_gr_abs_mat.argmin(axis=1)
    _sig_rows=np.arange(len(hgr))
    sig_gr_best_path=sig_mat[_sig_rows,sig_gr_best_idx].astype(np.float32)
    sig_gr_best_abs=sig_gr_abs_mat[_sig_rows,sig_gr_best_idx].astype(np.float32)
    sig_gr_best_resid=sig_gr_resid_mat[_sig_rows,sig_gr_best_idx].astype(np.float32)
    sig_gr_pred_std=sig_gr_pred_mat.std(1).astype(np.float32)
    sig_gr_resid_mean=sig_gr_resid_mat.mean(1).astype(np.float32)
    sig_gr_resid_std=sig_gr_resid_mat.std(1).astype(np.float32)

    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [5,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1)
        rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)
                   ).iloc[ev.index].values.astype(np.float32)

    hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
    slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

    mdd=hw['MD'].diff().replace(0,np.nan)
    dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh,np.float32(v),np.float32)

    feats={
        'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
        'last_known_tvt':sc(last_tvt),
        **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
        'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
        'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
        'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),  # score-weighted ensemble
        'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        'sig_std':sig_std,'sig_mean_d':sig_mean,
        'sig_med_d':(sig_med-np.float32(last_tvt)).astype(np.float32),
        'sig_q10_d':(sig_q10-np.float32(last_tvt)).astype(np.float32),
        'sig_q25_d':(sig_q25-np.float32(last_tvt)).astype(np.float32),
        'sig_q75_d':(sig_q75-np.float32(last_tvt)).astype(np.float32),
        'sig_q90_d':(sig_q90-np.float32(last_tvt)).astype(np.float32),
        'sig_iqr':sig_iqr,'sig_range':sig_range,'sig_mad':sig_mad,'sig_skew':sig_skew,
        'sig_gr_pred_std':sig_gr_pred_std,
        'sig_gr_resid_mean':sig_gr_resid_mean,'sig_gr_resid_std':sig_gr_resid_std,
        'sig_gr_best_d':(sig_gr_best_path-np.float32(last_tvt)).astype(np.float32),
        'sig_gr_best_idx':sig_gr_best_idx.astype(np.float32),
        'sig_gr_best_abs':sig_gr_best_abs,'sig_gr_best_resid':sig_gr_best_resid,
        'sig_gr_best_vs_med':(sig_gr_best_path-sig_med).astype(np.float32),
        **tvt_fs,
        **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
        'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),
        'spatial_knn_dist':knn_d,
        'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
        'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
        'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
        'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
        'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
        'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
        'beam_vs_spatial':(beam_ref-tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam':(sc_ens-beam_ref).astype(np.float32),
        'cal_a':sc(a_cal),'cal_b':sc(b_cal),
        'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
        'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
        'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
        'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),
        'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
        'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
        'z':z_ev,
        'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
        'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
        'dz':(z_ev-float(lk['Z'])).astype(np.float32),
        'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
        'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
        'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
        'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
        'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
        'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
    }
    _add_pf_feature_columns(
        feats, pf_runs, last_tvt, tvt_fs['tvtF_ANCC'], tvt_dense, hgr, tw_tvt, tw_gr, sc,
        hmd=hmd,
        beam_mean=(np.float32(last_tvt) + feats['beam_mean_d']).astype(np.float32),
        sc_ens=sc_ens,
        hyb_ref=hyb_ref,
    )

    # Shape/GR-match features for non-PF ensemble references.
    beam_mean_path=(np.float32(last_tvt)+feats['beam_mean_d']).astype(np.float32)
    beam_med_path=(np.float32(last_tvt)+feats['beam_med_d']).astype(np.float32)
    _add_path_shape_features(feats,'beam_mean',beam_mean_path,hmd)
    _add_path_shape_features(feats,'beam_med',beam_med_path,hmd)
    _add_path_shape_features(feats,'sc_ens',sc_ens,hmd)
    _add_path_shape_features(feats,'hyb',hyb_ref,hmd)
    _add_path_shape_features(feats,'sig_med',sig_med,hmd)
    _add_path_shape_features(feats,'sig_gr_best',sig_gr_best_path,hmd)
    _add_gr_match_features(feats,'beam_mean',beam_mean_path,hgr,tw_tvt,tw_gr,BEAM_OFFS)
    _add_gr_match_features(feats,'beam_med',beam_med_path,hgr,tw_tvt,tw_gr,BEAM_OFFS)
    _add_gr_match_features(feats,'sc_ens',sc_ens,hgr,tw_tvt,tw_gr,SC_OFFS)
    _add_gr_match_features(feats,'hyb',hyb_ref,hgr,tw_tvt,tw_gr,SC_OFFS)
    _add_gr_match_features(feats,'sig_med',sig_med,hgr,tw_tvt,tw_gr,SC_OFFS)
    _add_gr_match_features(feats,'sig_gr_best',sig_gr_best_path,hgr,tw_tvt,tw_gr,SC_OFFS)

    for k,v in rolls.items(): feats[k]=v

    # Additional GR morphology features. These use only observed horizontal/typewell GR.
    feats['gr_z21']=((hgr-rolls['grm21'])/(rolls['grs21']+1e-3)).astype(np.float32)
    feats['gr_z51']=((hgr-rolls['grm51'])/(rolls['grs51']+1e-3)).astype(np.float32)
    feats['gr_abs_d1']=np.abs(gr_d1).astype(np.float32)
    feats['gr_abs_d2']=np.abs(gr_d2).astype(np.float32)
    feats['gr_d1_rmean21']=_rolling_mean_np(gr_d1,21)
    feats['gr_abs_d1_rmean21']=_rolling_mean_np(np.abs(gr_d1),21)
    feats['gr_rough21']=_rolling_mean_np(np.abs(gr_d1),21)
    feats['gr_rough51']=_rolling_mean_np(np.abs(gr_d1),51)
    _tw_gr_sorted=np.sort(tw_gr.astype(np.float32))
    _kg_sorted=np.sort(kgr.astype(np.float32)) if len(kgr)>0 else _tw_gr_sorted
    feats['gr_pct_tw']=(np.searchsorted(_tw_gr_sorted,hgr,side='right')/max(len(_tw_gr_sorted),1)).astype(np.float32)
    feats['gr_pct_known']=(np.searchsorted(_kg_sorted,hgr,side='right')/max(len(_kg_sorted),1)).astype(np.float32)
    feats['gr_z_known']=((hgr-np.float32(np.mean(kgr)))/(np.float32(np.std(kgr))+1e-3)).astype(np.float32)
    feats['gr_z_tw']=((hgr-np.float32(np.mean(tw_gr)))/(np.float32(np.std(tw_gr))+1e-3)).astype(np.float32)
    feats['gr_vs_known_last']=(hgr-np.float32(kgr[-1])).astype(np.float32)
    feats['gr_vs_known_mean']=(hgr-np.float32(np.mean(kgr))).astype(np.float32)

    # Geometry roughness of the actual drilled path.
    step_xy=np.sqrt(dxdmd**2+dydmd**2).astype(np.float32)
    feats['xy_rate']=step_xy
    feats['xyz_rate']=np.sqrt(dxdmd**2+dydmd**2+dzdmd**2).astype(np.float32)
    feats['z_rate_abs']=np.abs(dzdmd).astype(np.float32)
    feats['dzdmd_rmean21']=_rolling_mean_np(dzdmd,21)
    feats['dzdmd_rstd21']=_rolling_std_np(dzdmd,21)
    feats['xy_rate_rmean21']=_rolling_mean_np(step_xy,21)
    feats['xy_rate_rstd21']=_rolling_std_np(step_xy,21)

    result=pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
    return result

def build_dataset(paths,is_train,label):
    args=[(str(p),str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'),is_train)
          for p in paths
          if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
    t0=time.time()
    res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
        delayed(build_well)(hp,tp,it) for hp,tp,it in args)
    parts=[r for r in res if r is not None]
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

# ==================== v26 smooth PF エンジン(固定ラグ, tune_smoother_r0_v25 と同一) ====================
def build_smoother_inputs(hw, tw_tvt, tw_gr, P):
    """smooth PF 入力を hw/tw と生PFパラメータから作る(v9の _smooth_radius_values/_grid を再利用)。無効なら None。"""
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0:
        return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    tw_gr_pf = _smooth_radius_values(tw_gr, tw_fb, P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(tw_tvt, tw_gr_pf)
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_sm = _smooth_radius_values(gr_full, tw_fb, P["hgr_smooth_r"])
    ev_pos = hw.index.get_indexer(ev.index); kpos = hw.index.get_indexer(kn.index)
    kn_tvtin = kn["TVT_input"].to_numpy(float)
    if len(kpos) < 20:
        gs = P["gr_sig_def"]
    else:
        resid = gr_sm[kpos] - np.interp(kn_tvtin, tw_tvt, tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
    gs = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    ls = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values)
    dm = np.diff(tail["MD"].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    return dict(md=ev["MD"].to_numpy(float), z=ev["Z"].to_numpy(float), gr=gr_sm[ev_pos].astype(np.float64),
                gg=gg, gmin=gmin, gst=gst, gs=gs, ls=ls, ir=ir, N=int(P["n_particles"]))

def _v26_smoother_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, P, N, device, gen, dtype=V26_DTYPE):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    L = int(P.get("smooth_lag", 32)); L = max(0, min(L, T))
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS * rn(); rate = ir[:, None] + IR * rn()
    w = torch.full((B, N), 1.0 / N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype) - 1) * step + CLIP
    glast = torch.gather(grid, 1, (glen - 1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    use_sm = L > 0
    if use_sm:
        buf = torch.full((B, L, N), float("nan"), device=device, dtype=dtype); ptr = 0
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur - pm).clamp_min(1.0)
        rate_n = ALPHA * rate + RN * rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n * dm[:, None] + PN * rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD * rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n - zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt + zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos - zi; ii = (v - vmin[:, None]) / step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen - 1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen - 2).clamp_min(0)[:, None])
        t = ii - i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c + 1)
        eg = gA * (1 - t) + gB * t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None] - eg) / gs[:, None]).abs(); dp = d ** GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5 * dp), zero).clamp_min(1e-300)
        avg = (w * lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate - dipp) / PHYS_SIG; lkp = torch.exp(-0.5 * dphys * dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w * lk * lkp, w * lkp)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new / ws.clamp_min(1e-300), torch.full_like(w, 1.0 / N))
        ne = (w * w).sum(1); need = act & ((1.0 / ne) < (RESAMP * N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype) * (1.0 / N)
            u = u0 + arN[None, :] / N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N - 1)
            pos_rs = torch.gather(pos, 1, idx) + RP * rn(); rate_rs = torch.gather(rate, 1, idx) + RR * rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0 / N), w)
            if use_sm:
                buf_g = torch.gather(buf, 2, idx[:, None, :].expand(B, L, N))
                buf = torch.where(need[:, None, None], buf_g, buf)
        pts_f[:, i] = (w * (pos - zi)).sum(1)
        if use_sm:
            if i >= L:
                old = buf[:, ptr, :]; pts_s[:, i - L] = (w * old).sum(1) - z[:, i - L]
            buf[:, ptr, :] = pos; ptr = (ptr + 1) % L
        pm = torch.where(act, cur, pm)
    if use_sm:
        for j in range(max(0, T - L), T):
            pts_s[:, j] = (w * buf[:, j % L, :]).sum(1) - z[:, j]
    else:
        pts_s = pts_f
    return pts_f, pts_s, log_lik

def _v26_pad(inps, device, dtype=V26_DTYPE):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn - 1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G - 1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir))

def run_smoother_ext(inps, P, seed, n_seeds, chunk):
    """各 well: smoothed(seed尤度加重) と std(smoothedのseed間加重std)。forwardはCLAMP基準。"""
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); out = [None] * W
    ls_scale = float(P["likelihood_scale"])
    for c0 in range(0, W, chunk):
        sub = inps[c0:c0 + chunk]; w = len(sub); pad = _v26_pad(sub, V26_DEVICE)
        rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=V26_DEVICE); gen.manual_seed(seed)
        pf, ps, ll = _v26_smoother_core(
            rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
            pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
            pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
            P, N, V26_DEVICE, gen)
        pf = pf.view(w, S, -1).double().cpu().numpy(); ps = ps.view(w, S, -1).double().cpu().numpy()
        ll = ll.view(w, S).cpu().numpy()
        for j in range(w):
            T = len(sub[j]["md"]); llw = ll[j]
            if np.isfinite(llw).any():
                mx = np.nanmax(llw[np.isfinite(llw)]); lk = np.where(np.isfinite(llw), llw - mx, -np.inf)
                ww = np.exp(lk / max(ls_scale, 1e-6)); s = ww.sum(); ww = ww / s if s > 1e-300 else np.full(S, 1.0 / S)
            else:
                ww = np.full(S, 1.0 / S)
            smean = (ww[:, None] * ps[j, :, :T]).sum(0); fmean = (ww[:, None] * pf[j, :, :T]).sum(0)
            sstd = np.sqrt(np.maximum((ww[:, None] * (ps[j, :, :T] - smean[None, :]) ** 2).sum(0), 0.))
            if CLAMP_FT is not None:
                bad = np.abs(smean - fmean) > CLAMP_FT
                if bad.any(): smean = np.where(bad, fmean, smean)
            out[c0 + j] = dict(mean=smean.astype(np.float32), std=sstd.astype(np.float32))
    if V26_DEVICE == "cuda": torch.cuda.synchronize()
    return out


# ==================== データ組み立て(v9 build_well + smooth PF 注入) ====================
# ==================== v28 追加: self/neighbor grid + Z傾斜 + 近傍(train)プール ====================
_SELF_ENS = {}; _NBR_ENS = {}; _ZGRAD = {}; _NBR_META = {}    # wid -> per-eval-row dict(mean,std) / meta

def _bin_grid(t, g, gmin, step, G):
    """(TVT, GR) を grid の TVT ビンに median 集約(疑似typewell)。無データビン=NaN。"""
    m = np.isfinite(t) & np.isfinite(g); t, g = t[m], g[m]
    idx = np.round((t - gmin) / step).astype(int); ok = (idx >= 0) & (idx < G); idx, gg = idx[ok], g[ok]
    out = np.full(G, np.nan)
    if len(gg):
        s = pd.Series(gg).groupby(idx).median(); out[s.index.to_numpy()] = s.to_numpy()
    return out

def build_inputs_self(hw, tw_tvt, tw_gr, P):
    """smooth PF 入力(tw)に、自分の既知prefix GR(TVT_input) を疑似typewell化して gg に上書き。"""
    base = build_smoother_inputs(hw, tw_tvt, tw_gr, P)
    if base is None: return None
    kn = hw[hw["TVT_input"].notna()]
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    gr_sm = _smooth_radius_values(gr_full, tw_fb, P["hgr_smooth_r"])
    kpos = hw.index.get_indexer(kn.index)
    gh = _bin_grid(kn["TVT_input"].to_numpy(float), gr_sm[kpos].astype(float), base["gmin"], base["gst"], len(base["gg"]))
    gg = base["gg"].copy(); have = np.isfinite(gh); gg[have] = SELF_MIX_W * gh[have] + (1 - SELF_MIX_W) * gg[have]
    base = dict(base); base["gg"] = gg; return base

def build_inputs_nbr(hw, tw_tvt, tw_gr, P, refs):
    """smooth PF 入力(tw)に、近傍train坑井の GR(TVT) を疑似typewell化して gg に上書き。refs無し=tw(fallback)。"""
    base = build_smoother_inputs(hw, tw_tvt, tw_gr, P)
    if base is None: return None
    if refs:
        rt = np.concatenate([r[1] for r in refs])
        rg = np.concatenate([_smooth_radius_values(r[0], float(np.nanmean(r[0])), P["hgr_smooth_r"]).astype(float) for r in refs])
        gh = _bin_grid(rt, rg, base["gmin"], base["gst"], len(base["gg"]))
        gg = base["gg"].copy(); have = np.isfinite(gh); gg[have] = NBR_MIX_W * gh[have] + (1 - NBR_MIX_W) * gg[have]
        base = dict(base); base["gg"] = gg
    return base

def z_gradient_eval(hw, r=ZGRAD_R):
    """Z傾斜 dZ/dMD を移動平均(断層影響回避)し eval 行のみ返す。"""
    ev = hw["TVT_input"].isna().to_numpy()
    md = hw["MD"].to_numpy(float); z = hw["Z"].to_numpy(float)
    with np.errstate(all="ignore"):
        g = np.gradient(z, md)
    g = pd.Series(g).rolling(2 * int(r) + 1, center=True, min_periods=1).mean().to_numpy()
    return g[ev].astype(np.float32)

# --- 近傍(train)プール: XY重心 ---
_TRAIN_CENT = None
def _train_centroids():
    global _TRAIN_CENT
    if _TRAIN_CENT is None:
        tdir = DATA_DIR_V26 / "train"
        wids = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(tdir / "*__horizontal_well.csv"))})
        xy = []
        for w in wids:
            h = pd.read_csv(tdir / f"{w}__horizontal_well.csv", usecols=["X", "Y"]); xy.append((float(h["X"].mean()), float(h["Y"].mean())))
        _TRAIN_CENT = (wids, np.array(xy))
    return _TRAIN_CENT
_REFC = {}
def _ref_grtvt(wid):
    if wid not in _REFC:
        h = pd.read_csv(DATA_DIR_V26 / "train" / f"{wid}__horizontal_well.csv", usecols=["GR", "TVT"])
        _REFC[wid] = (h["GR"].to_numpy(float), h["TVT"].to_numpy(float))
    return _REFC[wid]
def _nearest_dist(cx, cy, exclude_wid=None):
    """最近傍 train 坑井までの実距離(上限なし)。フォールバック坑井でも実数(孤立度)を返す。"""
    wids, xy = _train_centroids(); d = np.hypot(xy[:, 0] - cx, xy[:, 1] - cy)
    if exclude_wid is not None:
        keep = np.array([w != exclude_wid for w in wids]); d = d[keep]
    return float(d.min()) if len(d) else float("nan")
def neighbors_of(cx, cy, exclude_wid=None):
    wids, xy = _train_centroids(); d = np.hypot(xy[:, 0] - cx, xy[:, 1] - cy); out = []
    for j in np.argsort(d):
        w = wids[j]
        if w == exclude_wid: continue
        if d[j] > MAX_DIST: break
        out.append((w, float(d[j])))
        if len(out) >= K_MAX: break
    return out


def build_dataset_v28():
    base = DATA_DIR_V26 / SPLIT
    wells = sorted({os.path.basename(q).split("__")[0] for q in glob.glob(str(base / "*__horizontal_well.csv"))})
    if N_WELLS is not None:
        wells = wells[:N_WELLS]
    P_SETS = []
    for k, lab in enumerate(PF_BANK):
        raw = dict(PF_LIB[lab]); raw["n_seeds"] = N_SEED
        p = _make_pf_params(raw, k); p["PF_USE_PHYS"] = bool(raw.get("use_phys", False))
        p["PF_PHYS_SIG"] = float(raw.get("phys_sig", 1.0)); p["label"] = lab; p["_raw"] = PF_LIB[lab]
        P_SETS.append(p)
    print(f"[v28] PF: " + ", ".join(f"{p['label']}->{p['name']}" for p in P_SETS))

    names = []; paths = []
    for w in wells:
        hp = base / f"{w}__horizontal_well.csv"; tp = base / f"{w}__typewell.csv"
        try:
            hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
        except Exception:
            continue
        if IS_TRAIN and "TVT" not in hw.columns:
            continue
        if build_smoother_inputs(hw, tw["TVT"].to_numpy(float), tw["GR"].to_numpy(float), P_SETS[0]["_raw"]) is None:
            continue
        names.append(w); paths.append((hp, tp))
    print(f"[{SPLIT}] 使用坑井: {len(names)}  PF={PF_BANK}(smooth, tw/self/nbr各{len(P_SETS)}回=計{3*len(P_SETS)}回)  lag={SMOOTH_LAG}  n_seed={N_SEED}")
    def _praw(p): return {**p["_raw"], "smooth_lag": SMOOTH_LAG}   # per-param(smooth_lag付き)

    # (1) tw smooth PF: pf_1/2/3 それぞれで実行(v9 build_well 注入用)→ pf_ancc{suffix} が別物に
    _FWD_ENS.clear()
    for p in P_SETS:
        Pk = _praw(p)
        print(f"[smoothPF-tw] {p['label']} ..."); t0 = time.perf_counter()
        inps = []
        for (hp, tp) in paths:
            hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
            inps.append(build_smoother_inputs(hw, tw["TVT"].to_numpy(float), tw["GR"].to_numpy(float), Pk))
        outs = run_smoother_ext(inps, Pk, SEED_V26, N_SEED, WELL_CHUNK)
        for i, w in enumerate(names): _FWD_ENS[(w, p["name"])] = outs[i]
        if V26_DEVICE == "cuda": torch.cuda.empty_cache()
        print(f"      done {time.perf_counter()-t0:.1f}s")

    # (2) self smooth PF(自分prefix grid) × pf_1/2/3
    _SELF_ENS.clear()
    for p in P_SETS:
        Pk = _praw(p)
        print(f"[smoothPF-self] {p['label']} ..."); t0 = time.perf_counter()
        inps = []
        for (hp, tp) in paths:
            hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
            inps.append(build_inputs_self(hw, tw["TVT"].to_numpy(float), tw["GR"].to_numpy(float), Pk))
        outs = run_smoother_ext(inps, Pk, SEED_V26, N_SEED, WELL_CHUNK)
        for i, w in enumerate(names): _SELF_ENS[(w, p["name"])] = outs[i]     # dict(mean,std)
        if V26_DEVICE == "cuda": torch.cuda.empty_cache()
        print(f"      done {time.perf_counter()-t0:.1f}s")

    # (3) neighbor smooth PF × pf_1/2/3(近傍探索はparam非依存で1回、PF実行のみparam別)
    print("[smoothPF-nbr] 近傍探索 ..."); n_nbr = 0; NREFS = {}
    for (hp, tp) in paths:
        wid = os.path.basename(str(hp)).split("__")[0]
        hw = pd.read_csv(hp)
        cx, cy = float(hw["X"].mean()), float(hw["Y"].mean())
        refw = neighbors_of(cx, cy, exclude_wid=(wid if SPLIT == "train" else None))
        use = len(refw) >= NEED_REFS
        NREFS[wid] = [_ref_grtvt(r) for r, _ in refw] if use else []
        _NBR_META[wid] = dict(has=int(use), nn=_nearest_dist(cx, cy, exclude_wid=(wid if SPLIT == "train" else None)), nref=len(refw))
        if use: n_nbr += 1
    print(f"      近傍あり {n_nbr}/{len(names)}(残りは通常実行=tw)")
    _NBR_ENS.clear()
    for p in P_SETS:
        Pk = _praw(p)
        print(f"[smoothPF-nbr] {p['label']} ..."); t0 = time.perf_counter()
        inps = []
        for (hp, tp) in paths:
            wid = os.path.basename(str(hp)).split("__")[0]
            hw = pd.read_csv(hp); tw = pd.read_csv(tp).sort_values("TVT")
            inps.append(build_inputs_nbr(hw, tw["TVT"].to_numpy(float), tw["GR"].to_numpy(float), Pk, NREFS[wid]))
        outs = run_smoother_ext(inps, Pk, SEED_V26, N_SEED, WELL_CHUNK)
        for i, w in enumerate(names): _NBR_ENS[(w, p["name"])] = outs[i]     # dict(mean,std)
        if V26_DEVICE == "cuda": torch.cuda.empty_cache()
        print(f"      done {time.perf_counter()-t0:.1f}s")

    # (4) Z傾斜(移動平均)
    for (hp, tp) in paths:
        _ZGRAD[os.path.basename(str(hp)).split("__")[0]] = z_gradient_eval(pd.read_csv(hp))

    # (5) v9 build_well(tw smooth PF注入) + self/nbr(pf_1/2/3)/z_grad を列追加
    globals()["run_pf_ancc"] = _patched_run_pf_ancc
    from joblib import Parallel, delayed
    def one(w, hp, tp):
        _TLS.wid = w
        df = build_well(str(hp), str(tp), IS_TRAIN)
        if df is None or len(df) == 0: return None
        n = len(df); lk = df["last_known_tvt"].to_numpy(float)
        def col(a):
            out = np.full(n, np.nan, np.float32)
            if a is not None:
                mm = min(len(a), n); out[:mm] = np.asarray(a[:mm], np.float32)
            return out
        mt = _NBR_META.get(w, {})
        S_all = []; N_all = []; T_all = []
        for p in P_SETS:
            suf = p["suffix"]; nm = p["name"]
            twk = df[f"pf_ancc{suf}"].to_numpy(float) if f"pf_ancc{suf}" in df.columns else lk.copy()
            sf = _SELF_ENS.get((w, nm)); nb = _NBR_ENS.get((w, nm))
            ps = col(sf["mean"] if sf else None); pss = col(sf["std"] if sf else None)
            pn = col(nb["mean"] if nb else None); pns = col(nb["std"] if nb else None)
            df[f"pf_self{suf}"] = ps; df[f"pf_self_std{suf}"] = pss; df[f"pf_self_delta{suf}"] = (ps - lk).astype(np.float32)
            df[f"pf_nbr{suf}"]  = pn; df[f"pf_nbr_std{suf}"]  = pns; df[f"pf_nbr_delta{suf}"]  = (pn - lk).astype(np.float32)
            df[f"pf_self_vs_tw{suf}"] = (ps - twk).astype(np.float32)   # ★最有力: baseline tw との不一致
            df[f"pf_nbr_vs_tw{suf}"]  = (pn - twk).astype(np.float32)
            df[f"pf_self_vs_nbr{suf}"] = (ps - pn).astype(np.float32)
            st = np.stack([twk, ps.astype(float), pn.astype(float)], axis=1)
            df[f"pf_agree_std{suf}"]   = np.nanstd(st, axis=1).astype(np.float32)
            df[f"pf_agree_range{suf}"] = (np.nanmax(st, axis=1) - np.nanmin(st, axis=1)).astype(np.float32)
            S_all.append(ps.astype(float)); N_all.append(pn.astype(float)); T_all.append(twk)
        # pf_1/2/3 間の集約(平均 / param間ばらつき / 平均のtw不一致)
        SM = np.stack(S_all, 1); NM = np.stack(N_all, 1); TM = np.stack(T_all, 1)
        df["pf_self_mean"] = np.nanmean(SM, 1).astype(np.float32); df["pf_self_pstd"] = np.nanstd(SM, 1).astype(np.float32)
        df["pf_nbr_mean"]  = np.nanmean(NM, 1).astype(np.float32); df["pf_nbr_pstd"]  = np.nanstd(NM, 1).astype(np.float32)
        df["pf_self_mean_vs_tw"] = (np.nanmean(SM, 1) - np.nanmean(TM, 1)).astype(np.float32)
        df["pf_nbr_mean_vs_tw"]  = (np.nanmean(NM, 1) - np.nanmean(TM, 1)).astype(np.float32)
        # Z傾斜 + 近傍メタ(近傍を実際に使ったか/最近傍距離/本数)
        df["z_grad"] = col(_ZGRAD.get(w))
        df["pf_nbr_has"]    = np.float32(mt.get("has", 0))
        df["pf_nbr_nndist"] = np.float32(mt.get("nn", np.nan))
        df["pf_nbr_nrefs"]  = np.float32(mt.get("nref", 0))
        return df
    print("[build] v9 build_well(smooth PF注入) + self/nbr/z_grad 列 ...")
    t0 = time.perf_counter()
    res = Parallel(n_jobs=N_JOBS_CPU, prefer="threads", verbose=5)(
        delayed(one)(names[i], paths[i][0], paths[i][1]) for i in range(len(names)))
    parts = [r for r in res if r is not None]
    print(f"      done {time.perf_counter()-t0:.1f}s  parts={len(parts)}")
    if not parts:
        raise RuntimeError("有効な坑井がありません。")
    return pd.concat(parts, ignore_index=True)


def main():
    print(f"[v28] device={V26_DEVICE} smooth_lag={SMOOTH_LAG} n_seed={N_SEED} PF=r0 self/nbr/z_grad追加 chunk={WELL_CHUNK}")
    df = build_dataset_v28()
    OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(OUT_PARQUET, index=False)
    cols = list(df.columns)
    print(f"\\nsaved: {OUT_PARQUET}  rows={len(df)} cols={len(cols)}")
    for pre in ["pf_", "beam", "tvtF", "form", "dense", "sc"]:
        print(f"  {pre}* = {sum(c.startswith(pre) for c in cols)} 列")

In [ ]:
# ===== test 上書き + models/後処理 設定 =====
import os, glob, json, joblib, time
from scipy.signal import savgol_filter
SPLIT = "test"; IS_TRAIN = False; N_WELLS = None      # ★build_dataset_v28 はこの global を見る
CANDS_ART = [Path("/kaggle/input/rogii-artifact-v28"),
             Path(r"C:\Users\kosaka256\Documents\rogii_claude\v28\artifacts_v28")]
ART = next((p for p in CANDS_ART if (p / "models").exists() or (p / "features.json").exists()), CANDS_ART[-1])
PP = json.loads((ART / "pp_params.json").read_text()) if (ART / "pp_params.json").exists() else {"alpha":1.0,"tau":85,"w_pf":0.09}
SG_W, SG_P = 17, 3
print("DATA:", _V26_DATA, "| ART:", ART, "| PP:", PP)

In [ ]:
# ===== test 特徴化(v9+smooth PF+self/nbr/z_grad) → 9モデル平均 → apply_pp+sg → submission =====
t0 = time.time()
test_df = build_dataset_v28()          # ← create と同一エンジン(SPLIT=test)
print("test rows:", len(test_df), f"({time.time()-t0:.0f}s)")

features = json.loads((ART / "features.json").read_text())
for c in features:
    if c not in test_df.columns: test_df[c] = 0.0
Xte = test_df[features].astype("float32")

model_dirs = sorted([Path(p) for p in glob.glob(str(ART / "models" / "*")) if os.path.isdir(p)])
preds = []
for d in model_dirs:
    fps = sorted(glob.glob(str(d / "fold*.pkl")))
    if not fps: continue
    preds.append(np.mean([joblib.load(fp).predict(Xte) for fp in fps], axis=0)); print(f"  {d.name}: {len(fps)} folds")
res = np.mean(preds, axis=0).astype(np.float64)
print("models used:", len(preds))

lk  = test_df["last_known_tvt"].to_numpy(np.float64)
mds = test_df["md_since"].to_numpy(np.float64)
pf_res = test_df["pf_ancc"].to_numpy(np.float64) - lk if "pf_ancc" in test_df.columns else np.zeros(len(test_df))
d = res * (1 - PP["w_pf"]) + pf_res * PP["w_pf"]
if PP.get("tau"): d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(PP["tau"])))
d = d * float(PP.get("alpha", 1.0)); tvt = lk + d
out = tvt.copy(); wcol = test_df["well"].to_numpy()
for w in pd.unique(wcol):
    idx = np.where(wcol == w)[0]; v = tvt[idx]; n = len(v); wl = min(SG_W, n)
    if wl % 2 == 0: wl -= 1
    if wl >= SG_P + 2: out[idx] = savgol_filter(v, wl, SG_P)
pred_df = pd.DataFrame({"id": test_df["id"].values, "tvt": out})
print("predicted rows:", len(pred_df))

# [graft] per-id 予測を辞書へ確保(下流の sub_1 / fleongg 出力で使用)
_V28_PRED = dict(zip(pred_df["id"].astype(str), pred_df["tvt"].astype(float)))
print("[graft] _V28_PRED:", len(_V28_PRED), "rows")

In [ ]:
# ===== import =====
import os, glob, json, warnings, time
import numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.signal import savgol_filter
from numba import njit
warnings.filterwarnings("ignore"); np.seterr(all="ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32
print("device:", DEVICE, "| torch:", torch.__version__)

In [ ]:
# ===== データパス自動判定 =====
CANDS = [
    "/kaggle/input/rogii-wellbore-geology-prediction",
    "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
    "C:/Users/kosaka256/Documents/rogii_claude/rogii-wellbore-geology-prediction",
]
COMP = next((p for p in CANDS if os.path.exists(os.path.join(p, "sample_submission.csv"))), None)
if COMP is None:
    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    COMP = os.path.dirname(hits[0]) if hits else CANDS[-1]
COMP = Path(COMP); TEST = COMP / "test"
print("COMP:", COMP, "| test exists:", TEST.exists())

In [ ]:
# ===== 設定(tune_smoother_r0_v25.py と揃える) =====
SEED        = 0
N_SEED      = 128          # r0_seed32 regime
SMOOTH_LAG  = 192         # 先読みステップ数 L(smoother_r0.json の best。実測 lag192で train -0.229)
SMOOTH_BLEND = 1.0        # 最終=(1-b)*forward + b*smoothed。OOFは ~0.94 を示唆
CHUNK       = 80          # GPUバッチ坑井数。バッファ B*L*N(=CHUNK*N_SEED*L*N)。OOMなら下げる
CLAMP_FT    = None        # smoothedがforwardから±これ超で外れた行はforwardに戻す。None=無効

# --- カテゴリ別 beam + hold(tuned-heuristic-only-cat6.ipynb と同一値) ---
USE_BEAM = True
USE_HOLD = True
N_EVAL_THRESHOLD  = 4840.0
Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)
BEAM_W = {0: 0.0, 1: 0.0, 2: 0.2, 3: 0.0, 4: 0.05, 5: 0.2}
HOLD_W = {0: 0.2, 1: 0.15, 2: 0.15, 3: 0.15, 4: 0.05, 5: 0.05}
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1), (10, 14.0, 90.0, 5),
    (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3), (15, 25.0, 180.0, 2), (20, 30.0, 200.0, 2),
    (15, 10.0, 80.0, 4), (25, 6.0, 50.0, 3), (10, 40.0, 300.0, 1), (12, 18.0, 120.0, 5),
    (30, 8.0, 70.0, 2), (10, 50.0, 400.0, 0),
]

APPLY_PP    = True        # tauランプ+sg。単体を見たいなら False
PP_PARAMS   = {"alpha": 1.0, "tau": 85}
SG_W, SG_P  = 17, 3

# 固定 PF = r0_seed32 (use_phys=false)
R0 = {
    "n_particles": 400, "likelihood_scale": 10.536319667317654,
    "init_pos_std": 0.9331541486020234, "init_rate_std": 0.00047753793163756304,
    "mom": 0.9997911117425017, "vn": 0.0006297672017716754, "pn": 0.005014009140727263,
    "rp": 0.7627309890952053, "rr": 0.00045663430474523125, "resamp": 0.032049849143995945,
    "gr_sig_min": 11.865793233968713, "gr_sig_max": 44.98927964612316, "gr_sig_def": 119.25397464636981,
    "gr_sig_mult": 3.058318309943017, "tvt_clip_margin": 150.56111180234504, "rate_clip": 2.0,
    "gr_power": 1.7826896851395453, "hgr_smooth_r": 0, "tw_gr_smooth_r": 0,
    "jump_prob": 0.002, "jump_std": 5.0, "phys_sig": 0.013666550145707291, "use_phys": False,
}
print(f"PF=r0_seed32 n_seed={N_SEED} smooth_lag={SMOOTH_LAG} blend={SMOOTH_BLEND} chunk={CHUNK} "
      f"clamp={CLAMP_FT} beam={USE_BEAM} hold={USE_HOLD} pp={APPLY_PP}")
SPLIT = "test"; N_WELLS = None   # [graft] test 固定

In [ ]:
# ===== エンジン(tune_smoother_r0_v25.py と同一・インライン) =====
def _smooth(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt, tw_gr, step=0.2):
    a, b = float(tw_tvt.min()), float(tw_tvt.max()); g = np.arange(a, b+step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(a), float(step)

def well_code(c):
    n_bin = int(c["n_eval"] > N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(Z_SPAN_THRESHOLDS, c["z_span"], side="right"))
    return n_bin + 2*z_bin

def build_test(wid):
    hw = pd.read_csv(TEST / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(TEST / f"{wid}__typewell.csv").sort_values("TVT")
    if "TVT_input" not in hw.columns or "TVT" not in tw.columns:
        return None
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0 or len(kn) < 1:
        return None
    tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    if len(tw_tvt) < 3:
        return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    gr_filled = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    ev_pos = hw.index.get_indexer(ev.index); kpos = hw.index.get_indexer(kn.index)
    ls = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail["TVT_input"].values); dz = np.diff(tail["Z"].values)
    dm = np.diff(tail["MD"].values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.
    z_ev = ev["Z"].to_numpy(float)
    n_eval = int(len(z_ev)); z_span = float(np.nanmax(z_ev)-np.nanmin(z_ev)) if len(z_ev) else 0.0
    ev_ids = [f"{wid}_{i}" for i in ev.index]
    return dict(md=ev["MD"].to_numpy(float), z=z_ev, gr_filled=gr_filled, ev_pos=ev_pos, kpos=kpos,
                kn_tvtin=kn["TVT_input"].to_numpy(float), tw_tvt=tw_tvt, tw_gr=tw_gr, tw_fb=tw_fb,
                ls=ls, ir=ir, ev_ids=ev_ids, n_eval=n_eval, z_span=z_span,
                last_known_md=float(kn["MD"].iloc[-1]), last_known_tvt=float(kn["TVT_input"].iloc[-1]))

def make_inputs(c, P):
    tw_gr_pf = _smooth(c["tw_gr"], c["tw_fb"], P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(c["tw_tvt"], tw_gr_pf)
    gr_sm = _smooth(c["gr_filled"], c["tw_fb"], P["hgr_smooth_r"])
    if len(c["kpos"]) < 20:
        gs = P["gr_sig_def"]
    else:
        resid = gr_sm[c["kpos"]] - np.interp(c["kn_tvtin"], c["tw_tvt"], tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
    gs = float(np.clip(gs*P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"]+1e-6)))
    return dict(md=c["md"], z=c["z"], gr=gr_sm[c["ev_pos"]].astype(np.float64),
                gg=gg, gmin=gmin, gst=gst, gs=gs, ls=c["ls"], ir=c["ir"], N=int(P["n_particles"]))


def _pf_smoother_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, P, N, device, gen, dtype=DTYPE):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    L = int(P.get("smooth_lag", 32)); L = max(0, min(L, T))
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS*rn(); rate = ir[:, None] + IR*rn()
    w = torch.full((B, N), 1.0/N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype)-1)*step + CLIP
    glast = torch.gather(grid, 1, (glen-1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    use_sm = L > 0
    if use_sm:
        buf = torch.full((B, L, N), float("nan"), device=device, dtype=dtype); ptr = 0
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur-pm).clamp_min(1.0)
        rate_n = ALPHA*rate + RN*rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n*dm[:, None] + PN*rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD*rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n-zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt+zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos-zi; ii = (v-vmin[:, None])/step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen-1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen-2).clamp_min(0)[:, None])
        t = ii-i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c+1)
        eg = gA*(1-t)+gB*t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None]-eg)/gs[:, None]).abs(); dp = d**GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5*dp), zero).clamp_min(1e-300)
        avg = (w*lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate-dipp)/PHYS_SIG; lkp = torch.exp(-0.5*dphys*dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w*lk*lkp, w*lkp)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new/ws.clamp_min(1e-300), torch.full_like(w, 1.0/N))
        ne = (w*w).sum(1); need = act & ((1.0/ne) < (RESAMP*N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype)*(1.0/N)
            u = u0 + arN[None, :]/N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N-1)
            pos_rs = torch.gather(pos, 1, idx) + RP*rn(); rate_rs = torch.gather(rate, 1, idx) + RR*rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0/N), w)
            if use_sm:
                buf_g = torch.gather(buf, 2, idx[:, None, :].expand(B, L, N))
                buf = torch.where(need[:, None, None], buf_g, buf)
        pts_f[:, i] = (w*(pos-zi)).sum(1)
        if use_sm:
            if i >= L:
                old = buf[:, ptr, :]; pts_s[:, i-L] = (w*old).sum(1) - z[:, i-L]
            buf[:, ptr, :] = pos; ptr = (ptr+1) % L
        pm = torch.where(act, cur, pm)
    if use_sm:
        for j in range(max(0, T-L), T):
            pts_s[:, j] = (w*buf[:, j % L, :]).sum(1) - z[:, j]
    else:
        pts_s = pts_f
    return pts_f, pts_s, log_lik


def _pad(inps, device, dtype=DTYPE):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn-1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G-1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir))

def run_smoother_ens(inps, P, device, seed, n_seeds):
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); pad = _pad(inps, device)
    rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=device); gen.manual_seed(seed)
    pf, ps, ll = _pf_smoother_core(
        rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
        pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
        pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
        P, N, device, gen)
    pf = pf.view(W, S, -1).double().cpu().numpy(); ps = ps.view(W, S, -1).double().cpu().numpy()
    ll = ll.view(W, S).cpu().numpy()
    out_f = []; out_s = []
    for w in range(W):
        T = len(inps[w]["md"]); llw = ll[w]
        if np.isfinite(llw).any():
            mx = np.nanmax(llw[np.isfinite(llw)])
            lk = np.where(np.isfinite(llw), llw-mx, -np.inf)
            ww = np.exp(lk / max(P["likelihood_scale"], 1e-6)); s = ww.sum()
            ww = ww/s if s > 1e-300 else np.full(S, 1.0/S)
        else:
            ww = np.full(S, 1.0/S)
        out_f.append((ww[:, None]*pf[w, :, :T]).sum(0))
        out_s.append((ww[:, None]*ps[w, :, :T]).sum(0))
    if device == "cuda": torch.cuda.synchronize()
    return out_f, out_s


# ===== beam search(7.191移植, numba) =====
@njit(cache=False)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS*6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv - tw_gr[ni])**2/es + mc*(d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i+1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]; cC[i], cC[mi] = cC[mi], cC[i]; cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n-1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

def _nn(arr, val): return int(np.abs(arr - val).argmin())

def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt)
    sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    path = _beam_jit(sgr, tw_gr.astype(np.float64), si, int(bs), float(mc), float(es))
    return tw_tvt[path].astype(np.float32)

def run_beam(c):
    tw_tvt = c["tw_tvt"].astype(np.float64)
    fb = float(np.nanmean(c["tw_gr"])) if np.isfinite(np.nanmean(c["tw_gr"])) else 0.0
    tw_gr = np.where(np.isfinite(c["tw_gr"]), c["tw_gr"], fb).astype(np.float64)
    hgr = c["gr_filled"][c["ev_pos"]]; last_tvt = c["last_known_tvt"]
    res = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for (bs, mc, es, r) in BEAM_CONFIGS]
    return np.stack(res, 0).mean(0).astype(np.float64)


def post_process(pred, c):
    pred = np.asarray(pred, dtype=np.float64)
    if not APPLY_PP:
        return pred
    md_since = np.maximum(c["md"] - c["last_known_md"], 0.0)
    d = (pred - c["last_known_tvt"]) * (1.0 - np.exp(-md_since / float(PP_PARAMS["tau"]))) * float(PP_PARAMS.get("alpha", 1.0))
    out = c["last_known_tvt"] + d
    n = len(out); wl = min(SG_W, n)
    if wl % 2 == 0: wl -= 1
    return savgol_filter(out, wl, SG_P) if wl >= SG_P + 2 else out

In [ ]:
# ===== XY近傍(test←train) + 疑似typewell(mixed grid) + 距離連動α =====
# 議論した v27 XY近傍版を提出に注入。PFパラメータ(R0/N_SEED/SMOOTH_LAG…)は上のまま。
K_MAX      = 5          # 近傍の最大本数
MAX_DIST   = 1500.0     # これより遠いtrain坑井は参照にしない(α cutoff と整合。far参照ノイズ除去)
MIX_W      = 1.0        # 疑似typewell混合比(covered帯; 1.0=実HWテンプレで完全置換)
NEED_REFS  = 2          # 近傍がこの本数未満なら混ぜない(=現行tw)。2本ないと安定しない実測
SPLIT      = "test"     # "test"=提出 / "train"=train坑井でOOF検証(末尾の採点セルで真値TVTと比較)
N_WELLS    = None       # 検証を軽くする時に坑井数制限(例50)。None=全部
TRAIN = COMP / "train"
TEST  = COMP / SPLIT    # build_test/test_neighbors はこの TEST を読む(train検証時はtrainを読む)

# α(nn_dist): <=250→0.7 / 500→0.6 / 1000→0.45 / 1500→0.3 / >1500→0(混ぜない=先端)
def alpha_dist(d):
    if d <= 250: return 0.7
    if d > 1500: return 0.0
    return float(np.interp(d, [250, 500, 1000, 1500], [0.7, 0.6, 0.45, 0.3]))

# train坑井の XY重心
_train_wids = sorted(os.path.basename(p).split("__")[0]
                     for p in glob.glob(str(TRAIN / "*__horizontal_well.csv")))
_tw_cent = {}
for _w in _train_wids:
    _h = pd.read_csv(TRAIN / f"{_w}__horizontal_well.csv", usecols=["X", "Y"])
    _tw_cent[_w] = (float(_h["X"].mean()), float(_h["Y"].mean()))
_tw_xy = np.array([_tw_cent[w] for w in _train_wids])
print("train坑井(参照プール):", len(_train_wids))

# 参照train坑井の (GR, TVT) キャッシュ
_REFCACHE = {}
def ref_gr_tvt(wid):
    if wid not in _REFCACHE:
        _h = pd.read_csv(TRAIN / f"{wid}__horizontal_well.csv", usecols=["GR", "TVT"])
        _REFCACHE[wid] = (_h["GR"].to_numpy(float), _h["TVT"].to_numpy(float))
    return _REFCACHE[wid]

# test坑井の XY近傍(同名train=可視testのサブセットは除外=リーク防止)
def test_neighbors(test_wid):
    _h = pd.read_csv(TEST / f"{test_wid}__horizontal_well.csv", usecols=["X", "Y"])
    cx, cy = float(_h["X"].mean()), float(_h["Y"].mean())
    d = np.hypot(_tw_xy[:, 0] - cx, _tw_xy[:, 1] - cy)
    out = []
    for j in np.argsort(d):
        w = _train_wids[j]
        if w == test_wid: continue
        if d[j] > MAX_DIST: break
        out.append((w, float(d[j])))
        if len(out) >= K_MAX: break
    return out

# 疑似typewell(参照HW GRのTVTビン median)を typewell grid に混ぜた make_inputs
def make_inputs_mixed(c, refs, P, w_hw=MIX_W):
    base = make_inputs(c, P)
    gmin = base["gmin"]; step = base["gst"]; G = len(base["gg"])
    rt = np.concatenate([r[1] for r in refs])
    rg = np.concatenate([_smooth(r[0], float(np.nanmean(r[0])), P["hgr_smooth_r"]).astype(float) for r in refs])
    m = np.isfinite(rt) & np.isfinite(rg); rt, rg = rt[m], rg[m]
    idx = np.round((rt - gmin) / step).astype(int); ok = (idx >= 0) & (idx < G); idx, rgv = idx[ok], rg[ok]
    ghw = np.full(G, np.nan)
    med = pd.Series(rgv).groupby(idx).median(); ghw[med.index.to_numpy()] = med.to_numpy()  # median pool
    have = np.isfinite(ghw)
    gg = base["gg"].copy(); gg[have] = w_hw * ghw[have] + (1 - w_hw) * gg[have]
    base = dict(base); base["gg"] = gg
    return base
print(f"XY近傍注入 ready | K_MAX={K_MAX} MAX_DIST={MAX_DIST:.0f} MIX_W={MIX_W} NEED_REFS={NEED_REFS} α=距離連動")

In [ ]:
# ===== test 推論: tw-PF / mix-PF 2パス → 距離連動αで位置ブレンド → beam+hold+pp(最後に1回) =====
wids = sorted(set(Path(p).stem.replace("__horizontal_well", "")
                  for p in glob.glob(str(TEST / "*__horizontal_well.csv"))))
print("test wells:", len(wids))
CACHE = {}
for wid in wids:
    try:
        c = build_test(wid)
    except Exception:
        c = None
    if c is not None: CACHE[wid] = c
usable = [w for w in wids if w in CACHE]
if N_WELLS: usable = usable[:N_WELLS]
print("usable wells:", len(usable), "| SPLIT:", SPLIT)

# 近傍とα(test坑井ごと)
NB = {w: test_neighbors(w) for w in usable}
def alpha_of(w):
    nb = NB[w]
    return alpha_dist(nb[0][1]) if len(nb) >= NEED_REFS else 0.0
mix_wids = [w for w in usable if alpha_of(w) > 0]
print(f"混合対象(α>0, 近傍>={NEED_REFS}本 & nn<=1500m): {len(mix_wids)} / {len(usable)} 坑井")

P = {**R0, "smooth_lag": SMOOTH_LAG}
def _blend_fs(fwd, sm):
    if CLAMP_FT is not None:
        bad = np.abs(sm - fwd) > CLAMP_FT
        if bad.any(): sm = np.where(bad, fwd, sm)
    return (1.0 - SMOOTH_BLEND) * fwd + SMOOTH_BLEND * sm

# --- パス1: tw-PF(全坑井, typewell grid) ---
OUT_TW = {}; t0 = time.time()
for c0 in range(0, len(usable), CHUNK):
    sub = usable[c0:c0+CHUNK]
    out_f, out_s = run_smoother_ens([make_inputs(CACHE[w], P) for w in sub], P, DEVICE, SEED, N_SEED)
    for w, fwd, sm in zip(sub, out_f, out_s): OUT_TW[w] = _blend_fs(fwd, sm)
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(f"  tw {min(c0+CHUNK,len(usable))}/{len(usable)} ({time.time()-t0:.0f}s)")

# --- パス2: mix-PF(混合対象のみ, mixed grid=疑似typewell) ---
OUT_MIX = {}
for c0 in range(0, len(mix_wids), CHUNK):
    sub = mix_wids[c0:c0+CHUNK]
    inps = [make_inputs_mixed(CACHE[w], [ref_gr_tvt(r) for r, _ in NB[w]], P) for w in sub]
    out_f, out_s = run_smoother_ens(inps, P, DEVICE, SEED, N_SEED)
    for w, fwd, sm in zip(sub, out_f, out_s): OUT_MIX[w] = _blend_fs(fwd, sm)
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(f"  mix {min(c0+CHUNK,len(mix_wids))}/{len(mix_wids)} ({time.time()-t0:.0f}s)")

# --- 距離連動α で位置ブレンド → beam+hold+pp(最後に1回) ---
rows = []
for w in usable:
    c = CACHE[w]; a = alpha_of(w)
    pos = OUT_TW[w] if a == 0 else (1.0 - a) * OUT_TW[w] + a * OUT_MIX[w]
    code = well_code(c)
    bw = BEAM_W.get(code, 0.0) if USE_BEAM else 0.0
    hwt = HOLD_W.get(code, 0.0) if USE_HOLD else 0.0
    pred = np.asarray(pos, dtype=np.float64).copy()
    if bw > 0:
        beam = run_beam(c); m = min(len(pred), len(beam)); pred[:m] = (1.0 - bw) * pred[:m] + bw * beam[:m]
    if hwt > 0:
        pred = (1.0 - hwt) * pred + hwt * c["last_known_tvt"]
    pred = post_process(pred, c)
    ids = c["ev_ids"]; n = min(len(ids), len(pred))
    rows.extend((ids[i], float(pred[i])) for i in range(n))
pred_df = pd.DataFrame(rows, columns=["id", "tvt"])
print("predicted rows:", len(pred_df), f"({time.time()-t0:.0f}s)")

# [graft] per-id 予測を辞書へ確保(下流の sub_2 で使用)
_V27_PRED = dict(zip(pred_df["id"].astype(str), pred_df["tvt"].astype(float)))
print("[graft] _V27_PRED:", len(_V27_PRED), "rows")

In [ ]:
import os
os.environ["ROGII_GOLD_PROFILE"] = "conservative"

In [ ]:
import sys, os, glob, subprocess
# koolbox setup: wheel install or sys.path fallback
kb_dir = '/kaggle/input/koolbox-offline'
if not os.path.isdir(kb_dir):
    # alt path under datasets/
    cand = glob.glob('/kaggle/input/**/koolbox*', recursive=True)
    print('koolbox candidates:', cand[:5])
    if cand: kb_dir = cand[0]
print('using koolbox dir:', kb_dir)
if os.path.isdir(kb_dir):
    print('listing:', os.listdir(kb_dir)[:20])
    whls = glob.glob(f'{kb_dir}/**/*.whl', recursive=True)
    if whls:
        for w in whls:
            print('install', w)
            subprocess.run(['pip', 'install', '--no-deps', w], check=False)
    else:
        sys.path.insert(0, kb_dir)
        # also try subdirs
        for sub in os.listdir(kb_dir):
            sub_path = os.path.join(kb_dir, sub)
            if os.path.isdir(sub_path):
                sys.path.insert(0, sub_path)
import koolbox
print('koolbox OK:', koolbox.__file__)

In [ ]:
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
from koolbox import Trainer
from pathlib import Path
from numba import njit
import matplotlib.pyplot as plt
import multiprocessing
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
import joblib
import time
import glob
import os

warnings.filterwarnings("ignore")

In [ ]:
class CFG:
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    artifacts_path = Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts")
    
    seed = 42
    n_splits = 5
    cv = GroupKFold(n_splits=n_splits)
    
    metric = root_mean_squared_error

In [ ]:
SELECTOR_N_EVAL_THRESHOLD = 4840.0
SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)

SELECTOR_BIN_VARIANTS = {
    0: 'pf_scale_5_hold_0.2',
    1: 'pf_scale_3_hold_0.15',
    2: 'pf_scale_12_beam_0.2_hold_0.15',
    3: 'pf_scale_5_hold_0.15',
    4: 'pf_scale_5_beam_0.05_hold_0.05',
    5: 'pf_scale_12_beam_0.2_hold_0.05',
}

SELECTOR_GLOBAL_VARIANT = 'pf_scale_8_hold_0.2'
SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)

FORMATION_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2),
    (10,  8.0,  64.0, 2),
    ( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5),
    (20,  4.0,  36.0, 3),
    (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
    (20, 30.0, 200.0, 2),
    (15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3),
    (10, 40.0, 300.0, 1),
    (12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2),
    (10, 50.0, 400.0, 0),
]


def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    tw_g = tw_tr.dropna(subset=['Geology'])
    ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    if np.isnan(ref_tvt):
        ref_col = tw_g['Geology'].iloc[0]
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
    return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset


def load_well(wid, split='train'):
    base = CFG.dataset_path / split
    hw = pd.read_csv(base / f'{wid}__horizontal_well.csv')
    tw = pd.read_csv(base / f'{wid}__typewell.csv')
    return hw, tw


def run_particle_filter(hw, tw, n_particles=500, seed=42):
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z   = float(last['Z'])
    last_MD  = float(last['MD'])

    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N   = n_particles
    rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + 4.5 * rng.standard_normal(N)  # sp45 patch (sel15 vb best)
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N

    MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5

    md_v = ev['MD'].values.astype(float)
    z_v  = ev['Z'].values.astype(float)
    # Interpolate GR gaps before tracking
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v = gr_interp.values.astype(float)[ev.index]

    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev))
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos   = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        n_eff = 1.0 / (w**2).sum()
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0  = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos  = pos[idx]  + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w    = np.ones(N) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    preds = []
    liks  = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)

    liks   = np.array(liks)
    liks_n = liks - liks.max()
    weights = np.exp(liks_n / scale)
    weights /= weights.sum()

    return (weights[:, None] * np.stack(preds, 0)).sum(0)


def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
    preds = []
    liks = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)
    pred_arr = np.stack(preds, 0)
    liks = np.array(liks)
    liks_n = liks - liks.max()
    out = {}
    for scale in scales:
        weights = np.exp(liks_n / float(scale))
        weights /= weights.sum()
        out[f'pf_scale_{scale:g}'] = (weights[:, None] * pred_arr).sum(0)
    out['pf_mean'] = pred_arr.mean(0)
    return out


def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    n  = len(hgr)
    nt = len(tw_tvt)
    if n == 0:
        return np.array([last_tvt])

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC    = mc * np.array([2., 1., 0., 1., 2.])

    bidx  = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf)
    bcost[0] = 0.
    bn = 1

    result = np.zeros(n)

    for step in range(n):
        gv = sgr[step]
        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)

        gr_e = (gv - tw_gr[ci])**2 / es
        tot  = bcost[:bn, None] + gr_e + MC[None, :]
        tot  = np.where(valid, tot, np.inf)

        ni_f  = ni.flatten()
        tot_f = tot.flatten()
        vf    = valid.flatten()
        ni_f  = ni_f[vf]
        tot_f = tot_f[vf]

        order = np.argsort(tot_f)
        ni_s  = ni_f[order]
        tot_s = tot_f[order]

        _, first = np.unique(ni_s, return_index=True)
        ni_u  = ni_s[first]
        tot_u = tot_s[first]

        kept = min(bs, len(ni_u))
        top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        top  = top[np.argsort(tot_u[top])]

        bidx[:kept]  = ni_u[top]
        bcost[:kept] = tot_u[top]
        if kept < bs:
            bidx[kept:]  = bidx[kept - 1]
            bcost[kept:] = np.inf
        bn = kept

        result[step] = tw_tvt[bidx[0]]

    return result


def run_beam_ensemble(hw, tw):
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    last_tvt = float(kn.iloc[-1]['TVT_input'])
    tw_s  = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr    = gr_all[ev.index]

    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                    for (bs, mc, es, r) in BEAM_CONFIGS]

    beam_mean = np.stack(beam_results, 0).mean(0)

    out = hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)] = beam_mean
    return out


def selector_well_code(hw):
    eval_mask = hw['TVT_input'].isna().to_numpy()
    n_eval = float(eval_mask.sum())
    z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
    z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
    n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
    code = n_bin + 2 * z_bin
    variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
    return code, variant, n_eval, z_span


def parse_selector_variant(name):
    parts = name.split('_')
    scale = float(parts[2])
    beam_weight = 0.0
    hold_weight = 0.0
    if 'beam' in parts:
        beam_weight = float(parts[parts.index('beam') + 1])
    if 'hold' in parts:
        hold_weight = float(parts[parts.index('hold') + 1])
    return scale, beam_weight, hold_weight


def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
    scale, beam_weight, hold_weight = parse_selector_variant(name)
    base = pf_by_scale.get(f'pf_scale_{scale:g}')
    if base is None:
        base = pf_by_scale[SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
    pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
    pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
    return pred

In [ ]:
# ===== セル1: v20 GPU PF エンジン(自己完結・7.191 のどのヘルパにも依存しない)=====
import torch, numpy as _np, pandas as _pd, glob as _glob, os as _os, time as _time
_GDEV = "cuda" if torch.cuda.is_available() else "cpu"; _GDT = torch.float32
print("v20 GPU PF inject | device:", _GDEV)

# ---- r0(v20 強PF)raw パラメータ(pf_bank round0)----
_R0_RAW = dict(n_particles=500, likelihood_scale=6.265873932344672, init_pos_std=1.0481897997156377,
    init_rate_std=0.004385968881669577, mom=0.9995659517817024, vn=0.0009771590366402883,
    pn=0.02193433099852571, rp=1.266960649917433, rr=0.0003377853906820054, resamp=0.05083194815648496,
    gr_sig_min=10.293696795145928, gr_sig_max=49.46806691781805, gr_sig_def=129.97956203556075,
    gr_sig_mult=3.412937310824599, tvt_clip_margin=159.57810929194702, rate_clip=0.0,
    gr_power=1.5371545258734838, hgr_smooth_r=1, tw_gr_smooth_r=0, jump_prob=0.002, jump_std=5.0,
    phys_sig=4.0313262188520795, use_phys=True)
_GN_SEEDS = 128          # ★あなたのPFの調整regime(submit_v20=32)。128(=7.191自身のPF用)は流用しない
_GCHUNK   = 100           # ★GPUバッチの坑井数。実バッチ=_GCHUNK×_GN_SEEDS。OOMなら 8→4→2 と下げる

def _mk_p(r, n_seeds):
    p = {}
    p["ANCC_N"]=int(r["n_particles"]); p["PF_N_SEEDS"]=int(n_seeds)
    p["PF_LIKELIHOOD_SCALE"]=float(r["likelihood_scale"]); p["PF_INIT_SPR"]=float(r["init_pos_std"])
    p["PF_INIT_V_STD"]=float(r["init_rate_std"]); p["PF_MOM"]=float(r["mom"]); p["PF_VN"]=float(r["vn"])
    p["PF_PN"]=float(r["pn"]); p["PF_ROUGH_P"]=float(r["rp"]); p["PF_ROUGH_V"]=float(r["rr"])
    p["PF_RESAMP"]=float(r["resamp"]); p["PF_GR_SIG_MIN"]=float(r["gr_sig_min"]); p["PF_GR_SIG_MAX"]=float(r["gr_sig_max"])
    p["PF_GR_SIG_DEF"]=float(r["gr_sig_def"]); p["PF_GR_SIG_MULT"]=float(r["gr_sig_mult"])
    p["PF_TVT_CLIP_MARGIN"]=float(r["tvt_clip_margin"]); p["PF_RATE_CLIP"]=float(r["rate_clip"])
    p["PF_GR_POWER"]=float(r["gr_power"]); p["PF_HGR_SMOOTH_R"]=int(r["hgr_smooth_r"]); p["PF_TW_GR_SMOOTH_R"]=int(r["tw_gr_smooth_r"])
    p["PF_JUMP_PROB"]=float(r["jump_prob"]); p["PF_JUMP_STD"]=float(r["jump_std"])
    p["ANCC_ALPHA"]=p["PF_MOM"]; p["ANCC_RN"]=p["PF_VN"]; p["ANCC_PN"]=p["PF_PN"]; p["ANCC_IR"]=p["PF_INIT_V_STD"]
    p["ANCC_IS"]=p["PF_INIT_SPR"]; p["ANCC_RP"]=p["PF_ROUGH_P"]; p["ANCC_RR"]=p["PF_ROUGH_V"]
    p["PF_USE_PHYS"]=bool(r["use_phys"]); p["PF_PHYS_SIG"]=float(r["phys_sig"])
    return p
_GP = _mk_p(_R0_RAW, _GN_SEEDS)

# ---- 自前ヘルパ(7.191 の _smooth/_grid/_gr_sig とシグネチャ衝突を避け独自定義)----
def _gsm(vals, fb, r):
    s = _pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    if int(r) <= 0: return s.to_numpy(_np.float32)
    return s.rolling(2*int(r)+1, center=True, min_periods=1).mean().to_numpy(_np.float32)
def _ggrid(tw_tvt, tw_gr, step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max()); g=_np.arange(tmin,tmax+step,step)
    return _np.interp(g,tw_tvt,tw_gr).astype(_np.float64), float(tmin), float(step)
def _ggrsig(gr_proc, kn_pos, ktvt, tw_tvt, tw_gr_pf, p):
    if len(kn_pos) < 20: return float(p["PF_GR_SIG_DEF"])
    resid = gr_proc[kn_pos] - _np.interp(ktvt, tw_tvt, tw_gr_pf)
    g = float(_np.nanstd(resid)); g = g if (_np.isfinite(g) and g>0) else float(p["PF_GR_SIG_DEF"])
    return float(_np.clip(g*p["PF_GR_SIG_MULT"], p["PF_GR_SIG_MIN"], p["PF_GR_SIG_MAX"]))

def _gbuild(hw, tw, p):
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(kn) < 5 or len(ev) < 1: return None
    tw = tw.sort_values("TVT"); tw_tvt = tw["TVT"].to_numpy(_np.float64); tw_gr = tw["GR"].to_numpy(_np.float64)
    if len(tw_tvt) < 3: return None
    fb = float(_np.nanmean(tw_gr)) if _np.isfinite(_np.nanmean(tw_gr)) else 0.
    tw_gr_pf = _gsm(tw_gr, fb, p["PF_TW_GR_SMOOTH_R"]).astype(_np.float64)
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(fb).to_numpy(float)
    gr_proc = _gsm(gr_full, fb, p["PF_HGR_SMOOTH_R"]).astype(_np.float64)
    ev_pos = hw.index.get_indexer(ev.index); kn_pos = hw.index.get_indexer(kn.index)
    ls = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
    tail = kn.tail(30); dt=_np.diff(tail["TVT_input"].values); dz=_np.diff(tail["Z"].values); dm=_np.diff(tail["MD"].values); m=dm>0
    ir = float(_np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst = _ggrid(tw_tvt, tw_gr_pf)
    gs = _ggrsig(gr_proc, kn_pos, kn["TVT_input"].to_numpy(_np.float64), tw_tvt, tw_gr_pf, p)
    return dict(md=ev["MD"].to_numpy(_np.float64), z=ev["Z"].to_numpy(_np.float64),
                gr=gr_proc[ev_pos].astype(_np.float64), gg=gg, gmin=gmin, gst=gst, gs=gs, ls=ls, ir=ir, N=int(p["ANCC_N"]))

def _gpf_core(md,z,gr,valid,grid,glen,vmin,step,gs,ls,ir,p,N,device,gen,dtype=_GDT):
    B,T = md.shape
    AL=p["ANCC_ALPHA"];RN=p["ANCC_RN"];PN=p["ANCC_PN"];IR=p["ANCC_IR"];IS=p["ANCC_IS"];RP=p["ANCC_RP"];RR=p["ANCC_RR"]
    RES=p["PF_RESAMP"];RC=p["PF_RATE_CLIP"];GP=p["PF_GR_POWER"];JP=p["PF_JUMP_PROB"];JS=p["PF_JUMP_STD"];CL=p["PF_TVT_CLIP_MARGIN"]
    UP=bool(p.get("PF_USE_PHYS",False));PS=float(p.get("PF_PHYS_SIG",1.0))
    def rn(): return torch.randn((B,N),generator=gen,device=device,dtype=dtype)
    pos=ls[:,None]+IS*rn(); rate=ir[:,None]+IR*rn(); w=torch.full((B,N),1.0/N,device=device,dtype=dtype)
    log_lik=torch.zeros(B,device=device,dtype=torch.float64); pts=torch.zeros((B,T),device=device,dtype=dtype)
    tvt_lo=vmin-CL; tvt_hi=vmin+(glen.to(dtype)-1)*step+CL
    glast=torch.gather(grid,1,(glen-1).clamp_min(0).unsqueeze(1)); g0=grid[:,0:1]
    pm=md[:,0]-1.0; arN=torch.arange(N,device=device,dtype=dtype); zero=torch.zeros((),device=device,dtype=dtype); dipp=ir[:,None]
    for i in range(T):
        act=valid[:,i]; cur=md[:,i]; dm=(cur-pm).clamp_min(1.0)
        rate_n=AL*rate+RN*rn()
        if RC>0: rate_n=rate_n.clamp(-RC,RC)
        pos_n=pos+rate_n*dm[:,None]+PN*rn()
        if JP>0 and JS>0:
            jm=torch.rand((B,N),generator=gen,device=device,dtype=dtype)<JP; pos_n=pos_n+torch.where(jm,JS*rn(),zero)
        zi=z[:,i][:,None]; tvt=(pos_n-zi).clamp(tvt_lo[:,None],tvt_hi[:,None]); pos_n=tvt+zi; am=act[:,None]
        pos=torch.where(am,pos_n,pos); rate=torch.where(am,rate_n,rate)
        gri=gr[:,i]; obs=act & ~torch.isnan(gri); v=pos-zi
        ii=(v-vmin[:,None])/step[:,None]; i0=torch.floor(ii).long()
        below=i0<0; above=i0>=(glen-1)[:,None]; i0c=i0.clamp_min(0); i0c=torch.minimum(i0c,(glen-2).clamp_min(0)[:,None])
        t=ii-i0c.to(dtype); gA=torch.gather(grid,1,i0c); gB=torch.gather(grid,1,i0c+1); eg=gA*(1-t)+gB*t
        eg=torch.where(below,g0,eg); eg=torch.where(above,glast,eg)
        d=((gri[:,None]-eg)/gs[:,None]).abs(); dp=d**GP
        lk=torch.where(dp<600.0,torch.exp(-0.5*dp),zero).clamp_min(1e-300)
        avg=(w*lk).sum(1)
        log_lik=log_lik+torch.where(obs,torch.log(avg.clamp_min(1e-300).double()),torch.zeros((),device=device,dtype=torch.float64))
        if UP:
            dph=(rate-dipp)/PS; lkp=torch.exp(-0.5*dph*dph).clamp_min(1e-300)
        else: lkp=torch.ones_like(w)
        wn=torch.where(obs[:,None],w*lk*lkp,w*lkp); ws=wn.sum(1,keepdim=True)
        w=torch.where(ws>0,wn/ws.clamp_min(1e-300),torch.full_like(w,1.0/N))
        ne=(w*w).sum(1); need=act & ((1.0/ne)<(RES*N))
        if bool(need.any()):
            cumw=torch.cumsum(w,1); u0=torch.rand((B,1),generator=gen,device=device,dtype=dtype)*(1.0/N); u=u0+arN[None,:]/N
            idx=torch.searchsorted(cumw,u,right=False).clamp(0,N-1)
            pos_rs=torch.gather(pos,1,idx)+RP*rn(); rate_rs=torch.gather(rate,1,idx)+RR*rn()
            if RC>0: rate_rs=rate_rs.clamp(-RC,RC)
            nm=need[:,None]; pos=torch.where(nm,pos_rs,pos); rate=torch.where(nm,rate_rs,rate); w=torch.where(nm,torch.full_like(w,1.0/N),w)
        pts[:,i]=(w*(pos-zi)).sum(1); pm=torch.where(act,cur,pm)
    return pts, log_lik

def _gpad(inps, device, dtype=_GDT):
    W=len(inps); Tmax=max(len(x["md"]) for x in inps); Gmax=max(len(x["gg"]) for x in inps)
    md=torch.zeros((W,Tmax),dtype=dtype); z=torch.zeros((W,Tmax),dtype=dtype)
    gr=torch.full((W,Tmax),float("nan"),dtype=dtype); valid=torch.zeros((W,Tmax),dtype=torch.bool)
    grid=torch.zeros((W,Gmax),dtype=dtype); glen=torch.zeros(W,dtype=torch.long)
    vmin=torch.zeros(W,dtype=dtype); step=torch.zeros(W,dtype=dtype); gs=torch.zeros(W,dtype=dtype)
    ls=torch.zeros(W,dtype=dtype); ir=torch.zeros(W,dtype=dtype)
    for b,x in enumerate(inps):
        T=len(x["md"]); G=len(x["gg"])
        md[b,:T]=torch.from_numpy(x["md"].astype("float32"));
        if T<Tmax: md[b,T:]=md[b,T-1]
        z[b,:T]=torch.from_numpy(x["z"].astype("float32"))
        gr[b,:T]=torch.from_numpy(x["gr"].astype("float32")); valid[b,:T]=True
        grid[b,:G]=torch.from_numpy(x["gg"].astype("float32"))
        if G<Gmax: grid[b,G:]=grid[b,G-1]
        glen[b]=G; vmin[b]=x["gmin"]; step[b]=x["gst"]; gs[b]=x["gs"]; ls[b]=x["ls"]; ir[b]=x["ir"]
    to=lambda t: t.to(device)
    return dict(md=to(md),z=to(z),gr=to(gr),valid=to(valid),grid=to(grid),glen=to(glen),
                vmin=to(vmin),step=to(step),gs=to(gs),ls=to(ls),ir=to(ir))

def _grun_scales(inps,p,S,device,seed,chunk,scales):
    N=int(p["ANCC_N"]); W=len(inps); out=[None]*W
    def _ws(base,pr,sc):
        ww=_np.exp(_np.where(_np.isfinite(base),base,-_np.inf)/max(float(sc),1e-6))
        ww=ww/ww.sum() if ww.sum()>1e-300 else _np.full(len(base),1.0/len(base)); return (ww[:,None]*pr).sum(0).astype(_np.float32)
    for c in range(0,W,chunk):
        sub=inps[c:c+chunk]; w=len(sub); pad=_gpad(sub,device); rep=lambda t: t.repeat_interleave(S,dim=0)
        gen=torch.Generator(device=device); gen.manual_seed(seed)
        pts,ll=_gpf_core(rep(pad["md"]),rep(pad["z"]),rep(pad["gr"]),rep(pad["valid"]),rep(pad["grid"]),
                         pad["glen"].repeat_interleave(S),rep(pad["vmin"]),rep(pad["step"]),
                         rep(pad["gs"]),rep(pad["ls"]),rep(pad["ir"]),p,N,device,gen)
        pts=pts.view(w,S,-1).double().cpu().numpy(); ll=ll.view(w,S).cpu().numpy()
        del pad
        if device == "cuda": torch.cuda.empty_cache()      # チャンク毎に解放(OOM対策)
        for j in range(w):
            T=len(sub[j]["md"]); pr=pts[j,:,:T]
            base=ll[j]-_np.nanmax(ll[j]) if _np.isfinite(ll[j]).any() else _np.zeros_like(ll[j])
            d={f"pf_scale_{sc:g}": _ws(base,pr,sc) for sc in scales}
            d["pf_mean"]=_np.mean([d[f"pf_scale_{sc:g}"] for sc in scales],0).astype(_np.float32)
            out[c+j]=d
    return out
print("v20 GPU PF engine ready | r0 phys=%s seeds=%d" % (_GP["PF_USE_PHYS"], _GN_SEEDS))

In [ ]:
# ===== セル2: test 全坑井を GPU PF で前計算 → run_pf_lik_ensemble_scales を差し替え =====
def _gfp(hw):
    md = hw["MD"].to_numpy(float)
    return (len(hw), round(float(md[0]),1), round(float(md[-1]),1))

# データパス: 7.191 の CFG.dataset_path を優先、無ければ自動探索
try:    _COMP = str(CFG.dataset_path)
except Exception:
    import glob as _g2
    _s = _g2.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    _COMP = _os.path.dirname(_s[0]) if _s else "/kaggle/input/competitions/rogii-wellbore-geology-prediction"

_test_files = sorted(_glob.glob(_os.path.join(_COMP,"test","*__horizontal_well.csv")))
print("test wells:", len(_test_files), "| COMP:", _COMP)
_inps=[]; _fps=[]
for _hp in _test_files:
    _hw=_pd.read_csv(_hp); _tw=_pd.read_csv(_hp.replace("__horizontal_well","__typewell"))
    _x=_gbuild(_hw,_tw,_GP)
    if _x is None: continue
    _inps.append(_x); _fps.append(_gfp(_hw))
_t0=_time.perf_counter()
_GCACHE={}
if _inps:
    _res=_grun_scales(_inps,_GP,_GN_SEEDS,_GDEV,0,_GCHUNK,SELECTOR_SCALES)
    for _fp,_d in zip(_fps,_res): _GCACHE[_fp]=_d
print(f"GPU PF precomputed {len(_GCACHE)} wells in {_time.perf_counter()-_t0:.1f}s ({_GDEV})")

# 差し替え(同名・同戻り値=★坑井全長配列。既知行=TVT_input, 評価行=GPU PF予測)。
# 長さ不一致/未ヒットは安全に原版CPUへフォールバック。
_orig_pf_scales = run_pf_lik_ensemble_scales
def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
    _d = _GCACHE.get(_gfp(hw))
    _ev = hw[hw["TVT_input"].isna()]
    if _d is None or len(_ev) == 0:
        return _orig_pf_scales(hw, tw, scales=scales, n_particles=n_particles, n_seeds=n_seeds)
    _base = hw["TVT_input"].values.astype(float)        # 全長(既知=値, 評価=NaN)
    _idx = list(_ev.index)
    def _full(vec):
        if vec is None or len(vec) != len(_idx): return None
        o = _base.copy(); o[_idx] = vec; return o
    out = {}
    for _k, _v in _d.items():
        if not _k.startswith("pf_scale_"): continue
        _f = _full(_v)
        if _f is None:                                  # 長さズレ→安全にCPU
            return _orig_pf_scales(hw, tw, scales=scales, n_particles=n_particles, n_seeds=n_seeds)
        out[_k] = _f
    _fm = _full(_d["pf_mean"])
    if _fm is None:
        return _orig_pf_scales(hw, tw, scales=scales, n_particles=n_particles, n_seeds=n_seeds)
    out["pf_mean"] = _fm
    return out
print("OK: run_pf_lik_ensemble_scales -> v20 GPU PF(全長配列で返す / selector のみ差し替え / run_pf_ancc は原版)")

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
# [graft] sub_1 = v28 GBM(LightGBM/CatBoost/XGB 9モデル平均 + apply_pp + per-well sg, ridgeなし)
sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
_fb1 = float(np.nanmedian(list(_V28_PRED.values())))
sub_1 = sample_sub[['id']].copy()
sub_1['tvt'] = sub_1['id'].astype(str).map(_V28_PRED).fillna(_fb1)
print('[graft] sub_1(v28GBM):', sub_1.shape, '| missing->fill:', int(sample_sub['id'].astype(str).map(_V28_PRED).isna().sum()))
sub_1

In [ ]:
# [graft] sub_2 = v27 近傍ヒューリスティック(r0 smooth PF + XY近傍疑似typewell + beam+hold+pp)
sample = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')
_fb2 = float(np.nanmedian(list(_V27_PRED.values())))
sub_2 = sample[['id']].copy()
sub_2['tvt'] = sub_2['id'].astype(str).map(_V27_PRED).fillna(_fb2)
print('[graft] sub_2(v27heur):', sub_2.shape, '| missing->fill:', int(sample['id'].astype(str).map(_V27_PRED).isna().sum()))

In [ ]:
# [graft] sub_2 は上のセルで _V27_PRED から構築済み(旧 selector ループ廃止)

In [ ]:
sub = (
    sub_1.merge(sub_2, on='id', suffixes=('_1', '_2'))
       .assign(tvt=lambda x: 0.3 * x['tvt_1'] + 0.7 * x['tvt_2'])
       [['id', 'tvt']]
)
sub.to_csv("submission.csv", index=False)
sub

In [ ]:
# === robust low-order PROJECTION post-processing (patched degree=4, blend=0.75) (CV-validated: raw PF -0.54, deployed components -0.33) ===
# Runs AFTER the 0.3*ridge+0.7*selector blend writes submission.csv; OVERWRITES it with the projected
# version. Per-well robust deg-5 fit of dU = tvt + Z - anchor vs normalized MD -> denoise jitter +
# down-weight wrong-branch outliers. Deterministic; defensive per-well fallback to raw.
import numpy as _np, pandas as _pd
def _robfit(s, y, deg=5):
    if len(s) < deg + 2:
        return y.copy()
    c = _np.polyfit(s, y, deg)
    for _ in range(4):
        r = y - _np.polyval(c, s)
        sc = _np.median(_np.abs(r)) * 1.4826 + 1e-6
        c = _np.polyfit(s, y, deg, w=1.0 / (1.0 + (r / (2.0 * sc)) ** 2))
    return _np.polyval(c, s)
try:
    _base = _pd.read_csv("submission.csv")   # the just-written blended submission
    assert set(['id','tvt']).issubset(_base.columns)
    _base['well'] = _base['id'].str[:8]
    _base['row_idx'] = _base['id'].str[9:].astype(int)
    _out = dict(zip(_base['id'].values, _base['tvt'].astype(float).values))
    _n_ok = 0
    for _wid, _g in _base.groupby('well'):
        try:
            _hw = _pd.read_csv(CFG.dataset_path / 'test' / (_wid + '__horizontal_well.csv'))
            _kn = _hw[_hw['TVT_input'].notna()]
            if len(_kn) < 5:
                continue
            _last = _kn.iloc[-1]
            _anchor = float(_last['TVT_input']) + float(_last['Z'])
            _ps = float(_last['MD']); _end = float(_hw['MD'].iloc[-1])
            _gi = _g.sort_values('row_idx')
            _ri = _gi['row_idx'].values
            _Z = _hw['Z'].values[_ri].astype(float)
            _md = _hw['MD'].values[_ri].astype(float)
            _s = (_md - _ps) / max(_end - _ps, 1e-6)
            _tvt = _gi['tvt'].values.astype(float)
            _fit = _robfit(_s, (_tvt + _Z) - _anchor, 4)
            _tvt_fit_full = (_anchor + _fit) - _Z
            _tvt_fit = 0.25 * _tvt + 0.75 * _tvt_fit_full
            if not _np.all(_np.isfinite(_tvt_fit)):
                continue
            for _rid, _val in zip(_gi['id'].values, _tvt_fit):
                _out[_rid] = float(_val)
            _n_ok += 1
        except Exception as _e:
            print('proj fallback', _wid, _e)
    print('projection applied to', _n_ok, 'wells')
    _final = _base[['id']].copy()
    _final['tvt'] = _final['id'].map(_out).astype(float)
    _final[['id','tvt']].to_csv("submission.csv", index=False)
    print('wrote projected submission.csv', _final.shape)
except Exception as _e:
    print('PROJECTION SKIPPED (kept blended submission):', _e)

In [ ]:
# [graft] 旧GBM_A(tabular)は v28 に置換のためスキップ
pass

In [ ]:
from pathlib import Path as _BlendPath
import pandas as _blend_pd
_sp45_path = _BlendPath('/kaggle/working/submission.csv') if _BlendPath('/kaggle/working').exists() else _BlendPath('submission.csv')
_sp45_df = _blend_pd.read_csv(_sp45_path)
_sp45_df.to_csv((_BlendPath('/kaggle/working') if _BlendPath('/kaggle/working').exists() else _BlendPath('.')) / 'sp45_projection_submission.csv', index=False)
print('saved sp45_projection_submission.csv', _sp45_df.shape, flush=True)


# === fleongg pretrained inference section ===

In [ ]:
import os, sys, glob, time, warnings, multiprocessing
from pathlib import Path
import numpy as np
import pandas as pd
from numba import njit
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")
os.environ.setdefault("SHOW_FIGS", "0")

# ---- environment / paths (Kaggle or local) -------------------------------------
def _find_data():
    for c in ["/kaggle/input/competitions/rogii-wellbore-geology-prediction",
              "/kaggle/input/rogii-wellbore-geology-prediction"]:
        if Path(c).exists() and (Path(c)/"train").exists():
            return Path(c)
    # fallback: find any mounted folder that contains a train/ directory
    for p in glob.glob("/kaggle/input/**/train", recursive=True):
        return Path(p).parent
    return Path(os.environ.get("ROGII_DATA", "."))   # local override for development

class CFG:
    DATA = _find_data()
    OUT  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    seed = 42
    n_splits = 5
    n_jobs = min(8, multiprocessing.cpu_count())
    # lik-PF
    PF_SEEDS = 128
    PF_PARTICLES = 500
    PF_SCALES = (3., 5., 8., 12.)
    # FAST dev (local smoke test): limit train wells & trees
    FAST = bool(int(os.environ.get("FAST", "0")))
    N_TRAIN_WELLS = int(os.environ.get("N_TRAIN_WELLS", "0"))  # 0 = all
    USE_GPU = os.environ.get("USE_GPU", "auto")
    SHOW_FIGS = os.environ.get("SHOW_FIGS", "1") == "1"   # EDA plots (on in the notebook)

FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
def _demo_well():
    """A train well with TVT + a sizable eval zone, for the EDA plots."""
    for w in sorted(p.stem.replace("__horizontal_well", "")
                    for p in (CFG.DATA/"train").glob("*__horizontal_well.csv")):
        try:
            d = pd.read_csv(CFG.DATA/"train"/f"{w}__horizontal_well.csv", usecols=["TVT", "TVT_input"])
        except Exception:
            continue
        if "TVT" in d and d.TVT.notna().any() and d.TVT_input.isna().sum() > 2000:
            return w
    return None
print("DATA:", CFG.DATA, "| OUT:", CFG.OUT, "| cores:", CFG.n_jobs, "| FAST:", CFG.FAST)

def load_well(wid, split="train"):
    base = CFG.DATA / split
    hw = pd.read_csv(base / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(base / f"{wid}__typewell.csv").sort_values("TVT")
    return hw, tw

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float))**2)))

In [ ]:
def fig_overview(wid):
    import matplotlib.pyplot as plt
    hw, tw = load_well(wid)
    kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]; ps = kn.MD.iloc[-1]
    fig, ax = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True)
    ax[0].plot(hw.MD, hw.Z, lw=1.2, color="#333"); ax[0].axvline(ps, color="crimson", ls="--", label="PS")
    ax[0].set_ylabel("Z / TVD (ft)"); ax[0].legend(loc="upper right")
    ax[0].set_title(f"Well {wid}: trajectory · gamma-ray · TVT target")
    ax[1].plot(kn.MD, kn.GR, lw=.7, color="steelblue", label="GR known")
    ax[1].plot(ev.MD, ev.GR, lw=.7, color="darkorange", label="GR eval"); ax[1].axvline(ps, color="crimson", ls="--")
    ax[1].set_ylabel("GR (API)"); ax[1].legend(loc="upper right")
    ax[2].plot(kn.MD, kn.TVT, lw=1.6, color="seagreen", label="TVT known (=input)")
    ax[2].plot(ev.MD, ev.TVT, lw=1.6, color="crimson", label="TVT to predict"); ax[2].axvline(ps, color="crimson", ls="--")
    ax[2].set_ylabel("TVT (ft)"); ax[2].set_xlabel("MD (ft)"); ax[2].invert_yaxis(); ax[2].legend(loc="upper right")
    for a in ax: a.grid(alpha=.25)
    plt.tight_layout(); plt.show()

def fig_correlation(wid):
    import matplotlib.pyplot as plt
    hw, tw = load_well(wid); ev = hw[hw.TVT_input.isna()]
    fig, ax = plt.subplots(1, 2, figsize=(11, 6))
    ax[0].plot(tw.GR, tw.TVT, lw=1.0, color="black")
    ax[0].set_xlabel("GR (API)"); ax[0].set_ylabel("TVT (ft)"); ax[0].invert_yaxis()
    ax[0].set_title("Typewell signature: GR vs TVT")
    sc = ax[1].scatter(ev.GR, ev.TVT, s=4, c=ev.MD, cmap="viridis")
    ax[1].set_xlabel("GR (API)"); ax[1].set_ylabel("TVT (ft)"); ax[1].invert_yaxis()
    ax[1].set_title("Horizontal GR at its true TVT\nmatches the typewell signature")
    plt.colorbar(sc, ax=ax[1], label="MD (ft)")
    for a in ax: a.grid(alpha=.25)
    plt.tight_layout(); plt.show()

def fig_drift_tail(n_wells=250):
    import matplotlib.pyplot as plt
    wids = sorted(p.stem.replace("__horizontal_well", "") for p in (CFG.DATA/"train").glob("*__horizontal_well.csv"))
    rng = np.random.default_rng(1); samp = sorted(rng.choice(wids, min(n_wells, len(wids)), replace=False).tolist())
    per = []
    for wid in samp:
        try: hw = pd.read_csv(CFG.DATA/"train"/f"{wid}__horizontal_well.csv", usecols=["TVT_input", "TVT"])
        except: continue
        ev = hw[hw.TVT_input.isna()]; kn = hw[hw.TVT_input.notna()]
        if len(ev) == 0 or len(kn) < 10 or hw.TVT.isna().all(): continue
        t = ev.TVT.values
        if np.isnan(t).any(): continue
        per.append(np.sqrt(np.mean((t-kn.TVT_input.iloc[-1])**2)))
    per = np.array(per); srt = np.sort(per)[::-1]; cum = np.cumsum(srt**2)/np.sum(srt**2)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].hist(per, bins=40, color="indianred", alpha=.85)
    ax[0].axvline(np.median(per), color="k", ls="--", label=f"median={np.median(per):.1f}")
    ax[0].axvline(per.mean(), color="b", ls="--", label=f"mean={per.mean():.1f}")
    ax[0].set_xlabel("per-well last-known-baseline RMSE (ft)"); ax[0].set_ylabel("wells"); ax[0].legend()
    ax[0].set_title("Per-well error is heavily right-skewed")
    ax[1].plot(np.arange(1, len(srt)+1)/len(srt)*100, cum*100, color="purple"); ax[1].axhline(80, color="gray", ls=":")
    ax[1].set_xlabel("% of wells (worst first)"); ax[1].set_ylabel("% of pooled squared error")
    ax[1].set_title("A few drift wells dominate the metric")
    for a in ax: a.grid(alpha=.25)
    plt.tight_layout(); plt.show()

In [ ]:
DEMO = "00bbac68" if (CFG.DATA/"train"/"00bbac68__horizontal_well.csv").exists() else _demo_well()
if CFG.SHOW_FIGS:
    print("demo well:", DEMO)
    if DEMO:
        fig_overview(DEMO)
        fig_correlation(DEMO)
    fig_drift_tail()

In [ ]:
# ---- single particle filters (ANCC-anchored & Z-velocity-coupled), numba ---------
PF_N = 600; ANCC_N = 600
PF_MOM = 0.993; PF_VN = 0.005; PF_PN = 0.01
PF_GR_SIG_MIN = 10.; PF_GR_SIG_MAX = 60.; PF_GR_SIG_DEF = 30.
PF_GR_WIN = 5; PF_GR_WT = 0.3; PF_RESAMP = 0.5; PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003
ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001

BEAMS = [(10,20.,144.,2,"cons"),(10,8.,64.,2,"loose"),(8,35.,220.,1,"vcons"),
         (10,14.,90.,5,"sm5"),(20,4.,36.,3,"vloose"),(12,12.,100.,3,"mid"),(15,25.,180.,2,"stiff")]

@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1] = cum[j]+w[j]
    u0 = np.random.uniform(0., 1./N); np2 = np.empty(N); na = np.empty(N); ci = 0
    for j in range(N):
        u = u0+j/N
        while ci < N-1 and cum[ci+1] < u: ci += 1
        np2[j] = pos[ci]+rp*np.random.randn(); na[j] = aux[ci]+rv*np.random.randn()
    return np2, na

@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS*6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx+d
                if ni < 0 or ni >= nt: continue
                tot = cost+(gv-tw_gr[ni])**2/es+mc*(d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i+1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]; cC[i], cC[mi] = cC[mi], cC[i]; cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n-1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

@njit(cache=True)
def _pf_ancc(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N, ALPHA, RN, PN, IS, RP, RR, RESAMP):
    pos = np.empty(N); rate = np.empty(N); w = np.ones(N)/N
    for j in range(N):
        pos[j] = ls+IS*np.random.randn(); rate[j] = ir+0.01*np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0]-1.
    for i in range(len(md_v)):
        dm = md_v[i]-pm; dm = max(dm, 1.)
        for j in range(N):
            rate[j] = ALPHA*rate[j]+RN*np.random.randn(); pos[j] += rate[j]*dm+PN*np.random.randn()
            tvt_j = pos[j]-z_v[i]; tvt_j = max(tvt_j, vmin-50.); tvt_j = min(tvt_j, vmin+len(gg)*step+50.)
            pos[j] = tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                eg = _interp1(gg, pos[j]-z_v[i], vmin, step); d = (gr_v[i]-eg)/gs
                lk = max(np.exp(-0.5*d*d) if d*d < 600. else 0., 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1./N
        ne = 0.
        for j in range(N): ne += w[j]*w[j]
        if 1./ne < RESAMP*N:
            pos, rate = _resamp(pos, rate, w, N, RP, RR)
            for j in range(N): w[j] = 1./N
        tv = 0.
        for j in range(N): tv += w[j]*(pos[j]-z_v[i])
        pts[i] = tv; va = 0.
        for j in range(N): va += w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i] = va**0.5; pm = md_v[i]
    return pts, std_

@njit(cache=True)
def _pf_z(md_v, z_v, gr_v, gr_sm_v, gg_p, gg_s, vmin, step, gs, ip, iv, beta, icpt, zsig, N,
         MOM, VN, PN, GR_WT, RP, RV, RESAMP):
    pos = np.empty(N); vel = np.empty(N); w = np.ones(N)/N
    for j in range(N):
        pos[j] = ip+0.5*np.random.randn(); vel[j] = iv+0.02*np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0]-1.; pz = z_v[0]-1.
    for i in range(len(md_v)):
        dm = md_v[i]-pm; dm = max(dm, 1.); dzd = (z_v[i]-pz)/dm; ve = beta*dzd+icpt
        for j in range(N):
            vel[j] = MOM*vel[j]+VN*np.random.randn(); pos[j] += vel[j]*dm+PN*np.random.randn()
            pos[j] = max(pos[j], vmin-50.); pos[j] = min(pos[j], vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                ep = _interp1(gg_p, pos[j], vmin, step); dp = (gr_v[i]-ep)/gs
                lp = max(np.exp(-0.5*dp*dp) if dp*dp < 600. else 0., 1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es = _interp1(gg_s, pos[j], vmin, step); ds = (gr_sm_v[i]-es)/(gs*1.5)
                    lsm = max(np.exp(-0.5*ds*ds) if ds*ds < 600. else 0., 1e-300); lk = (1.-GR_WT)*lp+GR_WT*lsm
                else: lk = lp
                lk = max(lk, 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1./N
        ws2 = 0.
        for j in range(N):
            dv = (vel[j]-ve)/max(zsig*2., 0.005); lz = max(np.exp(-0.5*dv*dv) if dv*dv < 600. else 0., 1e-300)
            w[j] *= lz; ws2 += w[j]
        if ws2 > 0.:
            for j in range(N): w[j] /= ws2
        else:
            for j in range(N): w[j] = 1./N
        ne = 0.
        for j in range(N): ne += w[j]*w[j]
        if 1./ne < RESAMP*N:
            pos, vel = _resamp(pos, vel, w, N, RP, RV)
            for j in range(N): w[j] = 1./N
        wm = 0.
        for j in range(N): wm += w[j]*pos[j]
        pts[i] = wm; va = 0.
        for j in range(N): va += w[j]*(pos[j]-wm)**2
        std_[i] = va**0.5; pm = md_v[i]; pz = z_v[i]
    return pts, std_

def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax+step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)

def _gr_sig(hw, tw_tvt, tw_gr):
    kn = hw[hw.TVT_input.notna() & hw.GR.notna()]
    if len(kn) < 20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn.GR.values-np.interp(kn.TVT_input.values, tw_tvt, tw_gr)),
                         PF_GR_SIG_MIN, PF_GR_SIG_MAX))

def _nn(arr, v):
    i = int(np.searchsorted(arr, v, "left"))
    if i >= len(arr): return len(arr)-1
    if i > 0 and abs(arr[i-1]-v) <= abs(arr[i]-v): return i-1
    return i

def _smooth(vals, fb, r):
    s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(fb)
    return (s.rolling(r*2+1, center=True, min_periods=1).mean() if r > 0 else s).to_numpy(np.float32)

def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt); sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    return tw_tvt[_beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))].astype(np.float32)

def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr); kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    ls = float(kn.TVT_input.iloc[-1]+kn.Z.iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail.TVT_input.values); dz = np.diff(tail.Z.values); dm = np.diff(tail.MD.values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    pts, std = _pf_ancc(ev.MD.values.astype(np.float64), ev.Z.values.astype(np.float64), ev.GR.values.astype(np.float64),
                        gg, gmin, gst, gs, ls, ir, N, ANCC_ALPHA, ANCC_RN, ANCC_PN, ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)

def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr); tw_s = pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1).mean().values.astype(np.float32)
    kna = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    dz_k = np.diff(kna.Z.values); dvt = np.diff(kna.TVT_input.values); dmd_k = np.diff(kna.MD.values); m2 = dmd_k > 0
    if m2.sum() >= 10:
        vz = dz_k[m2]/dmd_k[m2]; vt = dvt[m2]/dmd_k[m2]; A = np.column_stack([vz, np.ones_like(vz)])
        c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt, zsig = float(c[0]), float(c[1]), max(float(np.std(vt-(c[0]*vz+c[1]))), 0.001)
    else: beta, icpt, zsig = -1., 0., 0.1
    t2 = kna.tail(20); dvt2 = np.diff(t2.TVT_input.values); dmd2 = np.diff(t2.MD.values); m3 = dmd2 > 0
    iv = float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr); gs2, _, _ = _grid(tw_tvt, tw_s)
    gr_sm = hw.GR.rolling(PF_GR_WIN, center=True, min_periods=1).mean()
    pts, std = _pf_z(ev.MD.values.astype(np.float64), ev.Z.values.astype(np.float64), ev.GR.values.astype(np.float64),
                     gr_sm.loc[ev.index].values.astype(np.float64), gg, gs2, gmin, gst, gs,
                     float(kna.TVT_input.iloc[-1]), iv, beta, icpt, zsig, N,
                     PF_MOM, PF_VN, PF_PN, PF_GR_WT, PF_ROUGH_P, PF_ROUGH_V, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)

def multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2*hw+1; nk = len(kgr); nh = len(hgr)
        if nk < win+1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk-win+1, stride, dtype=np.int32)
        if len(sts) == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None]+np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C-C.mean(1, keepdims=True))/(C.std(1, keepdims=True)+1e-6)
        hp = np.pad(hg, hw, mode="edge"); H = hp[np.arange(nh)[:, None]+np.arange(win)[None, :]].astype(np.float32)
        Hn = (H-H.mean(1, keepdims=True))/(H.std(1, keepdims=True)+1e-6)
        ncc = Hn@Cn.T/win; best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw, 0, nk-1)].astype(np.float32), score))
    tvts = np.stack([o[0] for o in out], 1); scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3.*scores); sw /= sw.sum(1, keepdims=True)+1e-9
    return out, (tvts*sw).sum(1).astype(np.float32)

In [ ]:
# ---- 128-seed likelihood-weighted particle filter (the workhorse), numba ---------
@njit(cache=True, nogil=True)
def _pf_lik_allseeds(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N, n_seeds, seed_base,
                     MOM, VN, PN, RP, RR, RESAMP, init_spr):
    n = len(md_v); preds = np.empty((n_seeds, n)); liks = np.empty(n_seeds); tmax = vmin + len(gg)*step
    for s in range(n_seeds):
        np.random.seed(seed_base + s)
        pos = np.empty(N); rate = np.empty(N); w = np.ones(N)/N
        for j in range(N):
            pos[j] = ls + init_spr*np.random.randn(); rate[j] = ir + 0.01*np.random.randn()
        log_lik = 0.0; prev_md = md_v[0] - 1.0
        for i in range(n):
            dm = md_v[i] - prev_md
            if dm < 1.0: dm = 1.0
            for j in range(N):
                rate[j] = MOM*rate[j] + VN*np.random.randn(); pos[j] += rate[j]*dm + PN*np.random.randn()
                tvt_j = pos[j] - z_v[i]
                if tvt_j < vmin-100.: tvt_j = vmin-100.
                if tvt_j > tmax+100.: tvt_j = tmax+100.
                pos[j] = tvt_j + z_v[i]
            avg_lk = 0.0
            for j in range(N):
                eg = _interp1(gg, pos[j]-z_v[i], vmin, step); d = (gr_v[i]-eg)/gs; dd = d*d
                if dd > 600.: dd = 600.
                lk = np.exp(-0.5*dd)
                if lk < 1e-300: lk = 1e-300
                avg_lk += w[j]*lk; w[j] = w[j]*lk
            if avg_lk < 1e-300: avg_lk = 1e-300
            log_lik += np.log(avg_lk)
            ws = 0.0
            for j in range(N): ws += w[j]
            if ws > 0.0:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1./N
            neff = 0.0
            for j in range(N): neff += w[j]*w[j]
            neff = 1.0/neff
            if neff < RESAMP*N:
                cum = np.empty(N); c = 0.0
                for j in range(N): c += w[j]; cum[j] = c
                u0 = np.random.uniform(0., 1./N); newpos = np.empty(N); newrate = np.empty(N); ci = 0
                for j in range(N):
                    u = u0 + j/N
                    while ci < N-1 and cum[ci] < u: ci += 1
                    newpos[j] = pos[ci] + RP*np.random.randn(); newrate[j] = rate[ci] + RR*np.random.randn()
                for j in range(N): pos[j] = newpos[j]; rate[j] = newrate[j]; w[j] = 1./N
            est = 0.0
            for j in range(N): est += w[j]*(pos[j]-z_v[i])
            preds[s, i] = est; prev_md = md_v[i]
        liks[s] = log_lik
    return preds, liks

def lik_pf(hw, tw, n_particles=CFG.PF_PARTICLES, n_seeds=CFG.PF_SEEDS, scales=CFG.PF_SCALES,
           init_spr=4.5, seed_base=0, with_quality=False):
    """Likelihood-weighted PF ensemble. Returns ({pf_scale_X: pred_eval}, ev_index[, quality])."""
    tw_s = tw.sort_values("TVT"); tw_tvt = tw_s.TVT.values.astype(float)
    tw_gr = tw_s.GR.fillna(tw_s.GR.mean()).values.astype(float)
    kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0: return {}, np.array([]), {}
    last = kn.iloc[-1]; ls = float(last.TVT_input) + float(last.Z)
    tw_at_k = np.interp(kn.TVT_input.values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn.GR.fillna(0).values - tw_at_k), 10., 60.))
    tail = kn.tail(30); dt = np.diff(tail.TVT_input.values); dz = np.diff(tail.Z.values); dm = np.diff(tail.MD.values); m = dm > 0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum() >= 3 else 0.0
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    gr_v = hw.GR.interpolate(limit_direction="both").fillna(tw_gr.mean()).values.astype(float)[ev.index]
    preds, liks = _pf_lik_allseeds(ev.MD.values.astype(float), ev.Z.values.astype(float), gr_v,
                                   gg, gmin, gst, gs, ls, ir, n_particles, n_seeds, seed_base,
                                   0.998, 0.002, 0.005, 0.1, 0.001, 0.5, init_spr)
    ln = liks - liks.max(); out = {}
    for sc in scales:
        wts = np.exp(ln/float(sc)); wts /= wts.sum(); out[f"pf_scale_{sc:g}"] = (wts[:, None]*preds).sum(0)
    out["pf_mean"] = preds.mean(0)
    q = {}
    if with_quality:
        q = {"pf_best_ll": float(liks.max())/len(ev), "pf_ll_spread": float(liks.std()),
             "pf_pt_std": preds.std(0).astype(np.float32), "pf_gr_sig": gs}
    return out, ev.index.values, q

# JIT warm-up so timings below are representative
_m = np.linspace(1, 50, 20); _z = np.zeros(20); _g = np.full(20, 50.); _gg = np.linspace(45, 55, 100)
_pf_ancc(_m, _z, _g, _gg, 45., .1, 20., 50., 0., 8, .998, .002, .005, .3, .1, .001, .5)
_pf_z(_m, _z, _g, _g, _gg, _gg, 45., .1, 20., 50., 0., -1., 0., .1, 8, .993, .005, .01, .3, .2, .003, .5)
_beam_jit(np.random.randn(30), np.random.randn(50), 25, 8, 15., 100.)
_pf_lik_allseeds(_m, _z, _g, _gg, 45., .1, 20., 50., 0., 64, 4, 0, .998, .002, .005, .1, .001, .5, 4.5)
print("trackers compiled.")

def fig_tracker_vs_truth(wid):
    import matplotlib.pyplot as plt
    hw, tw = load_well(wid); kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    tw_tvt = tw.TVT.to_numpy(np.float32); tw_gr = tw.GR.to_numpy(np.float32); last = float(kn.TVT_input.iloc[-1])
    pf, _ = run_pf_ancc(hw, tw_tvt, tw_gr); out, _, _ = lik_pf(hw, tw, scales=(3.,))
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(ev.MD, ev.TVT, lw=2.2, color="black", label="True TVT", zorder=5)
    ax.plot(ev.MD, np.full(len(ev), last), lw=1.1, color="gray", ls=":", label="last-known baseline")
    ax.plot(ev.MD, pf, lw=1.0, color="tab:blue", alpha=.8, label="single particle filter")
    ax.plot(ev.MD, out["pf_scale_3"], lw=1.5, color="crimson", alpha=.9, label="128-seed lik-weighted PF")
    ax.set_xlabel("MD (ft)"); ax.set_ylabel("TVT (ft)"); ax.invert_yaxis(); ax.grid(alpha=.25)
    ax.set_title(f"Well {wid}: trackers vs ground truth — the lik-PF resists drift"); ax.legend(loc="best")
    plt.tight_layout(); plt.show()

In [ ]:
if CFG.SHOW_FIGS and DEMO:
    fig_tracker_vs_truth(DEMO)

In [ ]:
PLANE_K = 10; DENSE_SPW = 60; DENSE_K = 20

def robust_slope(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float); m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.std(x[m]) < 1e-6: return 0.
    return float(np.polyfit(x[m], y[m], 1)[0])

def affine_cal(kgr, tw_at_k, min_pts=20):
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        return 1., float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1); return float(a), float(b)

def seg_b_well(ktvt, kz, form_col):
    bv = ktvt+kz-form_col; n = len(bv); b_full = float(np.median(bv))
    b_late = float(np.median(bv[max(0, n-50):])) if n >= 5 else b_full
    t1, t2 = n//3, 2*n//3
    b_early = float(np.median(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.median(bv[t1:max(t1+1, t2)])) if t2 > t1 else b_full
    w = np.exp(0.02*np.arange(n)); w /= w.sum()
    return b_full, b_early, b_mid, b_late, float(np.dot(w, bv))

class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            try: df = pd.read_csv(data_dir/f"{wid}__horizontal_well.csv", usecols=["X","Y"]+FORMATIONS).dropna()
            except: continue
            if len(df) == 0: continue
            row = {"wid": wid, "x": float(df.X.median()), "y": float(df.Y.median())}
            for c in FORMATIONS: row[f"{c}_m"] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df.wid)}
        xy = self.df[["x","y"]].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree = cKDTree(xy/self.scale); self.xa = self.df.x.to_numpy(); self.ya = self.df.y.to_numpy()
        self.fa = self.df[[f"{c}_m" for c in FORMATIONS]].to_numpy(np.float64)
    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q/self.scale; nf = min(k+5, len(self.df)); dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ordr = np.argpartition(dist, min(k-1, nf-1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1./(dk+1e-3), 0.).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w*xn; wy = w*yn
        A = np.zeros((len(q), 3, 3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs = np.stack([(wx[:,:,None]*fn).sum(1), (wy[:,:,None]*fn).sum(1), (w[:,:,None]*fn).sum(1)], 1)
        try: coef = np.linalg.solve(A, rhs)
        except:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try: coef[r] = np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq = xy_q[:,0]; Yq = xy_q[:,1]
        pred = (Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)

class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, an, wd = [], [], [], []
        for wid in well_ids:
            try: df = pd.read_csv(data_dir/f"{wid}__horizontal_well.csv", usecols=["X","Y","ANCC"]).dropna()
            except: continue
            if len(df) == 0: continue
            ix = np.linspace(0, len(df)-1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s.X.values); ys.append(s.Y.values); an.append(s.ANCC.values); wd.extend([wid]*len(s))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(an).astype(np.float32); self.wids = np.array(wd)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1., self.xy.std(0)); self.tree = cKDTree(self.xy/self.scale)
    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q/self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid: dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        ordr = np.argpartition(dist, min(k-1, nf-1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1./(dk+1e-3), 0.); sw = w.sum(1); safe = np.where(sw < 1e-9, 1., sw)
        a = self.ancc[ik]; ap = (a*w).sum(1)/safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((a-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32), np.sqrt(np.maximum(var, 0.)).astype(np.float32), np.where(vk, dk, np.inf).min(1).astype(np.float32)

_FI = None; _DI = None
ANCH_OFFS = np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80], np.float32)
BEAM_OFFS = np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40], np.float32)
SC_OFFS = np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30], np.float32)
PF_OFFS = SC_OFFS.copy()

In [ ]:
def build_well(hw_path, tw_path, is_train, likpf_map=None):
    global _FI, _DI
    wid = Path(hw_path).stem.replace("__horizontal_well", "")
    try: hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path).sort_values("TVT")
    except: return None
    if is_train and "TVT" not in hw.columns: return None
    kn = hw[hw.TVT_input.notna()]; ev = hw[hw.TVT_input.isna()]
    if len(ev) == 0 or len(kn) < 10: return None
    if is_train and hw.TVT.isna().all(): return None
    tw_tvt = tw.TVT.to_numpy(np.float32); tw_gr = tw.GR.to_numpy(np.float32)
    if len(tw_tvt) < 3: return None
    pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr)
    if len(pf_a) == 0: return None
    pf_z, std_z = run_pf_z(hw, tw_tvt, tw_gr)
    pf_use = pf_a.astype(np.float32); std_use = std_a.astype(np.float32)
    has_z = len(pf_z) == len(pf_a) and not np.any(np.isnan(pf_z))
    lk = kn.iloc[-1]; last_tvt = float(lk.TVT_input)
    gr_full = hw.GR.astype(float).interpolate(limit_direction="both").fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[ev.index[0]:].to_numpy(np.float32); kgr = gr_full.iloc[:len(kn)].to_numpy(np.float32)
    bpaths = {tag: beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for (bs, mc, es, r, tag) in BEAMS}
    beam_ref = (bpaths["cons"]+bpaths["sm5"])/2.
    ktvt = kn.TVT_input.to_numpy(np.float32)
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8, sc8s = sc_res[0]; sc15, sc15s = sc_res[1]; sc25, sc25s = sc_res[2]; sc_cons = (sc8+sc15+sc25)/3.
    sc_trust = float(np.clip(len(kn)/200., 0., 0.6)); hyb_ref = (1-sc_trust)*beam_ref+sc_trust*sc_ens
    tw_at_k = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32); a_cal, b_cal = affine_cal(kgr, tw_at_k)
    kmd = kn.MD.to_numpy(np.float32); kz = kn.Z.to_numpy(np.float32)
    pfx_rmse = float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
    slp_all = robust_slope(kmd, ktvt); slp_50 = robust_slope(kmd[-50:], ktvt[-50:]); slp_z = robust_slope(kz, ktvt)
    swid = wid if is_train else None
    xy_ev = ev[["X","Y"]].to_numpy(np.float64); xy_kn = kn[["X","Y"]].to_numpy(np.float64)
    form_ev, knn_d = _FI.impute(xy_ev, self_wid=swid); form_kn, _ = _FI.impute(xy_kn, self_wid=swid)
    z_kn = kn.Z.to_numpy(np.float32); z_ev = ev.Z.to_numpy(np.float32)
    tvt_fs = {}; form_rmse = {}; form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev+form_ev[:, fi2]+b_full).astype(np.float32)
        tvt_fs[f"tvtF_{fn}"]=tvt_f; tvt_fs[f"tvtFw_{fn}"]=(-z_ev+form_ev[:,fi2]+b_wls).astype(np.float32)
        tvt_fs[f"tvtF50_{fn}"]=(-z_ev+form_ev[:,fi2]+b_late).astype(np.float32)
        tvt_fs[f"bw_{fn}"]=np.float32(b_full); tvt_fs[f"bww_{fn}"]=np.float32(b_wls); tvt_fs[f"bw50_{fn}"]=np.float32(b_late)
        tvt_fs[f"bw_early_{fn}"]=np.float32(b_early); tvt_fs[f"bw_mid_{fn}"]=np.float32(b_mid)
        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2))); form_list.append(tvt_f)
    fs = np.stack(form_list, 1)
    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32); form_std_d=fs.std(1).astype(np.float32); form_rng_d=(fs.max(1)-fs.min(1)).astype(np.float32)
    d_ancc, d_std, d_dist = _DI.impute(xy_ev, self_wid=swid); d_kn, d_std_kn, _ = _DI.impute(xy_kn, self_wid=swid)
    _, b_de, b_dm, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn); b_d = float(np.median(ktvt+z_kn-d_kn))
    tvt_dense=(-z_ev+d_ancc+b_d).astype(np.float32); tvt_densew=(-z_ev+d_ancc+b_dw).astype(np.float32); tvt_dense50=(-z_ev+d_ancc+b_dl).astype(np.float32)
    res_kn = ktvt+z_kn-d_kn; d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))
    all_sigs=[pf_use]+list(bpaths.values())+[sc8,sc15,sc25,sc_ens,tvt_fs["tvtF_ANCC"],tvt_dense]
    sig_mat=np.stack(all_sigs,1); sig_std=sig_mat.std(1).astype(np.float32); sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)
    gr_s=pd.Series(gr_full.values); rolls={}
    for w in [5,21,51,101]:
        r=gr_s.rolling(w,center=True,min_periods=1); rolls[f"grm{w}"]=r.mean().iloc[ev.index].values.astype(np.float32); rolls[f"grs{w}"]=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1,5,15,30]:
        rolls[f"glag{lag}"]=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32); rolls[f"glead{lag}"]=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32); gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)).iloc[ev.index].values.astype(np.float32)
    hmd=ev.MD.to_numpy(np.float32); md_since=hmd-float(lk.MD)
    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32); slp_b_50=(last_tvt+slp_50*md_since).astype(np.float32)
    mdd=hw.MD.diff().replace(0,np.nan)
    dzdmd=(hw.Z.diff()/mdd).iloc[ev.index].values.astype(np.float32); dxdmd=(hw.X.diff()/mdd).iloc[ev.index].values.astype(np.float32); dydmd=(hw.Y.diff()/mdd).iloc[ev.index].values.astype(np.float32)
    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
    def sc(v): return np.full(nh, np.float32(v), np.float32)
    feats={"well":wid,"id":[f"{wid}_{i}" for i in ev.index],"last_known_tvt":sc(last_tvt),
        "pf_ancc":pf_use,"pf_ancc_std":std_use,"pf_ancc_delta":(pf_use-last_tvt).astype(np.float32),
        "pf_z":(pf_z.astype(np.float32) if has_z else sc(last_tvt)),"pf_z_delta":((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
        "pf_vs_z":((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f"beam_{t}_d":(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
        "beam_mean_d":np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
        "beam_std_d":np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
        "beam_med_d":np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
        "sc8_d":(sc8-np.float32(last_tvt)).astype(np.float32),"sc8_sc":sc8s,"sc15_d":(sc15-np.float32(last_tvt)).astype(np.float32),"sc15_sc":sc15s,
        "sc25_d":(sc25-np.float32(last_tvt)).astype(np.float32),"sc25_sc":sc25s,"sc_cons_d":(sc_cons-np.float32(last_tvt)).astype(np.float32),
        "sc_ens_d":(sc_ens-np.float32(last_tvt)).astype(np.float32),"sc_trust":sc(sc_trust),"hyb_d":(hyb_ref-np.float32(last_tvt)).astype(np.float32),
        "sig_std":sig_std,"sig_mean_d":sig_mean,**tvt_fs,**{f"frm_rmse_{fn}":sc(form_rmse[fn]) for fn in FORMATIONS},
        "form_mean_d":form_mean_d,"form_std_d":form_std_d,"form_rng_d":form_rng_d,
        "spatial_ancc_d":(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),"spatial_knn_dist":knn_d,
        "dense_ancc":d_ancc,"dense_std":d_std,"dense_dist":d_dist,"tvt_dense_d":(tvt_dense-last_tvt).astype(np.float32),
        "tvt_densew_d":(tvt_densew-last_tvt).astype(np.float32),"tvt_dense50_d":(tvt_dense50-last_tvt).astype(np.float32),
        "dense_rmse":sc(d_rmse),"dense_bias":sc(d_bias),"dense_nb_std":sc(d_nb_std),
        "pf_vs_spatial":(pf_use-tvt_fs["tvtF_ANCC"]).astype(np.float32),"pf_vs_dense":(pf_use-tvt_dense).astype(np.float32),
        "spatial_vs_dense":(tvt_fs["tvtF_ANCC"]-tvt_dense).astype(np.float32),"beam_vs_spatial":(bpaths["cons"]-tvt_fs["tvtF_ANCC"]).astype(np.float32),
        "sc_vs_beam":(sc_ens-bpaths["cons"]).astype(np.float32),"cal_a":sc(a_cal),"cal_b":sc(b_cal),
        "pfx_rmse":sc(pfx_rmse),"known_len":sc(len(kn)),"eval_len":sc(nh),"slp_all":sc(slp_all),"slp_50":sc(slp_50),"slp_z":sc(slp_z),
        "slp_b_d_all":(slp_b_all-last_tvt).astype(np.float32),"slp_b_d_50":(slp_b_50-last_tvt).astype(np.float32),
        "ktvt_range":sc(float(np.ptp(ktvt))),"ktvt_std":sc(float(ktvt.std())),"md_since":md_since,"frac":frac,"frac2":frac**2,"sqrt_frac":np.sqrt(frac),
        "z":z_ev,"dx":(ev.X-float(lk.X)).to_numpy(np.float32),"dy":(ev.Y-float(lk.Y)).to_numpy(np.float32),"dz":(z_ev-float(lk.Z)).astype(np.float32),
        "dxy":np.sqrt((ev.X-float(lk.X))**2+(ev.Y-float(lk.Y))**2).to_numpy(np.float32),"dzdmd":dzdmd,"dxdmd":dxdmd,"dydmd":dydmd,
        "gr":hgr,"gr_d1":gr_d1,"gr_d2":gr_d2,"gr_env":gr_env,"gr_nrg":gr_nrg,
        "gr_vs_tw_anc":hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),"gr_vs_slp_all":hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
        **{f"tda{int(o)}":hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
        **{f"tdbc{int(o)}":hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f"tdsc{int(o)}":hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f"tdpf{int(o)}":hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},
        "tw_range":sc(float(np.ptp(tw_tvt))),"tw_gr_mean":sc(float(tw_gr.mean()))}
    for k,v in rolls.items(): feats[k]=v
    res = pd.DataFrame(feats)
    if is_train: res["target"]=(ev.TVT.to_numpy(np.float32)-np.float32(last_tvt))
    return res

def init_imputers(train_wids):
    global _FI, _DI
    _FI = FormationPlaneKNN(train_wids, CFG.DATA/"train"); _DI = DenseANCCImputer(train_wids, CFG.DATA/"train")

def _likpf_rows(wid, split):
    hw, tw = load_well(wid, split)
    out, idx, _ = lik_pf(hw, tw)
    if not len(out): return None
    d = {"id": [f"{wid}_{i}" for i in idx]}
    for k, v in out.items():
        d["likpf_" + k.replace("pf_scale_", "scale_").replace("pf_mean", "mean")] = v.astype(np.float32)
    return pd.DataFrame(d)

def build_likpf(wids, split):
    # threads are safe here: the lik-PF numba kernel is compiled with nogil=True, so it
    # releases the GIL and parallelises across threads (no pickling of numba code needed).
    res = Parallel(n_jobs=CFG.n_jobs, prefer="threads")(delayed(_likpf_rows)(w, split) for w in wids)
    return pd.concat([r for r in res if r is not None], ignore_index=True)

def build_features(wids, split, is_train):
    paths = [CFG.DATA/split/f"{w}__horizontal_well.csv" for w in wids]
    res = Parallel(n_jobs=CFG.n_jobs, prefer="threads")(
        delayed(build_well)(str(p), str(p.parent/f"{p.stem.replace('__horizontal_well','')}__typewell.csv"), is_train)
        for p in paths if (p.parent/f"{p.stem.replace('__horizontal_well','')}__typewell.csv").exists())
    parts = [r for r in res if r is not None]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def add_likpf_features(df, likpf):
    df = df.merge(likpf, on="id", how="left")
    for c in [c for c in likpf.columns if c != "id"]:
        df[c] = df[c].fillna(df["last_known_tvt"]); df[c+"_d"] = (df[c]-df["last_known_tvt"]).astype(np.float32)
    return df

In [ ]:
def _device():
    if CFG.USE_GPU == "cpu": return "cpu", "CPU"
    if CFG.USE_GPU == "gpu": return "gpu", "GPU"
    try:  # detect a real NVIDIA GPU (Kaggle GPU accelerator) via nvidia-smi
        import subprocess
        if subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0:
            return "gpu", "GPU"
    except Exception:
        pass
    return "cpu", "CPU"

def lgb_configs(dev):
    base = dict(boosting_type="gbdt", objective="regression", verbose=-1, n_jobs=-1, max_bin=255)
    if dev == "gpu": base.update(device_type="gpu", gpu_use_dp=False)
    n = 600 if CFG.FAST else 5000
    return [
        dict(**base, num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
             colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, learning_rate=0.03, n_estimators=n, seed=123),
        dict(**base, num_leaves=64, min_child_samples=40, subsample=0.474, subsample_freq=1,
             colsample_bytree=0.393, reg_lambda=95.75, reg_alpha=10.79, min_child_weight=0.24,
             learning_rate=0.0093, n_estimators=min(2*n, 10000), random_state=0),
        dict(**base, num_leaves=64, min_child_samples=40, subsample=0.474, subsample_freq=1,
             colsample_bytree=0.393, reg_lambda=95.75, reg_alpha=10.79, min_child_weight=0.24,
             learning_rate=0.0093, n_estimators=min(2*n, 10000), random_state=29),
    ]

def cb_configs(dev):
    tt = "GPU" if dev == "gpu" else "CPU"
    n = 800 if CFG.FAST else 8000
    return [
        dict(iterations=n, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254,
             loss_function="RMSE", task_type=tt, od_type="Iter", od_wait=300, verbose=0, learning_rate=0.02, random_seed=7),
        dict(iterations=n, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254,
             loss_function="RMSE", task_type=tt, od_type="Iter", od_wait=300, verbose=0, learning_rate=0.03, random_seed=123),
    ]

def train_stack(train_df, test_df, features):
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    from catboost import CatBoostRegressor
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    dev, devname = _device(); print("device:", devname)
    X = train_df[features].values.astype(np.float32); y = train_df["target"].values.astype(np.float32)
    g = train_df["well"].values; Xt = test_df[features].values.astype(np.float32)
    cv = GroupKFold(CFG.n_splits); oof_cols = {}; test_cols = {}
    def run(name, make, fit_kw, is_lgb):
        # LightGBM: slice to best_iteration_ via num_iteration. CatBoost: use_best_model
        # already trims to the best tree, and its predict() takes no num_iteration kwarg.
        oof = np.zeros(len(train_df)); tp = np.zeros(len(test_df))
        for tr, va in cv.split(X, y, groups=g):
            m = make(); m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], **fit_kw)
            if is_lgb:
                it = m.best_iteration_
                oof[va] = m.predict(X[va], num_iteration=it); tp += m.predict(Xt, num_iteration=it) / CFG.n_splits
            else:
                oof[va] = m.predict(X[va]); tp += m.predict(Xt) / CFG.n_splits
        oof_cols[name] = oof; test_cols[name] = tp
        print(f"  {name}: OOF RMSE={rmse(y, oof):.4f}", flush=True)
    for i, p in enumerate(lgb_configs(dev)):
        run(f"lgb{i}", lambda p=p: LGBMRegressor(**p),
            dict(eval_metric="rmse", callbacks=[early_stopping(250, verbose=False), log_evaluation(0)]), True)
    for i, p in enumerate(cb_configs(dev)):
        run(f"cb{i}", lambda p=p: CatBoostRegressor(**p),
            dict(early_stopping_rounds=250, use_best_model=True), False)
    OOF = pd.DataFrame(oof_cols); TEST = pd.DataFrame(test_cols)
    rid = Ridge(alpha=1.66, positive=True, fit_intercept=True); meta = np.zeros(len(train_df))
    for tr, va in cv.split(OOF.values, y, groups=g):
        rid.fit(OOF.values[tr], y[tr]); meta[va] = rid.predict(OOF.values[va])
    rid.fit(OOF.values, y); meta_test = rid.predict(TEST.values)
    print(f"  ridge-stack OOF RMSE={rmse(y, meta):.4f}")
    return meta, meta_test, OOF, TEST

In [ ]:
class PP:   # tuned on 773-well GroupKFold OOF (Nelder-Mead + grid; the optimum is flat)
    alpha = 1.0         # global scale on the learned delta (tuned ~1.0)
    tau = 85.0          # warm-up length in ft: damps the first feet after PS (tuned ~90)
    w_pf = 0.0          # blending the model with the single PF no longer helps once lik-PF is a feature
    w_sub1 = 0.60       # weight on the learned model; lik-PF gets 1-w_sub1. CV optimum ~0.68 (flat
                        # 0.55-0.68); 0.60 is a small hedge toward the drift-robust lik-PF for LB transfer.
    sub2_scale = "scale_5"   # which likelihood-scale of the lik-PF to use as sub2 (3/5/8 ~equivalent)
    sg_win = 61         # per-well Savitzky-Golay smoothing window (effect is small, ~0.01 ft)
    sg_poly = 3

def warmup(md_since, tau): return 1.-np.exp(-np.maximum(md_since, 0.)/tau) if tau > 1e-6 else 1.0

def make_prediction(df, model_delta, likpf):
    last = df["last_known_tvt"].values.astype(float)
    pf_delta = df["pf_ancc"].values.astype(float) - last
    lp = df[f"likpf_{PP.sub2_scale}"].values.astype(float) - last
    sub1 = PP.alpha*warmup(df["md_since"].values.astype(float), PP.tau)*(model_delta*(1-PP.w_pf)+pf_delta*PP.w_pf)
    delta = PP.w_sub1*sub1 + (1-PP.w_sub1)*lp
    pred = last + delta
    # per-well Savitzky-Golay smoothing
    out = pred.copy(); dfx = df.reset_index(drop=True)
    for _, idx in dfx.groupby("well", sort=False).groups.items():
        pos = dfx.index.get_indexer(idx); v = pred[pos]; n = len(v); wl = min(PP.sg_win, n)
        if wl % 2 == 0: wl -= 1
        if wl >= PP.sg_poly+2: out[pos] = savgol_filter(v, wl, PP.sg_poly)
    return out

In [ ]:
# [graft] fleongg(2つ目のtabular GBM)も v28 に置換。gold/smoother 用に _FI/_DI(FormationPlaneKNN 等)だけ構築。
train_wids = sorted(p.stem.replace('__horizontal_well','') for p in (CFG.DATA/'train').glob('*__horizontal_well.csv'))
if getattr(CFG, 'N_TRAIN_WELLS', None): train_wids = train_wids[:CFG.N_TRAIN_WELLS]
print(f'[graft] init_imputers on {len(train_wids)} train wells (gold用 _FI/_DI)…', flush=True)
init_imputers(train_wids)
_fbf = float(np.nanmedian(list(_V28_PRED.values())))
sub = pd.read_csv(CFG.DATA/'sample_submission.csv')
sub['tvt'] = sub['id'].astype(str).map(_V28_PRED).fillna(_fbf)
sub.to_csv(CFG.OUT/'submission.csv', index=False)
cv_final = None
print(f'[graft] fleongg=v28 submission.csv written ({len(sub)} rows)')
sub.head()

In [ ]:
def fig_results():
    import matplotlib.pyplot as plt
    names = ["last-known", "LGBM (orig. feats)", "stack + lik-PF feats", "baseline recipe", "ours (final)"]
    vals = [15.91, 10.85, 9.69, 9.75, cv_final if cv_final else 9.21]
    colors = ["#bbb", "#7aa", "#5a8", "#caa", "crimson"]
    fig, ax = plt.subplots(figsize=(9, 4))
    b = ax.barh(names[::-1], vals[::-1], color=colors[::-1])
    for r, v in zip(b, vals[::-1]): ax.text(v+0.1, r.get_y()+r.get_height()/2, f"{v:.2f}", va="center")
    ax.set_xlabel("CV pooled-RMSE (ft, lower is better)"); ax.set_title("Ablation — GroupKFold CV")
    ax.grid(alpha=.25, axis="x"); plt.tight_layout(); plt.show()

if CFG.SHOW_FIGS:
    fig_results()

In [ ]:
from pathlib import Path as _FinalBlendPath
import numpy as _final_np
import pandas as _final_pd

_WORK = _FinalBlendPath('/kaggle/working') if _FinalBlendPath('/kaggle/working').exists() else _FinalBlendPath('.')
_BLEND_WEIGHTS_SP45 = (0.50, 0.52, 0.55, 0.58, 0.60)
_SELECTED_SP45_WEIGHT = 0.55
_INPUT_FILES = {
    'fleongg': _WORK / 'submission.csv',
    'sp45': _WORK / 'sp45_projection_submission.csv',
}


def _read_submission_frame(path, label):
    frame = _final_pd.read_csv(path)
    missing = {'id', 'tvt'} - set(frame.columns)
    if missing:
        raise RuntimeError(f'{label} submission is missing columns: {sorted(missing)}')

    frame = frame[['id', 'tvt']].copy()
    frame['id'] = frame['id'].astype(str)
    frame['tvt'] = frame['tvt'].astype(float)

    if not _final_np.isfinite(frame['tvt'].to_numpy(dtype=float)).all():
        raise RuntimeError(f'Non-finite values in {label} tvt')
    return frame


def _merge_blend_inputs(sp45, fleongg):
    merged = sp45.rename(columns={'tvt': 'tvt_sp45'}).merge(
        fleongg.rename(columns={'tvt': 'tvt_fleongg'}),
        on='id',
        how='inner',
    )
    if len(merged) != len(sp45) or len(merged) != len(fleongg):
        raise RuntimeError(
            f'Blend id mismatch: sp45={len(sp45)}, fleongg={len(fleongg)}, merged={len(merged)}'
        )
    return merged


def _weighted_submission(merged, w_sp45):
    w_fleongg = 1.0 - float(w_sp45)
    out = merged[['id']].copy()
    out['tvt'] = (
        float(w_sp45) * merged['tvt_sp45'].astype(float)
        + w_fleongg * merged['tvt_fleongg'].astype(float)
    )
    return out


def _candidate_report_row(candidate, merged, file_name, w_sp45):
    diff = candidate['tvt'].to_numpy(dtype=float) - merged['tvt_sp45'].to_numpy(dtype=float)
    return {
        'file': file_name,
        'w_sp45': float(w_sp45),
        'w_fleongg': float(1.0 - w_sp45),
        'rows': int(len(candidate)),
        'mean_tvt': float(candidate['tvt'].mean()),
        'std_tvt': float(candidate['tvt'].std()),
        'rmse_vs_sp45': float(_final_np.sqrt(_final_np.mean(diff * diff))),
        'p95_abs_vs_sp45': float(_final_np.quantile(_final_np.abs(diff), 0.95)),
    }


_fle = _read_submission_frame(_INPUT_FILES['fleongg'], 'fleongg')
_fle.to_csv(_WORK / 'fleongg_pretrained_submission.csv', index=False)
_sp45 = _read_submission_frame(_INPUT_FILES['sp45'], 'sp45')
_merged = _merge_blend_inputs(_sp45, _fle)

_report_rows = []
for _w_sp45 in _BLEND_WEIGHTS_SP45:
    _candidate = _weighted_submission(_merged, _w_sp45)
    _name = f'submission_sp45_fleongg_w{_w_sp45:.2f}.csv'
    _candidate.to_csv(_WORK / _name, index=False)
    _report_rows.append(_candidate_report_row(_candidate, _merged, _name, _w_sp45))

_final_name = f'submission_sp45_fleongg_w{_SELECTED_SP45_WEIGHT:.2f}.csv'
_final = _final_pd.read_csv(_WORK / _final_name)
_final.to_csv(_WORK / 'submission.csv', index=False)

_report = _final_pd.DataFrame(_report_rows)
_report.to_csv(_WORK / 'sp45_fleongg_blend_report.csv', index=False)
print(_report.to_string(index=False), flush=True)
print('wrote final submission.csv from', _final_name, _final.shape, flush=True)


In [ ]:
# Lesson learned: hidden rerun copies of "overlap" wells are NOT guaranteed to be
# same-version / row-aligned with their train copies - a blind 100% lookup can inject error.
# Guard: per well, validate the contacts reconstruction against the TEST copy's known
# prefix (TVT_input), interpolated BY MD (not row index); override only if rmse < 1 ft,
# and only rows whose MD lies inside the train copy's range. Otherwise keep the blend.
# By construction this is >= the plain blend: exact wells win, mismatched wells are skipped.
import os as _ov_os, glob as _ov_glob
import numpy as _ov_np, pandas as _ov_pd
from pathlib import Path as _OvPath

def _ov_tvt_from_contacts(hw_tr, tw_tr, ref_col="EGFDU"):
    tw_g = tw_tr.dropna(subset=["Geology"])
    ref_tvt = tw_g[tw_g["Geology"] == ref_col]["TVT"].min()
    if _ov_np.isnan(ref_tvt):
        ref_col = tw_g["Geology"].iloc[0]; ref_tvt = tw_g[tw_g["Geology"] == ref_col]["TVT"].min()
    offset = (hw_tr["TVT"] - (ref_tvt - (hw_tr["Z"] - hw_tr[ref_col]))).mean()
    return (ref_tvt - (hw_tr["Z"] - hw_tr[ref_col]) + offset).to_numpy(dtype=float)

try:
    _W = _OvPath("/kaggle/working") if _OvPath("/kaggle/working").exists() else _OvPath(".")
    _DATA = None
    for _c in [_OvPath("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
               _OvPath("/kaggle/input/rogii-wellbore-geology-prediction")]:
        if _c.exists() and (_c / "train").exists():
            _DATA = _c; break
    if _DATA is None:
        for _p in _ov_glob.glob("/kaggle/input/**/train/*__horizontal_well.csv", recursive=True):
            _DATA = _OvPath(_p).parent.parent; break
    _sub = _ov_pd.read_csv(_W / "submission.csv")
    _sub["well"] = _sub["id"].str[:8]; _sub["row_idx"] = _sub["id"].str[9:].astype(int)
    _pred = dict(zip(_sub["id"].astype(str), _sub["tvt"].astype(float)))
    _train_wells = set(_ov_os.path.basename(f).split("__")[0]
                       for f in _ov_glob.glob(str(_DATA / "train" / "*__horizontal_well.csv")))
    _n_ok = _n_skip = 0
    for _wid, _g in _sub.groupby("well"):
        if _wid not in _train_wells:
            continue
        try:
            _hw_te = _ov_pd.read_csv(_DATA / "test" / (_wid + "__horizontal_well.csv"))
            _hw_tr = _ov_pd.read_csv(_DATA / "train" / (_wid + "__horizontal_well.csv"))
            _tw_tr = _ov_pd.read_csv(_DATA / "train" / (_wid + "__typewell.csv"))
            _phys = _ov_tvt_from_contacts(_hw_tr, _tw_tr)
            _md_raw = _hw_tr["MD"].to_numpy(dtype=float)
            _m_fin = _ov_np.isfinite(_phys) & _ov_np.isfinite(_md_raw)
            if _m_fin.sum() < 100:
                print("override SKIP %s too few valid phys rows=%d" % (_wid, int(_m_fin.sum()))); _n_skip += 1; continue
            _o = _ov_np.argsort(_md_raw[_m_fin])
            _md_tr = _md_raw[_m_fin][_o]; _ph_tr = _phys[_m_fin][_o]
            # --- self-check: TEST copy known prefix (TVT_input) vs lookup, interpolated by MD ---
            _kn = _hw_te[_hw_te["TVT_input"].notna()]
            _kn = _kn[(_kn["MD"] >= _md_tr[0]) & (_kn["MD"] <= _md_tr[-1])]
            if len(_kn) < 50:
                print("override SKIP %s too few comparable known-prefix rows=%d" % (_wid, len(_kn))); _n_skip += 1; continue
            _rk = float(_ov_np.sqrt(_ov_np.mean(
                (_ov_np.interp(_kn["MD"].to_numpy(dtype=float), _md_tr, _ph_tr)
                 - _kn["TVT_input"].to_numpy(dtype=float)) ** 2)))
            if (not _ov_np.isfinite(_rk)) or _rk > 1.0:
                print("override SKIP %s known-prefix rmse=%.3f (train copy != test copy, keeping blend)" % (_wid, _rk))
                _n_skip += 1; continue
            # --- check passed -> override via MD interpolation (no row-index alignment), in-range rows only ---
            _md_te = _hw_te["MD"].to_numpy(dtype=float)
            _n_row = 0
            for _rid, _ri in zip(_g["id"].astype(str).values, _g["row_idx"].values):
                _ri = int(_ri)
                if 0 <= _ri < len(_md_te):
                    _m = float(_md_te[_ri])
                    if _md_tr[0] <= _m <= _md_tr[-1]:
                        _pred[_rid] = float(_ov_np.interp(_m, _md_tr, _ph_tr)); _n_row += 1
            print("override OK   %s known-prefix rmse=%.4f rows overridden=%d/%d" % (_wid, _rk, _n_row, len(_g)))
            _n_ok += 1
        except Exception as _e:
            print("override fallback %s: %s" % (_wid, _e)); _n_skip += 1
    _new = _sub["id"].astype(str).map(_pred).astype(float)
    assert _new.notna().all(), "override produced NaN, aborting"
    _sub["tvt"] = _new
    _sub[["id", "tvt"]].to_csv(_W / "submission.csv", index=False)
    print("GUARDED override done: overridden=%d skipped=%d (skipped = kept the blend)" % (_n_ok, _n_skip))
except Exception as _e:
    print("GUARDED override skipped entirely (kept the blend):", _e)

In [ ]:
# Version 2 artifact audit: verify the final file without changing predictions.
import hashlib as _audit_hashlib
import json as _audit_json
import pandas as _audit_pd
import numpy as _audit_np
from pathlib import Path as _AuditPath

_AUDIT_WORK = _AuditPath('/kaggle/working') if _AuditPath('/kaggle/working').exists() else _AuditPath('.')
_AUDIT_SUBMISSION = _AUDIT_WORK / 'submission.csv'

# CFG was redefined in the fleongg section (using .DATA instead of .dataset_path)
_AUDIT_DATA = getattr(CFG, 'DATA', getattr(CFG, 'dataset_path', _AuditPath('/kaggle/input/competitions/rogii-wellbore-geology-prediction')))
_AUDIT_SAMPLE = _AUDIT_DATA / 'sample_submission.csv'


def _sha256_file(path):
    h = _audit_hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def _build_submission_audit(sub_path, sample_path):
    sub = _audit_pd.read_csv(sub_path)
    sample = _audit_pd.read_csv(sample_path)
    if list(sub.columns) != ['id', 'tvt']:
        raise RuntimeError(f'Unexpected submission columns: {list(sub.columns)}')
    if len(sub) != len(sample):
        raise RuntimeError(f'Unexpected row count: submission={len(sub)} sample={len(sample)}')
    if not sub['id'].astype(str).equals(sample['id'].astype(str)):
        raise RuntimeError('Submission id order does not match sample_submission.csv')
    tvt = sub['tvt'].to_numpy(dtype=float)
    if not _audit_np.isfinite(tvt).all():
        raise RuntimeError('Submission contains non-finite tvt values')
    return {
        'rows': int(len(sub)),
        'columns': list(sub.columns),
        'id_order_matches_sample': True,
        'tvt_min': float(_audit_np.min(tvt)),
        'tvt_max': float(_audit_np.max(tvt)),
        'tvt_mean': float(_audit_np.mean(tvt)),
        'tvt_std': float(_audit_np.std(tvt)),
        'sha256_submission_csv': _sha256_file(sub_path),
    }


_audit = _build_submission_audit(_AUDIT_SUBMISSION, _AUDIT_SAMPLE)
with open(_AUDIT_WORK / 'version2_submission_audit.json', 'w', encoding='utf-8') as f:
    _audit_json.dump(_audit, f, indent=2, sort_keys=True)

# Keep a named copy for manual inspection; Kaggle still submits submission.csv.
_audit_pd.read_csv(_AUDIT_SUBMISSION).to_csv(_AUDIT_WORK / 'submission_version2_refactor_copy.csv', index=False)
print('Version 2 submission audit:', _audit, flush=True)

In [ ]:

# Gold visible-prefix calibration overlay.
# It runs AFTER the public final blend and guarded contact override.
# The public submission remains the anchor; this layer only makes a per-well move
# when the visible-prefix backtest says a geology/PF candidate beats the default tracker.
import os as _gold_os
import glob as _gold_glob
import json as _gold_json
import time as _gold_time
import hashlib as _gold_hashlib
from pathlib import Path as _GoldPath

import numpy as _gold_np
import pandas as _gold_pd

_GOLD_ENABLE = _gold_os.environ.get('ROGII_GOLD_PREFIX_CAL', '1') == '1'
_GOLD_PROFILE = _gold_os.environ.get('ROGII_GOLD_PROFILE', 'balanced').strip().lower()
_GOLD_INCLUDE_PF = _gold_os.environ.get('ROGII_GOLD_INCLUDE_PF', '1') == '1'
_GOLD_CAL_SEEDS = int(_gold_os.environ.get('ROGII_GOLD_CAL_SEEDS', '24'))
_GOLD_FINAL_SEEDS = int(_gold_os.environ.get('ROGII_GOLD_FINAL_SEEDS', '48'))
_GOLD_PARTICLES = int(_gold_os.environ.get('ROGII_GOLD_PARTICLES', '350'))
_GOLD_CUT_FRACS = tuple(float(x) for x in _gold_os.environ.get('ROGII_GOLD_CUT_FRACS', '0.55,0.70,0.84').split(',') if x.strip())
_GOLD_MAX_WELLS = int(_gold_os.environ.get('ROGII_GOLD_MAX_WELLS', '1000000'))

_GOLD_PROFILES = {
    'conservative': dict(min_gain=1.00, max_best=9.0, min_consistency=0.67, base=0.06, gain_scale=0.12, margin_scale=0.04, quality_bonus=0.02, cap=0.22, clip_base=8.0, clip_gain=3.0, clip_max=18.0, delta_soft=22.0, p95_hard=55.0),
    'balanced':     dict(min_gain=0.55, max_best=12.0, min_consistency=0.50, base=0.08, gain_scale=0.20, margin_scale=0.06, quality_bonus=0.04, cap=0.36, clip_base=10.0, clip_gain=4.5, clip_max=28.0, delta_soft=30.0, p95_hard=75.0),
    'aggressive':   dict(min_gain=0.25, max_best=15.0, min_consistency=0.34, base=0.12, gain_scale=0.32, margin_scale=0.10, quality_bonus=0.06, cap=0.56, clip_base=14.0, clip_gain=7.0, clip_max=45.0, delta_soft=42.0, p95_hard=110.0),
}
if _GOLD_PROFILE not in _GOLD_PROFILES:
    print(f'Unknown ROGII_GOLD_PROFILE={_GOLD_PROFILE!r}; using balanced')
    _GOLD_PROFILE = 'balanced'


def _gold_work_dir():
    return _GoldPath('/kaggle/working') if _GoldPath('/kaggle/working').exists() else _GoldPath('.')


def _gold_find_data():
    candidates = []
    obj = globals().get('CFG')
    if obj is not None:
        for attr in ('dataset_path', 'DATA'):
            if hasattr(obj, attr):
                candidates.append(_GoldPath(getattr(obj, attr)))
    candidates.extend([
        _GoldPath('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
        _GoldPath('/kaggle/input/rogii-wellbore-geology-prediction'),
        _GoldPath('.'),
    ])
    for c in candidates:
        try:
            if (c / 'train').exists() and (c / 'test').exists() and (c / 'sample_submission.csv').exists():
                return c
        except Exception:
            pass
    for p in _gold_glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True):
        c = _GoldPath(p).parent
        if (c / 'train').exists() and (c / 'test').exists():
            return c
    raise RuntimeError('Could not locate ROGII data directory')


def _gold_split_ids(df):
    out = df.copy()
    parts = out['id'].astype(str).str.rsplit('_', n=1, expand=True)
    if parts.shape[1] != 2:
        raise RuntimeError('Unexpected id format; expected well_rowindex')
    out['well'] = parts[0]
    out['row_idx'] = parts[1].astype(int)
    return out


def _gold_rmse(a, b):
    a = _gold_np.asarray(a, dtype=float)
    b = _gold_np.asarray(b, dtype=float)
    m = _gold_np.isfinite(a) & _gold_np.isfinite(b)
    if int(m.sum()) == 0:
        return float('inf')
    d = a[m] - b[m]
    return float(_gold_np.sqrt(_gold_np.mean(d * d)))


def _gold_sha256(path):
    h = _gold_hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def _gold_robust_poly_predict(x_known, y_known, x_all, deg):
    x_known = _gold_np.asarray(x_known, dtype=float)
    y_known = _gold_np.asarray(y_known, dtype=float)
    x_all = _gold_np.asarray(x_all, dtype=float)
    m = _gold_np.isfinite(x_known) & _gold_np.isfinite(y_known)
    x_known = x_known[m]
    y_known = y_known[m]
    if len(x_known) < 3:
        fill = float(_gold_np.nanmedian(y_known)) if len(y_known) else 0.0
        return _gold_np.full_like(x_all, fill, dtype=float)
    deg = int(min(max(1, deg), len(x_known) - 1))
    x0 = float(x_known[0])
    xs = float(_gold_np.nanmax(x_known) - _gold_np.nanmin(x_known))
    if (not _gold_np.isfinite(xs)) or xs < 1e-6:
        xs = 1.0
    xk = (x_known - x0) / xs
    xa = (x_all - x0) / xs
    try:
        coef = _gold_np.polyfit(xk, y_known, deg)
        for _ in range(5):
            fit = _gold_np.polyval(coef, xk)
            res = y_known - fit
            sc = 1.4826 * float(_gold_np.nanmedian(_gold_np.abs(res - _gold_np.nanmedian(res)))) + 1e-6
            weights = 1.0 / (1.0 + (res / (2.5 * sc)) ** 2)
            coef = _gold_np.polyfit(xk, y_known, deg, w=weights)
        return _gold_np.polyval(coef, xa).astype(float)
    except Exception:
        return _gold_np.full_like(x_all, float(_gold_np.nanmedian(y_known)), dtype=float)


def _gold_variant_grid():
    variants = set()
    try:
        variants.update(SELECTOR_BIN_VARIANTS.values())
        variants.add(SELECTOR_GLOBAL_VARIANT)
    except Exception:
        pass
    for scale in (3, 5, 8, 12):
        for hold in (0.0, 0.05, 0.10, 0.15, 0.20, 0.25):
            variants.add(f'pf_scale_{scale:g}_hold_{hold:g}')
        for beam in (0.05, 0.10, 0.20, 0.30):
            for hold in (0.0, 0.05, 0.10, 0.15, 0.20):
                variants.add(f'pf_scale_{scale:g}_beam_{beam:g}_hold_{hold:g}')
    return sorted(variants)


def _gold_tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    tw_g = tw_tr.dropna(subset=['Geology']) if 'Geology' in tw_tr.columns else tw_tr.copy()
    if 'Geology' in tw_g.columns and ref_col in hw_tr.columns:
        ref_tvt = tw_g.loc[tw_g['Geology'] == ref_col, 'TVT'].min()
        if _gold_np.isnan(ref_tvt):
            ref_col = tw_g['Geology'].iloc[0]
            ref_tvt = tw_g.loc[tw_g['Geology'] == ref_col, 'TVT'].min()
        offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
        return (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset).to_numpy(dtype=float)
    return hw_tr['TVT'].to_numpy(dtype=float)


def _gold_contact_candidate(wid, hw, data_dir):
    out = {}
    try:
        hw_tr_path = data_dir / 'train' / f'{wid}__horizontal_well.csv'
        tw_tr_path = data_dir / 'train' / f'{wid}__typewell.csv'
        if not hw_tr_path.exists() or not tw_tr_path.exists():
            return out
        hw_tr = _gold_pd.read_csv(hw_tr_path)
        tw_tr = _gold_pd.read_csv(tw_tr_path)
        phys = _gold_tvt_from_contacts(hw_tr, tw_tr)
        md = hw_tr['MD'].to_numpy(dtype=float)
        m = _gold_np.isfinite(md) & _gold_np.isfinite(phys)
        if int(m.sum()) < 100:
            return out
        order = _gold_np.argsort(md[m])
        md_s = md[m][order]
        ph_s = phys[m][order]
        pred = _gold_np.interp(hw['MD'].to_numpy(dtype=float), md_s, ph_s, left=_gold_np.nan, right=_gold_np.nan)
        out['contact_md_lookup'] = pred.astype(float)
    except Exception:
        return out
    return out


def _gold_poly_candidates(hw_masked):
    out = {}
    tvt = hw_masked['TVT_input'].to_numpy(dtype=float)
    md = hw_masked['MD'].to_numpy(dtype=float)
    z = hw_masked['Z'].to_numpy(dtype=float)
    kn = _gold_np.flatnonzero(_gold_np.isfinite(tvt) & _gold_np.isfinite(md) & _gold_np.isfinite(z))
    if len(kn) < 30:
        return out
    u = tvt + z
    for tail in (80, 160, 320, 640, 1000000):
        sel = kn[-min(int(tail), len(kn)):]
        if len(sel) < 30:
            continue
        tag = 'all' if tail >= 1000000 else f'tail{tail}'
        for deg in (1, 2, 3):
            if len(sel) < deg + 12:
                continue
            uhat = _gold_robust_poly_predict(md[sel], u[sel], md, deg)
            out[f'poly_u_deg{deg}_{tag}'] = (uhat - z).astype(float)
    return out


def _gold_surface_candidates(hw_masked, wid, data_dir):
    out = {}
    tvt = hw_masked['TVT_input'].to_numpy(dtype=float)
    z = hw_masked['Z'].to_numpy(dtype=float)
    xy = hw_masked[['X', 'Y']].to_numpy(dtype=float)
    kn = _gold_np.isfinite(tvt) & _gold_np.isfinite(z) & _gold_np.isfinite(xy).all(axis=1)
    if int(kn.sum()) < 30:
        return out
    formations = list(globals().get('FORMATIONS', ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']))
    fi = globals().get('_FI', globals().get('FI', None))
    di = globals().get('_DI', globals().get('DI', None))
    surf_names = []
    try:
        if fi is not None:
            form_all, _ = fi.impute(xy, self_wid=None)
            form_all = _gold_np.asarray(form_all, dtype=float)
            for i, fn in enumerate(formations[:form_all.shape[1]]):
                f = form_all[:, i]
                good = kn & _gold_np.isfinite(f)
                if int(good.sum()) < 30:
                    continue
                b_med = float(_gold_np.nanmedian(tvt[good] + z[good] - f[good]))
                out[f'surface_{fn}_median'] = (-z + f + b_med).astype(float)
                surf_names.append(f'surface_{fn}_median')
                if callable(globals().get('seg_b_well')):
                    try:
                        b_full, _, _, b_late, b_wls = seg_b_well(
                            tvt[good].astype(_gold_np.float32),
                            z[good].astype(_gold_np.float32),
                            f[good].astype(_gold_np.float32),
                        )
                        out[f'surface_{fn}_full'] = (-z + f + float(b_full)).astype(float)
                        out[f'surface_{fn}_late'] = (-z + f + float(b_late)).astype(float)
                        out[f'surface_{fn}_wls'] = (-z + f + float(b_wls)).astype(float)
                        surf_names.extend([f'surface_{fn}_full', f'surface_{fn}_late', f'surface_{fn}_wls'])
                    except Exception:
                        pass
    except Exception as e:
        print('surface imputer skipped', wid, e)
    try:
        if di is not None:
            dense, _, _ = di.impute(xy, self_wid=None)
            dense = _gold_np.asarray(dense, dtype=float)
            good = kn & _gold_np.isfinite(dense)
            if int(good.sum()) >= 30:
                b_med = float(_gold_np.nanmedian(tvt[good] + z[good] - dense[good]))
                out['dense_ancc_median'] = (-z + dense + b_med).astype(float)
                surf_names.append('dense_ancc_median')
                if callable(globals().get('seg_b_well')):
                    try:
                        b_full, _, _, b_late, b_wls = seg_b_well(
                            tvt[good].astype(_gold_np.float32),
                            z[good].astype(_gold_np.float32),
                            dense[good].astype(_gold_np.float32),
                        )
                        out['dense_ancc_full'] = (-z + dense + float(b_full)).astype(float)
                        out['dense_ancc_late'] = (-z + dense + float(b_late)).astype(float)
                        out['dense_ancc_wls'] = (-z + dense + float(b_wls)).astype(float)
                        surf_names.extend(['dense_ancc_full', 'dense_ancc_late', 'dense_ancc_wls'])
                    except Exception:
                        pass
    except Exception as e:
        print('dense imputer skipped', wid, e)
    ens_names = [n for n in surf_names if n in out]
    if len(ens_names) >= 2:
        errs = _gold_np.array([_gold_rmse(out[n][kn], tvt[kn]) for n in ens_names], dtype=float)
        finite = _gold_np.isfinite(errs)
        if int(finite.sum()) >= 2:
            names = [n for n, ok in zip(ens_names, finite) if ok]
            errs = errs[finite]
            weights = 1.0 / _gold_np.maximum(errs, 0.25) ** 2
            weights = weights / weights.sum()
            mat = _gold_np.vstack([out[n] for n in names])
            out['surface_weighted_prefix'] = (weights[:, None] * mat).sum(axis=0).astype(float)
    out.update(_gold_contact_candidate(wid, hw_masked, data_dir))
    return out


def _gold_pf_candidates(hw_masked, tw, variants, n_seeds, n_particles):
    out = {}
    if not _GOLD_INCLUDE_PF:
        return out
    if not callable(globals().get('run_pf_lik_ensemble_scales')) or not callable(globals().get('apply_selector_variant')):
        return out
    kn = hw_masked[hw_masked['TVT_input'].notna()]
    ev = hw_masked[hw_masked['TVT_input'].isna()]
    if len(kn) < 30 or len(ev) == 0:
        return out
    try:
        pf_by_scale = run_pf_lik_ensemble_scales(
            hw_masked,
            tw,
            scales=tuple(globals().get('SELECTOR_SCALES', (3.0, 5.0, 8.0, 12.0))),
            n_particles=int(n_particles),
            n_seeds=int(n_seeds),
        )
        try:
            tvt_beam = run_beam_ensemble(hw_masked, tw)
        except Exception:
            tvt_beam = pf_by_scale.get('pf_mean')
            if tvt_beam is None:
                tvt_beam = next(iter(pf_by_scale.values()))
        last_known_tvt = float(kn['TVT_input'].iloc[-1])
        for variant in variants:
            try:
                pred = apply_selector_variant(variant, pf_by_scale, tvt_beam, last_known_tvt)
                if pred is not None and len(pred) == len(hw_masked):
                    out['pf|' + variant] = _gold_np.asarray(pred, dtype=float)
            except Exception:
                pass
    except Exception as e:
        print('PF calibration skipped:', e)
    return out


def _gold_candidate_pool(wid, hw_masked, tw, data_dir, variants, include_pf=True, n_seeds=24, n_particles=350):
    pool = {}
    pool.update(_gold_poly_candidates(hw_masked))
    pool.update(_gold_surface_candidates(hw_masked, wid, data_dir))
    if include_pf:
        pool.update(_gold_pf_candidates(hw_masked, tw, variants, n_seeds=n_seeds, n_particles=n_particles))
    clean = {}
    for name, pred in pool.items():
        arr = _gold_np.asarray(pred, dtype=float)
        if len(arr) == len(hw_masked) and _gold_np.isfinite(arr).sum() >= max(20, len(hw_masked) // 20):
            clean[name] = arr
    return clean


def _gold_default_pf_name(hw):
    try:
        return 'pf|' + selector_well_code(hw)[1]
    except Exception:
        try:
            return 'pf|' + SELECTOR_GLOBAL_VARIANT
        except Exception:
            return None


def _gold_calibrate_well(wid, hw, tw, data_dir, variants):
    tvt = hw['TVT_input'].to_numpy(dtype=float)
    is_known = _gold_np.isfinite(tvt)
    is_hidden = ~is_known
    if not bool(is_hidden.any()):
        return None
    first_hidden = int(_gold_np.flatnonzero(is_hidden)[0])
    known_prefix = _gold_np.flatnonzero(is_known & (_gold_np.arange(len(hw)) < first_hidden))
    if len(known_prefix) < 140:
        return dict(well=wid, status='skip_short_prefix', known_prefix=int(len(known_prefix)))
    cuts = []
    for frac in _GOLD_CUT_FRACS:
        cut_pos = int(round(len(known_prefix) * float(frac)))
        cut_pos = max(50, min(cut_pos, len(known_prefix) - 35))
        if cut_pos <= 0 or cut_pos >= len(known_prefix):
            continue
        cutoff_idx = int(known_prefix[cut_pos - 1])
        hold_idx = known_prefix[cut_pos:]
        if len(hold_idx) >= 35:
            cuts.append((float(frac), cutoff_idx, hold_idx))
    if not cuts:
        return dict(well=wid, status='skip_no_holdout', known_prefix=int(len(known_prefix)))
    scores = {}
    cut_rows = []
    default_name = _gold_default_pf_name(hw)
    for frac, cutoff_idx, hold_idx in cuts:
        hw_m = hw.copy(deep=True)
        hw_m.loc[hw_m.index > cutoff_idx, 'TVT_input'] = _gold_np.nan
        pool = _gold_candidate_pool(
            wid, hw_m, tw, data_dir, variants,
            include_pf=_GOLD_INCLUDE_PF,
            n_seeds=_GOLD_CAL_SEEDS,
            n_particles=_GOLD_PARTICLES,
        )
        y = tvt[hold_idx]
        row = {'well': wid, 'cut_frac': frac, 'holdout_rows': int(len(hold_idx)), 'candidates': int(len(pool))}
        local = []
        for name, pred in pool.items():
            err = _gold_rmse(pred[hold_idx], y)
            if _gold_np.isfinite(err):
                scores.setdefault(name, []).append(err)
                local.append((err, name))
        local.sort()
        if local:
            row['best_name'] = local[0][1]
            row['best_rmse'] = float(local[0][0])
            if default_name in pool:
                row['default_rmse'] = float(_gold_rmse(pool[default_name][hold_idx], y))
            else:
                row['default_rmse'] = float('nan')
        cut_rows.append(row)
    if not scores:
        return dict(well=wid, status='skip_no_scores', known_prefix=int(len(known_prefix)))
    agg = {}
    for name, vals in scores.items():
        vals = _gold_np.asarray(vals, dtype=float)
        agg[name] = float(_gold_np.nanmedian(vals) + 0.10 * _gold_np.nanstd(vals))
    ordered = sorted((v, k) for k, v in agg.items() if _gold_np.isfinite(v))
    if not ordered:
        return dict(well=wid, status='skip_nonfinite_scores', known_prefix=int(len(known_prefix)))
    best_score, best_name = ordered[0]
    second_score = ordered[1][0] if len(ordered) > 1 else best_score
    if default_name is not None and default_name in agg:
        default_score = float(agg[default_name])
    else:
        pf_scores = [v for k, v in agg.items() if k.startswith('pf|')]
        default_score = float(_gold_np.nanmedian(pf_scores)) if pf_scores else float(second_score)
    consistency = 0.0
    comparable = 0
    for row in cut_rows:
        if _gold_np.isfinite(row.get('default_rmse', _gold_np.nan)):
            comparable += 1
            if row.get('best_rmse', float('inf')) <= row['default_rmse'] - 0.25:
                consistency += 1.0
    if comparable:
        consistency /= comparable
    else:
        winners = [r.get('best_name') for r in cut_rows if r.get('best_name')]
        consistency = float(sum(w == best_name for w in winners)) / max(1, len(winners))
    return dict(
        well=wid,
        status='ok',
        known_prefix=int(len(known_prefix)),
        cuts=int(len(cut_rows)),
        candidate_count=int(len(agg)),
        best_name=best_name,
        best_score=float(best_score),
        second_score=float(second_score),
        default_name=default_name,
        default_score=float(default_score),
        gain=float(default_score - best_score),
        rank_margin=float(second_score - best_score),
        consistency=float(consistency),
        cut_rows=cut_rows,
    )


def _gold_alpha(report, delta_rmse, delta_p95, profile_name):
    p = _GOLD_PROFILES[profile_name]
    if report.get('status') != 'ok':
        return 0.0
    gain = float(report.get('gain', 0.0))
    best = float(report.get('best_score', float('inf')))
    margin = float(report.get('rank_margin', 0.0))
    consistency = float(report.get('consistency', 0.0))
    if (not _gold_np.isfinite(best)) or best > p['max_best'] or gain < p['min_gain'] or consistency < p['min_consistency']:
        return 0.0
    alpha = p['base']
    alpha += p['gain_scale'] * min(max(gain, 0.0), 5.0) / 5.0
    alpha += p['margin_scale'] * min(max(margin, 0.0), 3.0) / 3.0
    if best <= 5.0:
        alpha += p['quality_bonus']
    best_name = str(report.get('best_name', ''))
    if (best_name.startswith('surface_') or best_name.startswith('dense_') or best_name.startswith('poly_') or best_name.startswith('contact_')) and consistency >= 0.67:
        alpha += 0.03 if profile_name != 'aggressive' else 0.06
    if _gold_np.isfinite(delta_rmse) and delta_rmse > p['delta_soft']:
        alpha *= max(0.20, p['delta_soft'] / max(delta_rmse, 1e-6))
    if _gold_np.isfinite(delta_p95) and delta_p95 > p['p95_hard']:
        return 0.0
    return float(min(p['cap'], max(0.0, alpha)))


def _gold_profile_output(base_sub, candidate_by_id, reports_by_well, profile_name):
    prof = _GOLD_PROFILES[profile_name]
    out = base_sub.copy()
    move_rows = []
    for wid, rep in reports_by_well.items():
        ids = out.loc[out['well'] == wid, 'id'].astype(str).tolist()
        if not ids:
            continue
        cand = _gold_np.array([candidate_by_id.get(i, _gold_np.nan) for i in ids], dtype=float)
        idx = out.index[out['well'] == wid].to_numpy()
        base = out.loc[idx, 'tvt'].to_numpy(dtype=float)
        ok = _gold_np.isfinite(cand) & _gold_np.isfinite(base)
        if int(ok.sum()) != len(base):
            rep = dict(rep)
            rep['apply_status'] = 'skip_nonfinite_candidate'
            move_rows.append(rep)
            continue
        diff = cand - base
        delta_rmse = float(_gold_np.sqrt(_gold_np.mean(diff * diff))) if len(diff) else float('nan')
        delta_p95 = float(_gold_np.quantile(_gold_np.abs(diff), 0.95)) if len(diff) else float('nan')
        alpha = _gold_alpha(rep, delta_rmse, delta_p95, profile_name)
        gain = max(0.0, float(rep.get('gain', 0.0)))
        max_move = min(prof['clip_max'], prof['clip_base'] + prof['clip_gain'] * _gold_np.sqrt(gain + 1e-9))
        ramp = 1.0 - _gold_np.exp(-_gold_np.arange(len(diff), dtype=float) / max(80.0, 0.12 * max(1, len(diff))))
        move = _gold_np.clip(alpha * ramp * diff, -max_move, max_move)
        out.loc[idx, 'tvt'] = base + move
        row = dict(rep)
        row.update(dict(
            profile=profile_name,
            alpha=float(alpha),
            delta_rmse_vs_public=float(delta_rmse),
            delta_p95_vs_public=float(delta_p95),
            max_move_clip=float(max_move),
            applied_rows=int(len(idx)),
            mean_abs_move=float(_gold_np.mean(_gold_np.abs(move))) if len(move) else 0.0,
            max_abs_move=float(_gold_np.max(_gold_np.abs(move))) if len(move) else 0.0,
            apply_status='applied' if alpha > 0 else 'kept_public',
        ))
        move_rows.append(row)
    return out, move_rows


def _gold_reapply_guarded_contact_override(sub_df, data_dir):
    sub = _gold_split_ids(sub_df[['id', 'tvt']])
    pred = dict(zip(sub['id'].astype(str), sub['tvt'].astype(float)))
    train_wells = set(p.stem.replace('__horizontal_well', '') for p in (data_dir / 'train').glob('*__horizontal_well.csv'))
    n_ok = 0
    n_skip = 0
    for wid, g in sub.groupby('well'):
        if wid not in train_wells:
            continue
        try:
            hw_te = _gold_pd.read_csv(data_dir / 'test' / f'{wid}__horizontal_well.csv')
            hw_tr = _gold_pd.read_csv(data_dir / 'train' / f'{wid}__horizontal_well.csv')
            tw_tr = _gold_pd.read_csv(data_dir / 'train' / f'{wid}__typewell.csv')
            phys = _gold_tvt_from_contacts(hw_tr, tw_tr)
            md_raw = hw_tr['MD'].to_numpy(dtype=float)
            m = _gold_np.isfinite(phys) & _gold_np.isfinite(md_raw)
            if int(m.sum()) < 100:
                n_skip += 1
                continue
            order = _gold_np.argsort(md_raw[m])
            md_tr = md_raw[m][order]
            ph_tr = phys[m][order]
            kn = hw_te[hw_te['TVT_input'].notna()]
            kn = kn[(kn['MD'] >= md_tr[0]) & (kn['MD'] <= md_tr[-1])]
            if len(kn) < 50:
                n_skip += 1
                continue
            rk = _gold_rmse(_gold_np.interp(kn['MD'].to_numpy(dtype=float), md_tr, ph_tr), kn['TVT_input'].to_numpy(dtype=float))
            if (not _gold_np.isfinite(rk)) or rk > 1.0:
                n_skip += 1
                continue
            md_te = hw_te['MD'].to_numpy(dtype=float)
            n_row = 0
            for rid, ri in zip(g['id'].astype(str).values, g['row_idx'].values):
                ri = int(ri)
                if 0 <= ri < len(md_te):
                    mte = float(md_te[ri])
                    if md_tr[0] <= mte <= md_tr[-1]:
                        pred[rid] = float(_gold_np.interp(mte, md_tr, ph_tr))
                        n_row += 1
            print('gold contact override OK %s rmse=%.4f rows=%d/%d' % (wid, rk, n_row, len(g)))
            n_ok += 1
        except Exception as e:
            print('gold contact override fallback', wid, e)
            n_skip += 1
    sub['tvt'] = sub['id'].astype(str).map(pred).astype(float)
    print('gold contact override done: overridden=%d skipped=%d' % (n_ok, n_skip))
    return sub[['id', 'tvt']]


def _gold_validate_and_write(sub, sample, path):
    out = sub[['id', 'tvt']].copy()
    out['id'] = out['id'].astype(str)
    out['tvt'] = out['tvt'].astype(float)
    if list(out.columns) != ['id', 'tvt']:
        raise RuntimeError('bad output columns')
    if len(out) != len(sample):
        raise RuntimeError('bad output length')
    if not out['id'].equals(sample['id'].astype(str)):
        raise RuntimeError('id order mismatch')
    if not _gold_np.isfinite(out['tvt'].to_numpy(dtype=float)).all():
        raise RuntimeError('non-finite tvt in output')
    out.to_csv(path, index=False)
    return out


if not _GOLD_ENABLE:
    print('Gold visible-prefix calibration disabled; keeping public submission.csv')
else:
    _gold_t0 = _gold_time.time()
    _GOLD_WORK = _gold_work_dir()
    _GOLD_DATA = _gold_find_data()
    _gold_sample = _gold_pd.read_csv(_GOLD_DATA / 'sample_submission.csv')[['id']].copy()
    _gold_sample['id'] = _gold_sample['id'].astype(str)
    _gold_base = _gold_pd.read_csv(_GOLD_WORK / 'submission.csv')[['id', 'tvt']].copy()
    _gold_base['id'] = _gold_base['id'].astype(str)
    _gold_base['tvt'] = _gold_base['tvt'].astype(float)
    _gold_validate_and_write(_gold_base, _gold_sample, _GOLD_WORK / 'submission_public_self_verified.csv')
    _gold_base = _gold_split_ids(_gold_base)
    _gold_variants = _gold_variant_grid()
    print('Gold visible-prefix calibration:', dict(
        profile=_GOLD_PROFILE,
        include_pf=_GOLD_INCLUDE_PF,
        cal_seeds=_GOLD_CAL_SEEDS,
        final_seeds=_GOLD_FINAL_SEEDS,
        particles=_GOLD_PARTICLES,
        cut_fracs=_GOLD_CUT_FRACS,
        variants=len(_gold_variants),
    ))

    _gold_reports = []
    _gold_cut_reports = []
    _gold_candidate_by_id = {}
    _gold_wells = list(_gold_base['well'].drop_duplicates())[:_GOLD_MAX_WELLS]
    for _wi, _wid in enumerate(_gold_wells, 1):
        try:
            _hw_path = _GOLD_DATA / 'test' / f'{_wid}__horizontal_well.csv'
            _tw_path = _GOLD_DATA / 'test' / f'{_wid}__typewell.csv'
            if not _hw_path.exists() or not _tw_path.exists():
                _gold_reports.append(dict(well=_wid, status='skip_missing_files'))
                continue
            _hw = _gold_pd.read_csv(_hw_path)
            _tw = _gold_pd.read_csv(_tw_path)
            print('[gold %d/%d] calibrating %s' % (_wi, len(_gold_wells), _wid), flush=True)
            _rep = _gold_calibrate_well(_wid, _hw, _tw, _GOLD_DATA, _gold_variants)
            if _rep is None:
                _rep = dict(well=_wid, status='skip_none')
            _cut_rows = _rep.pop('cut_rows', []) if isinstance(_rep, dict) else []
            _gold_cut_reports.extend(_cut_rows)
            if _rep.get('status') == 'ok':
                _best_name = _rep['best_name']
                _need_pf_final = str(_best_name).startswith('pf|')
                _pool_final = _gold_candidate_pool(
                    _wid, _hw, _tw, _GOLD_DATA, _gold_variants,
                    include_pf=_need_pf_final,
                    n_seeds=_GOLD_FINAL_SEEDS,
                    n_particles=_GOLD_PARTICLES,
                )
                if _best_name not in _pool_final and _need_pf_final:
                    _pool_final = _gold_candidate_pool(
                        _wid, _hw, _tw, _GOLD_DATA, _gold_variants,
                        include_pf=False,
                        n_seeds=0,
                        n_particles=_GOLD_PARTICLES,
                    )
                if _best_name in _pool_final:
                    _g = _gold_base[_gold_base['well'] == _wid]
                    _arr = _pool_final[_best_name]
                    for _rid, _ri in zip(_g['id'].astype(str).values, _g['row_idx'].astype(int).values):
                        if 0 <= int(_ri) < len(_arr) and _gold_np.isfinite(_arr[int(_ri)]):
                            _gold_candidate_by_id[_rid] = float(_arr[int(_ri)])
                    _rep['final_candidate_available'] = True
                else:
                    _rep['final_candidate_available'] = False
                    _rep['status'] = 'skip_no_final_candidate'
            _gold_reports.append(_rep)
            print('  report:', {k: _rep.get(k) for k in ['status', 'best_name', 'best_score', 'default_score', 'gain', 'consistency']}, flush=True)
        except Exception as _e:
            print('gold calibration fallback', _wid, _e)
            _gold_reports.append(dict(well=_wid, status='error', error=str(_e)))

    _gold_report_df = _gold_pd.DataFrame(_gold_reports)
    _gold_report_df.to_csv(_GOLD_WORK / 'gold_prefix_calibration_report.csv', index=False)
    if _gold_cut_reports:
        _gold_pd.DataFrame(_gold_cut_reports).to_csv(_GOLD_WORK / 'gold_prefix_cut_report.csv', index=False)
    _reports_by_well = {r.get('well'): r for r in _gold_reports if isinstance(r, dict) and r.get('well')}

    _profile_summaries = {}
    for _profile_name in ('conservative', 'balanced', 'aggressive'):
        _profile_sub, _move_rows = _gold_profile_output(_gold_base, _gold_candidate_by_id, _reports_by_well, _profile_name)
        _profile_sub = _gold_reapply_guarded_contact_override(_profile_sub[['id', 'tvt']], _GOLD_DATA)
        _path = _GOLD_WORK / f'submission_gold_prefix_{_profile_name}.csv'
        _profile_sub = _gold_validate_and_write(_profile_sub, _gold_sample, _path)
        _move_df = _gold_pd.DataFrame(_move_rows)
        _move_df.to_csv(_GOLD_WORK / f'gold_prefix_moves_{_profile_name}.csv', index=False)
        _profile_summaries[_profile_name] = dict(
            file=str(_path),
            rows=int(len(_profile_sub)),
            sha256=_gold_sha256(_path),
            applied_wells=int((_move_df.get('apply_status') == 'applied').sum()) if 'apply_status' in _move_df else 0,
            mean_abs_move=float(_move_df['mean_abs_move'].mean()) if 'mean_abs_move' in _move_df and len(_move_df) else 0.0,
            max_abs_move=float(_move_df['max_abs_move'].max()) if 'max_abs_move' in _move_df and len(_move_df) else 0.0,
        )

    _chosen_path = _GOLD_WORK / f'submission_gold_prefix_{_GOLD_PROFILE}.csv'
    _chosen = _gold_pd.read_csv(_chosen_path)
    _chosen = _gold_validate_and_write(_chosen, _gold_sample, _GOLD_WORK / 'submission.csv')
    _audit = dict(
        selected_profile=_GOLD_PROFILE,
        selected_sha256=_gold_sha256(_GOLD_WORK / 'submission.csv'),
        public_anchor_sha256=_gold_sha256(_GOLD_WORK / 'submission_public_self_verified.csv'),
        elapsed_sec=float(_gold_time.time() - _gold_t0),
        wells=int(len(_gold_wells)),
        candidates_with_final_values=int(len(_gold_candidate_by_id)),
        profiles=_profile_summaries,
    )
    with open(_GOLD_WORK / 'gold_prefix_submission_audit.json', 'w', encoding='utf-8') as f:
        _gold_json.dump(_audit, f, indent=2, sort_keys=True)
    print('Gold visible-prefix selected submission.csv:', _audit, flush=True)


In [ ]:
# === Bayesian forward-backward smoother (grid / discretized-particle representation) ===
# Geosteering as state estimation. Latent stratigraphic datum U(MD)=TVT+Z (the same quantity
# the ANCC/Z particle filters track) follows a random walk around a DRIFT PRIOR; the emission
# is GR(MD) vs typewell GR(TVT). Two priors are blended via w_struct:
#   * linear  : last_U + (dU/dMD from the prefix tail) * dMD          (flat dip)
#   * struct  : offset-well FormationPlaneKNN surface + per-well offset (real dip from neighbours)
# Unlike the forward-only particle filters this runs FORWARD + BACKWARD, so future GR inside the
# eval zone retroactively constrains earlier rows. A grid posterior has no particle degeneracy.
# Non-destructive: defines functions + runs train CV that auto-selects w_struct.
import numpy as np, pandas as pd, glob, os
from pathlib import Path
from scipy.ndimage import gaussian_filter1d


def _smoother_data_dir():
    for c in ["/kaggle/input/competitions/rogii-wellbore-geology-prediction",
              "/kaggle/input/rogii-wellbore-geology-prediction"]:
        p = Path(c)
        if p.exists() and (p / "train").exists():
            return p
    for g in glob.glob("/kaggle/input/**/train", recursive=True):
        return Path(g).parent
    return Path(os.environ.get("ROGII_DATA", "."))


SM_DATA = _smoother_data_dir()
print("smoother DATA:", SM_DATA)
SM_FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]


def _dense_gr(tw_tvt, tw_gr, pad=250.0, step=0.1):
    lo = float(np.min(tw_tvt)) - pad
    hi = float(np.max(tw_tvt)) + pad
    grid = np.arange(lo, hi + step, step)
    return np.interp(grid, tw_tvt, tw_gr).astype(np.float32), lo, step


def _gr_at(gg, lo, step, tvt):
    idx = ((tvt - lo) / step).astype(np.int64)
    np.clip(idx, 0, len(gg) - 1, out=idx)
    return gg[idx]


def _robust_slope(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3 or np.std(x[m]) < 1e-6:
        return 0.0
    return float(np.polyfit(x[m], y[m], 1)[0])


def _struct_u_prior(hw, U_known, known, fi, self_wid):
    """Offset-well structural surface for U=TVT+Z, per row. Returns U_struct or None."""
    if fi is None:
        return None
    try:
        xy = hw[["X", "Y"]].to_numpy(float)
        form_all, _ = fi.impute(xy, self_wid=self_wid)        # (T, n_form)
    except Exception:
        return None
    Us = []
    for f in range(form_all.shape[1]):
        fv = form_all[:, f].astype(float)
        bk = U_known - fv[known]
        bk = bk[np.isfinite(bk)]
        if len(bk) == 0:
            continue
        b_f = float(np.median(bk))
        if np.isfinite(b_f):
            Us.append(fv + b_f)
    if not Us:
        return None
    return np.nanmedian(np.stack(Us, 0), axis=0)


def bayes_smoother_well(hw, tw, fi=None, w_struct=0.0, wid=None, is_train=False,
                        grid_half=50.0, grid_step=0.5, q_walk=0.35,
                        gr_sig_min=8.0, gr_sig_max=60.0, gr_sig_def=30.0,
                        pin_sig=0.5, ll_cap=80.0, tail=50):
    """Full-length TVT estimate (known rows kept exact). hw is in as-loaded MD order."""
    tw = tw.sort_values("TVT")
    tw_tvt = tw["TVT"].to_numpy(float)
    tw_gr = tw["GR"].fillna(tw["GR"].mean()).to_numpy(float)
    if len(tw_tvt) < 3:
        return None
    MD = hw["MD"].to_numpy(float); Z = hw["Z"].to_numpy(float)
    ti = hw["TVT_input"].to_numpy(float)
    known = np.isfinite(ti); ev = ~known
    if known.sum() < 5 or ev.sum() == 0:
        return ti.copy()
    gr = (hw["GR"].astype(float).interpolate(limit_direction="both")
          .fillna(float(np.mean(tw_gr))).to_numpy(float))
    # --- linear drift prior on U = TVT + Z ---
    kpos = np.where(known)[0]; last = kpos[-1]
    last_U = ti[last] + Z[last]; last_MD = MD[last]
    tk = kpos[-min(tail, len(kpos)):]
    ir = _robust_slope(MD[tk], ti[tk] + Z[tk])               # dU/dMD from the prefix tail
    u_prior = last_U + ir * (MD - last_MD)
    # --- blend in the structural (offset-well) prior ---
    if fi is not None and w_struct > 0:
        U_struct = _struct_u_prior(hw, ti[known] + Z[known], known, fi,
                                   wid if is_train else None)
        if U_struct is not None:
            ok = np.isfinite(U_struct)
            u_prior = u_prior.copy()
            u_prior[ok] = (1.0 - w_struct) * u_prior[ok] + w_struct * U_struct[ok]
    # --- GR emission noise from known-prefix residuals (typewell match) ---
    res = gr[known] - np.interp(ti[known], tw_tvt, tw_gr)
    gr_sig = float(np.clip(np.std(res) if (len(res) > 20 and np.isfinite(res).all()) else gr_sig_def,
                           gr_sig_min, gr_sig_max))
    # --- residual grid r = U - u_prior ---
    rr = np.arange(-grid_half, grid_half + grid_step, grid_step).astype(np.float32)
    K = len(rr)
    gg, glo, gstep = _dense_gr(tw_tvt, tw_gr)
    T = len(MD)
    tvt_cells = (u_prior[:, None] + rr[None, :]) - Z[:, None]      # (T,K)
    gr_pred = _gr_at(gg, glo, gstep, tvt_cells)
    resid = (gr[:, None] - gr_pred) / gr_sig
    ll = (-0.5 * np.minimum(resid * resid, ll_cap)).astype(np.float32)
    # hard pin on known rows: posterior must sit at the known TVT_input
    kp = (tvt_cells - ti[:, None]) / pin_sig
    ll_known = (-0.5 * np.minimum(kp * kp, 1e6)).astype(np.float32)
    ll = np.where(known[:, None], ll_known, ll)
    E = np.exp(ll - ll.max(1, keepdims=True)); np.maximum(E, 1e-30, out=E)
    # --- forward / backward over a random-walk transition (Gaussian convolution) ---
    dm = np.empty(T); dm[1:] = np.maximum(MD[1:] - MD[:-1], 1e-3)
    dm[0] = dm[1] if T > 1 else 1.0
    sig_cells = np.clip(q_walk * np.sqrt(dm) / grid_step, 1e-3, K / 3.0)
    A = np.empty((T, K), np.float32); a = E[0] / E[0].sum(); A[0] = a
    for t in range(1, T):
        a = gaussian_filter1d(a, sig_cells[t], mode="constant")
        a = a * E[t]; s = a.sum()
        a = a / s if s > 0 else np.full(K, 1.0 / K, np.float32); A[t] = a
    B = np.empty((T, K), np.float32); b = np.full(K, 1.0 / K, np.float32); B[T - 1] = b
    for t in range(T - 2, -1, -1):
        bb = b * E[t + 1]; bb = gaussian_filter1d(bb, sig_cells[t + 1], mode="constant")
        s = bb.sum(); b = bb / s if s > 0 else np.full(K, 1.0 / K, np.float32); B[t] = b
    G = (A * B); G /= G.sum(1, keepdims=True)
    r_hat = G @ rr
    tvt_hat = (u_prior + r_hat) - Z
    tvt_hat[known] = ti[known]
    return tvt_hat


def smoother_predict_well(wid, split, **kw):
    hw = pd.read_csv(SM_DATA / split / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(SM_DATA / split / f"{wid}__typewell.csv")
    out = bayes_smoother_well(hw, tw, wid=wid, is_train=(split == "train"), **kw)
    if out is None:
        return None
    ev_idx = hw.index[hw["TVT_input"].isna()].to_numpy()
    return {f"{wid}_{i}": float(out[i]) for i in ev_idx}


# ---- structural imputer (reuse the notebook's tested FormationPlaneKNN if available) ----
_SM_FI = None
if "FormationPlaneKNN" in globals():
    try:
        _sm_train_wids = sorted(p.stem.replace("__horizontal_well", "")
                                for p in (SM_DATA / "train").glob("*__horizontal_well.csv"))
        _SM_FI = FormationPlaneKNN(_sm_train_wids, SM_DATA / "train")
        print("structural imputer built on", len(_sm_train_wids), "wells")
    except Exception as _e:
        print("structural imputer unavailable, linear prior only:", _e)
else:
    print("FormationPlaneKNN not in globals -> linear prior only (run earlier cells first)")


# ---- CV on train wells (TVT known): pooled RMSE, the notebook's CV metric ----
def smoother_cv(n_wells=200, fi=None, w_struct=0.0, **kw):
    wids = sorted(p.stem.replace("__horizontal_well", "")
                  for p in (SM_DATA / "train").glob("*__horizontal_well.csv"))
    if n_wells:
        wids = wids[:n_wells]
    se_sm = se_base = 0.0; n = 0; per = []
    for wid in wids:
        try:
            hw = pd.read_csv(SM_DATA / "train" / f"{wid}__horizontal_well.csv")
            tw = pd.read_csv(SM_DATA / "train" / f"{wid}__typewell.csv")
        except Exception:
            continue
        if "TVT" not in hw.columns or hw["TVT"].isna().all():
            continue
        ev = hw["TVT_input"].isna().to_numpy()
        yt = hw["TVT"].to_numpy(float)
        if ev.sum() == 0 or (~ev).sum() < 5 or not np.isfinite(yt[ev]).any():
            continue
        out = bayes_smoother_well(hw, tw, fi=fi, w_struct=w_struct,
                                  wid=wid, is_train=True, **kw)
        if out is None:
            continue
        m = ev & np.isfinite(yt)
        if m.sum() == 0:
            continue
        last_known = float(hw["TVT_input"].dropna().iloc[-1])
        e_sm = out[m] - yt[m]; e_bs = last_known - yt[m]
        se_sm += float(np.sum(e_sm ** 2)); se_base += float(np.sum(e_bs ** 2)); n += int(m.sum())
        per.append((wid, float(np.sqrt(np.mean(e_sm ** 2)))))
    rmse_sm = (se_sm / n) ** 0.5 if n else float("nan")
    rmse_bs = (se_base / n) ** 0.5 if n else float("nan")
    print(f"  w_struct={w_struct}: pooled RMSE smoother={rmse_sm:.4f} | "
          f"baseline(last-known)={rmse_bs:.4f} | wells={len(per)} rows={n}")
    return rmse_sm, rmse_bs, pd.DataFrame(per, columns=["well", "rmse"])


# Sweep the prior mix and auto-select. Raise SM_CV_N (or 0 = all) once you trust it; tune
# grid_half / q_walk / grid_step. MSE on the LB is dominated by long eval zones, so watch
# whether structural prior helps most there.
SM_CV_N = 200
print("\n=== smoother CV: linear vs structural drift prior ===")
_cv_scores = {}
for _ws in (0.0, 0.5, 1.0) if _SM_FI is not None else (0.0,):
    _cv_scores[_ws] = smoother_cv(n_wells=SM_CV_N, fi=_SM_FI, w_struct=_ws)[0]
SM_W_STRUCT = min(_cv_scores, key=_cv_scores.get)
print(f"selected w_struct = {SM_W_STRUCT}  (best pooled RMSE = {_cv_scores[SM_W_STRUCT]:.4f})")


In [ ]:
# === Smoother: test predictions + non-destructive blend candidates ===
# Uses the CV-selected SM_W_STRUCT and the structural imputer _SM_FI. Writes sub_smoother.csv
# and submission_blend_smoother_w*.csv. Does NOT overwrite submission.csv. Adopt a weight only
# after checking CV / LB.
import pandas as pd, numpy as np
from pathlib import Path

SM_OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
_W_STRUCT = globals().get("SM_W_STRUCT", 0.0)
_FI = globals().get("_SM_FI", None)
print(f"smoother submit: w_struct={_W_STRUCT}, structural_prior={_FI is not None}")

_test_wids = sorted(p.stem.replace("__horizontal_well", "")
                    for p in (SM_DATA / "test").glob("*__horizontal_well.csv"))
print(f"predicting {len(_test_wids)} test wells...")

_rows = {}
for _wid in _test_wids:
    try:
        _r = smoother_predict_well(_wid, "test", fi=_FI, w_struct=_W_STRUCT)
        if _r is not None:
            _rows.update(_r)
    except Exception as _e:
        print("  smoother fail", _wid, _e)

_sample = pd.read_csv(SM_DATA / "sample_submission.csv")
_sub_sm = pd.DataFrame({"id": list(_rows.keys()), "tvt": list(_rows.values())})
_sub_sm = _sample[["id"]].merge(_sub_sm, on="id", how="left")
_n_missing = int(_sub_sm["tvt"].isna().sum())
_fallback = float(_sub_sm["tvt"].dropna().mean()) if _sub_sm["tvt"].notna().any() else 0.0
_sub_sm["tvt"] = _sub_sm["tvt"].fillna(_fallback)
_sub_sm.to_csv(SM_OUT / "sub_smoother.csv", index=False)
print("wrote sub_smoother.csv", _sub_sm.shape, "| rows filled by fallback:", _n_missing)

# Blend candidates against the current pipeline output (read-only; submission.csv untouched).
_cur_path = SM_OUT / "submission.csv"
if _cur_path.exists():
    _cur = pd.read_csv(_cur_path)[["id", "tvt"]].rename(columns={"tvt": "tvt_cur"})
    _m = _cur.merge(_sub_sm.rename(columns={"tvt": "tvt_sm"}), on="id", how="inner")
    _d = _m["tvt_sm"].to_numpy(float) - _m["tvt_cur"].to_numpy(float)
    print(f"smoother vs current: rows={len(_m)} "
          f"rmse_diff={np.sqrt(np.mean(_d**2)):.3f} p95|d|={np.quantile(np.abs(_d),0.95):.3f}")
    for _w in (0.15, 0.25, 0.35, 0.50):
        _c = _m[["id"]].copy()
        _c["tvt"] = (1 - _w) * _m["tvt_cur"] + _w * _m["tvt_sm"]
        _c.to_csv(SM_OUT / f"submission_blend_smoother_w{_w:.2f}.csv", index=False)
        print(f"  wrote submission_blend_smoother_w{_w:.2f}.csv")
    print("To adopt a blend: pick a weight by CV/LB, then copy that file to submission.csv.")
else:
    print("submission.csv not found yet - run the main pipeline first. sub_smoother.csv is standalone.")
